<a href="https://colab.research.google.com/github/IRay3477/DatEng/blob/main/Kelompok_5_Scientific_Publication_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Scientific Publication Pipeline — Sprint 1 (Revisi) & Sprint 2

**Automated Scientific Publication Metadata & Open Access Enrichment Pipeline — Kelompok 5**

Notebook ini berisi dua bagian besar:
- **Walkthrough (Bagian 1–8):** setiap tahap dijalankan satu per satu dan ditampilkan hasilnya (extract, profiling, data issues, cleaning, integrasi).
- **Repeatable pipeline (Bagian 9–11):** tahap yang sama dibungkus menjadi fungsi, dijalankan lewat satu fungsi utama `run_pipeline()`, lalu **dijalankan dua kali** untuk membuktikan hasilnya konsisten (idempotent).

### 📝 Perbaikan dari masukan dosen
| No | Masukan | Perbaikan |
|----|---------|-----------|
| 1 | Hapus 191 baris tanpa ISSN terlalu agresif | Baris tidak dihapus: `match_status = "no_issn"`. Tahun tidak valid ditandai `is_year_valid` (nilai asli di `issued_year_raw`) |
| 2 | Ambil 1 license pertama terlalu agresif | `license_raw` utuh + `license_count` + tabel jembatan `journal_license` |
| 3 | Cara penggunaan kurs | Kurs dari API, respons mentah disimpan; setiap konversi punya `apc_currency`, `fx_rate`, `fx_date` |
| 4 | Data didominasi tahun 2011 | Stratified random sampling: ±1.250 artikel acak per tahun 2011–2026 |
| 5 | Tambah database | SQLite dengan PK/FK, view, dan log run |
| 6 | Traceability (asistensi Sprint 2) | Flag ISSN ambigu, `ambiguous_issn`, log entri DOAJ ganda yang dihapus |
| 7 | Label "tidak terindeks" terlalu menyimpulkan | `is_matched_to_doaj` + status `not_matched_to_doaj`; atribut DOAJ dibiarkan NULL |
| 8 | Snapshot parsial tidak boleh menimpa snapshot valid | File ditulis ke `.tmp` lalu di-rename hanya jika lengkap; processed lama baru diganti setelah read-back validation lolos |

---

## 📦 0. Setup & Persiapan Environment

In [1]:
!pip install requests pandas pyarrow -q

In [2]:
import os
import re
import json
import math
import time
import gzip
import shutil
import sqlite3
import requests
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.dataset as ds
from pathlib import Path
from contextlib import closing
from collections import Counter

print("✅ Library siap!")

✅ Library siap!


### 💾 Lokasi penyimpanan: Google Drive

Disk Colab terhapus saat sesi berakhir. Karena hasil sampling Crossref acak, **raw snapshot harus disimpan permanen** agar pipeline bisa diulang dari data yang sama. Karena itu folder pipeline diletakkan di Google Drive. Jika notebook dijalankan di luar Colab, folder lokal `pipeline_data/` dipakai.

In [3]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/kelompok5_publication_pipeline")
except ImportError:
    BASE_DIR = Path("pipeline_data")

BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Folder pipeline: {BASE_DIR.resolve()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Folder pipeline: /content/drive/MyDrive/kelompok5_publication_pipeline


Sel Konfigurasi

In [4]:
# Snapshot yang diproses.
#   None          → pakai tanggal hari ini (mengambil snapshot baru bila belum ada)
#   "YYYY-MM-DD"  → pakai snapshot raw yang sudah tersimpan (tanpa request ulang)
LOCAL_TZ = "Asia/Makassar"
INGESTION_DATE = "2026-11-07"
SNAPSHOT_DATE = INGESTION_DATE or pd.Timestamp.now(tz=LOCAL_TZ).strftime("%Y-%m-%d")

# Rentang tahun yang dianalisis
MIN_YEAR = 2011
CURRENT_YEAR = int(SNAPSHOT_DATE[:4])                  # tahun > ini dianggap tidak valid
CROSSREF_YEARS = list(range(MIN_YEAR, CURRENT_YEAR + 1))

# Crossref API — stratified random sampling per tahun
CROSSREF_URL = "https://api.crossref.org/works"
CROSSREF_ROWS_PER_YEAR = 1250                           # kuota sama tiap tahun
CROSSREF_SAMPLE_SIZE = 100                              # batas maksimum parameter `sample`
CROSSREF_EXTRA_REQUESTS = 10                            # cadangan request/tahun untuk mengganti DOI duplikat
CROSSREF_EMAIL = "ray.adhya4@gmail.com"
CROSSREF_DELAY = 1.0                                    # jeda antar request (detik) — polite pool

# DOAJ & kurs
DOAJ_CSV_URL = "https://doaj.org/csv"
DOAJ_MIN_ROWS = 20000                                   # di bawah ini dianggap download tidak lengkap
DOAJ_PRINT_ISSN_COL = "Journal ISSN (print version)"
DOAJ_ONLINE_ISSN_COL = "Journal EISSN (online version)"
FX_API_URL = "https://open.er-api.com/v6/latest/USD"
FX_MIN_CURRENCIES = 100                                 # di bawah ini dianggap respons kurs tidak lengkap

# Retry semua request HTTP
HTTP_MAX_RETRIES = 3

# Aturan validasi
ISSN_REGEX = r"^[0-9]{4}-[0-9]{3}[0-9X]$"               # format standar ISSN: XXXX-XXXX

# Lokasi layer
RAW_DIR = BASE_DIR / "raw"
STAGING_DIR = BASE_DIR / "staging"
PROCESSED_DIR = BASE_DIR / "processed"                  # selalu berisi snapshot TERBARU
BACKFILL_DIR = BASE_DIR / "processed_backfill"          # hasil backfill: 1 folder per snapshot lama
LOG_DIR = BASE_DIR / "logs"
LOCAL_WORK_DIR = Path("work")                           # processed dibangun & divalidasi di sini dulu
for folder in [RAW_DIR, STAGING_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Konfigurasi dimuat.")
print(f"   Snapshot            : {SNAPSHOT_DATE}")
print(f"   Rentang tahun       : {MIN_YEAR}–{CURRENT_YEAR} ({len(CROSSREF_YEARS)} tahun)")
print(f"   Target Crossref     : {CROSSREF_ROWS_PER_YEAR * len(CROSSREF_YEARS):,} artikel ({CROSSREF_ROWS_PER_YEAR:,} per tahun)")
print(f"   Estimasi request    : ±{math.ceil(CROSSREF_ROWS_PER_YEAR / CROSSREF_SAMPLE_SIZE) * len(CROSSREF_YEARS)}")

✅ Konfigurasi dimuat.
   Snapshot            : 2026-11-07
   Rentang tahun       : 2011–2026 (16 tahun)
   Target Crossref     : 20,000 artikel (1,250 per tahun)
   Estimasi request    : ±208


---

# 📥 BAGIAN 1 — EXTRACT (Raw Layer)

Data dari 3 sumber disimpan **apa adanya** di raw layer, dipartisi per sumber dan tanggal snapshot:

```
raw/
├── crossref/ingestion_date=YYYY-MM-DD/crossref_2011.jsonl.gz ... crossref_2026.jsonl.gz
├── doaj/ingestion_date=YYYY-MM-DD/doaj_journals.csv
├── fx_rates/ingestion_date=YYYY-MM-DD/fx_rates_usd.json
└── _manifests/manifest_YYYY-MM-DD.json      ← penanda snapshot LENGKAP + metadata run
```

### 🛡️ Aturan agar snapshot parsial tidak menimpa snapshot valid
1. Setiap file ditulis dulu ke `*.tmp`, lalu di-*rename* ke nama final **hanya jika isinya lengkap** dan lolos validasi. Rename bersifat atomik: file final selalu utuh, tidak pernah setengah jadi.
2. File final yang sudah ada **tidak pernah ditimpa**. Saat dijalankan ulang, tahun/sumber yang sudah lengkap dilewati, sehingga ekstraksi bisa dilanjutkan dari titik gagal.
3. Tahun Crossref yang kuotanya tidak terpenuhi disimpan sebagai `*.incomplete` (untuk debugging, tidak dipakai).
4. **Manifest** hanya ditulis bila ketiga sumber lengkap. Tahap processed hanya mau membaca snapshot yang punya manifest.

In [5]:
class IncompleteSnapshotError(RuntimeError):
    """Snapshot raw tidak lengkap → tahap processed tidak boleh dijalankan."""


def raw_snapshot_dir(source, snapshot_date):
    """Folder raw untuk satu sumber pada satu tanggal snapshot."""
    folder = RAW_DIR / source / f"ingestion_date={snapshot_date}"
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def manifest_path(snapshot_date):
    return RAW_DIR / "_manifests" / f"manifest_{snapshot_date}.json"

def snapshot_config(snapshot_date):
    """Daftar tahun & kuota milik SATU snapshot.
    Snapshot yang sudah lengkap → dibaca dari manifest (apa adanya saat diambil).
    Snapshot baru               → dihitung dari tanggal snapshot (bukan tanggal hari ini)."""
    path = manifest_path(snapshot_date)
    if path.exists():
        crossref = json.loads(path.read_text(encoding="utf-8"))["sources"]["crossref"]
        years, quota = [int(y) for y in crossref["years"]], int(crossref["quota_per_year"])
    else:
        years = list(range(MIN_YEAR, int(snapshot_date[:4]) + 1))
        quota = CROSSREF_ROWS_PER_YEAR
    return {"years": years, "quota_per_year": quota, "max_valid_year": max(years)}


class NonRetryableError(RuntimeError):
    """Error permanen (mis. HTTP 400/404): diulang pun hasilnya sama → pipeline langsung berhenti."""


# Retry hanya untuk transient failure: server sibuk/sementara error, rate limit, timeout, koneksi putus
RETRYABLE_STATUS = {429, 500, 502, 503, 504}
ACTIVE_TRACKER = None   # diisi oleh run_pipeline(); dipakai untuk mencatat status "retrying"


def http_get_with_retry(url, params=None, headers=None, timeout=60, label=""):
    """GET dengan retry + exponential backoff (2s, 4s) KHUSUS untuk transient failure.
    Return response (200), None jika transient failure tetap terjadi setelah semua percobaan,
    atau raise NonRetryableError untuk error permanen (4xx selain 429)."""
    for attempt in range(1, HTTP_MAX_RETRIES + 1):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=timeout)
            if response.status_code == 200:
                return response
            if response.status_code not in RETRYABLE_STATUS:
                raise NonRetryableError(
                    f"{label}: HTTP {response.status_code} bukan transient failure → tidak di-retry "
                    f"(cek URL/parameter request).")
            reason = f"{label}: status {response.status_code}"
        except (requests.Timeout, requests.ConnectionError) as error:
            reason = f"{label}: {type(error).__name__}"
        print(f"      ⚠️ {reason} (percobaan {attempt}/{HTTP_MAX_RETRIES})")
        if attempt < HTTP_MAX_RETRIES:
            if ACTIVE_TRACKER is not None:
                ACTIVE_TRACKER.retrying(f"{reason}, percobaan ulang ke-{attempt}")
            time.sleep(CROSSREF_DELAY * 2 ** attempt)
    return None


print("✅ Helper extract siap.")

✅ Helper extract siap.


## 1A. Crossref — Stratified Random Sampling per Tahun

**Masalah versi lama:** paginasi cursor dengan filter `from-pub-date:2011-01-01` mengembalikan record sesuai urutan indeks Crossref, sehingga 88% artikel terbit tahun 2011.

**Solusi:** untuk setiap tahun, ambil artikel **acak** memakai parameter `sample` (maks. 100 per request):
```
filter = type:journal-article,from-pub-date:{tahun}-01-01,until-pub-date:{tahun}-12-31
```
- Kuota sama per tahun → sebaran antar tahun seimbang; acak di dalam tahun → tidak didominasi publisher tertentu
- Request berbeda bisa mengembalikan DOI yang sama → dedup DOI + request tambahan sampai kuota terpenuhi
- Hasil acak berbeda tiap run → karena itu raw snapshot disimpan, dan pengulangan pipeline memakai snapshot yang sama

⏱️ ±210 request → sekitar 5–15 menit.

In [6]:
def extract_crossref(snapshot_date):
    """Ambil sampel acak per tahun. Hanya tahun yang kuotanya lengkap yang disimpan sebagai file final."""
    out_dir = raw_snapshot_dir("crossref", snapshot_date)
    config = snapshot_config(snapshot_date)
    quota = config["quota_per_year"]
    headers = {"User-Agent": f"TugasDataEngineering/1.0 (mailto:{CROSSREF_EMAIL})"}
    max_requests = math.ceil(quota / CROSSREF_SAMPLE_SIZE) + CROSSREF_EXTRA_REQUESTS
    extraction_log, incomplete_years = [], []

    for year in config["years"]:
        final_file = out_dir / f"crossref_{year}.jsonl.gz"
        if final_file.exists():
            print(f"   ✅ {year}: snapshot lengkap sudah ada, dilewati")
            continue

        year_items = {}                                  # DOI (lowercase) → item mentah dari API
        n_requests = n_duplicates = n_failed = 0
        start_year = time.time()
        params = {
            "filter": f"type:journal-article,from-pub-date:{year}-01-01,until-pub-date:{year}-12-31",
            "mailto": CROSSREF_EMAIL,
        }
        while len(year_items) < quota and n_requests < max_requests:
            params["sample"] = min(CROSSREF_SAMPLE_SIZE, quota - len(year_items))
            response = http_get_with_retry(CROSSREF_URL, params=params, headers=headers, label=str(year))
            n_requests += 1
            if response is None:
                n_failed += 1
                continue
            for item in response.json().get("message", {}).get("items", []):
                doi = str(item.get("DOI", "")).lower()
                if doi in year_items:
                    n_duplicates += 1
                else:
                    year_items[doi] = item
            time.sleep(CROSSREF_DELAY)

        # Tulis ke .tmp dulu (1 baris JSON = 1 item respons API apa adanya)
        tmp_file = out_dir / f"crossref_{year}.jsonl.gz.tmp"
        with gzip.open(tmp_file, "wt", encoding="utf-8") as f:
            for item in year_items.values():
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        shortfall = quota - len(year_items)
        if shortfall == 0:
            os.replace(tmp_file, final_file)             # commit atomik: hanya snapshot lengkap
            status = "✅"
        else:
            os.replace(tmp_file, out_dir / f"crossref_{year}.jsonl.gz.incomplete")
            incomplete_years.append(year)
            status = f"❌ kurang {shortfall}, tidak di-commit"
        extraction_log.append({
            "snapshot_date": snapshot_date, "year": year, "requests": n_requests,
            "failed_requests": n_failed, "articles": len(year_items),
            "duplicate_doi_skipped": n_duplicates, "shortfall": shortfall,
            "seconds": round(time.time() - start_year, 1),
        })
        print(f"   📄 {year} | {n_requests:2d} request | {len(year_items):,} artikel | "
              f"{n_duplicates} DOI duplikat dilewati | {status}")

    if extraction_log:
        log_file = LOG_DIR / "crossref_extraction_log.csv"
        pd.DataFrame(extraction_log).to_csv(log_file, mode="a", header=not log_file.exists(), index=False)
    if incomplete_years:
        raise IncompleteSnapshotError(
            f"Crossref belum lengkap untuk tahun {incomplete_years}. Jalankan ulang: "
            f"tahun yang sudah lengkap akan dilewati.")
    return pd.DataFrame(extraction_log)

## 1B. DOAJ — CSV Master Data

Snapshot DOAJ berubah dari waktu ke waktu, sehingga disimpan per tanggal. File hanya di-commit bila bisa dibaca, minimal `DOAJ_MIN_ROWS` baris, dan memiliki kolom kunci.

In [7]:
def extract_doaj(snapshot_date):
    """Download CSV DOAJ utuh (52 kolom). Commit hanya jika lolos validasi kelengkapan."""
    final_file = raw_snapshot_dir("doaj", snapshot_date) / "doaj_journals.csv"
    if final_file.exists():
        print(f"   ✅ DOAJ: snapshot sudah ada, dilewati")
        return

    response = http_get_with_retry(DOAJ_CSV_URL, timeout=120, label="DOAJ")
    if response is None:
        raise IncompleteSnapshotError("Download DOAJ gagal setelah semua percobaan.")

    tmp_file = final_file.with_name(final_file.name + ".tmp")
    tmp_file.write_bytes(response.content)
    check = pd.read_csv(tmp_file, encoding="utf-8", encoding_errors="replace", low_memory=False)
    required_cols = {"Journal title", DOAJ_PRINT_ISSN_COL, DOAJ_ONLINE_ISSN_COL, "Journal license", "APC amount"}
    missing_cols = required_cols - set(check.columns)
    if len(check) < DOAJ_MIN_ROWS or missing_cols:
        tmp_file.unlink()
        raise IncompleteSnapshotError(
            f"CSV DOAJ tidak lengkap ({len(check):,} baris, kolom hilang: {missing_cols}).")
    os.replace(tmp_file, final_file)
    print(f"   💾 DOAJ: {len(check):,} jurnal × {check.shape[1]} kolom "
          f"({final_file.stat().st_size / 1024**2:.1f} MB)")

## 1C. Kurs Mata Uang (ExchangeRate-API)

- Sumber: `https://open.er-api.com/v6/latest/USD` (±160 mata uang, tanpa API key, update harian)
- Format **1 USD = X mata uang** → konversi memakai pembagian: `apc_amount_usd = apc_amount_original / rate_per_usd`
- Respons mentah disimpan di raw layer, sehingga setiap angka USD bisa ditelusuri ke sumber dan tanggal kursnya
- Tanggal kurs = tanggal snapshot DOAJ, karena APC di DOAJ adalah tarif saat ini

<a href="https://www.exchangerate-api.com">Rates By Exchange Rate API</a> (atribusi wajib sesuai ketentuan layanan)

In [8]:
def extract_fx(snapshot_date):
    """Ambil kurs basis USD. Commit hanya jika respons sukses dan cakupan mata uang memadai."""
    final_file = raw_snapshot_dir("fx_rates", snapshot_date) / "fx_rates_usd.json"
    if final_file.exists():
        print(f"   ✅ Kurs: snapshot sudah ada, dilewati")
        return

    response = http_get_with_retry(FX_API_URL, timeout=60, label="Kurs")
    if response is None:
        raise IncompleteSnapshotError("Request kurs gagal setelah semua percobaan.")
    fx_raw = response.json()
    rates = fx_raw.get("rates", {})
    if fx_raw.get("result") != "success" or fx_raw.get("base_code") != "USD" or len(rates) < FX_MIN_CURRENCIES:
        raise IncompleteSnapshotError(
            f"Respons kurs tidak valid (result={fx_raw.get('result')}, mata uang={len(rates)}).")

    tmp_file = final_file.with_name(final_file.name + ".tmp")
    tmp_file.write_text(json.dumps(fx_raw, indent=2), encoding="utf-8")
    os.replace(tmp_file, final_file)
    print(f"   💾 Kurs: {len(rates)} mata uang, update {fx_raw.get('time_last_update_utc')}")

In [9]:
def write_manifest(snapshot_date):
    """Tulis manifest (penanda snapshot LENGKAP). Hanya dipanggil setelah ketiga extract berhasil."""
    config = snapshot_config(snapshot_date)
    crossref_dir = raw_snapshot_dir("crossref", snapshot_date)
    crossref_files = sorted(crossref_dir.glob("crossref_*.jsonl.gz"))
    crossref_records = 0
    for path in crossref_files:
        with gzip.open(path, "rt", encoding="utf-8") as f:
            crossref_records += sum(1 for _ in f)
    doaj_file = raw_snapshot_dir("doaj", snapshot_date) / "doaj_journals.csv"
    fx_file = raw_snapshot_dir("fx_rates", snapshot_date) / "fx_rates_usd.json"
    fx_raw = json.loads(fx_file.read_text(encoding="utf-8"))

    manifest = {
        "snapshot_date": snapshot_date,
        "completed_at": pd.Timestamp.now(tz="UTC").isoformat(),
        "sources": {
            "crossref": {
                "endpoint": CROSSREF_URL,
                "method": "stratified random sampling per tahun (parameter sample)",
                "filter_template": "type:journal-article,from-pub-date:{tahun}-01-01,until-pub-date:{tahun}-12-31",
                "quota_per_year": config["quota_per_year"],
                "years": config["years"],
                "records": crossref_records,
                "files": [p.name for p in crossref_files],
            },
            "doaj": {
                "endpoint": DOAJ_CSV_URL,
                "records": len(pd.read_csv(doaj_file, encoding="utf-8", encoding_errors="replace", low_memory=False)),
                "file": doaj_file.name,
                "size_bytes": doaj_file.stat().st_size,
            },
            "fx_rates": {
                "endpoint": FX_API_URL,
                "provider_last_update_utc": fx_raw["time_last_update_utc"],
                "currencies": len(fx_raw["rates"]),
                "file": fx_file.name,
            },
        },
    }
    path = manifest_path(snapshot_date)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_file = path.with_name(path.name + ".tmp")
    tmp_file.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    os.replace(tmp_file, path)
    print(f"📝 Manifest snapshot tersimpan: {path}")
    return manifest


def extract_snapshot(snapshot_date):
    """Ketiga extract + manifest secara berurutan (dipakai di walkthrough Bagian 1).
    Di run_pipeline(), keempat langkah ini dijalankan sebagai task terpisah sesuai DAG."""
    if manifest_path(snapshot_date).exists():
        print(f"✅ Snapshot {snapshot_date} sudah lengkap (manifest ada). Extract dilewati.")
        return json.loads(manifest_path(snapshot_date).read_text(encoding="utf-8"))

    config = snapshot_config(snapshot_date)
    print(f"📡 Crossref ({config['quota_per_year']:,} artikel × {len(config['years'])} tahun)...")
    extract_crossref(snapshot_date)
    print(f"📡 DOAJ...")
    extract_doaj(snapshot_date)
    print(f"📡 Kurs...")
    extract_fx(snapshot_date)
    return write_manifest(snapshot_date)

### ▶️ Jalankan extract untuk snapshot ini

Jika sel ini berhenti dengan `IncompleteSnapshotError` (mis. koneksi putus), **jalankan ulang sel yang sama**: bagian yang sudah lengkap dilewati dan ekstraksi dilanjutkan.

In [10]:
print("=" * 60)
print(f"📥 EXTRACT SNAPSHOT {SNAPSHOT_DATE}")
print("=" * 60)
start_extract = time.time()
snapshot_manifest = extract_snapshot(SNAPSHOT_DATE)
print(f"\n✅ Selesai dalam {time.time() - start_extract:.1f} detik")
for name, info in snapshot_manifest["sources"].items():
    count = info.get("records", info.get("currencies"))
    print(f"   • {name:9s}: {count:,}")

📥 EXTRACT SNAPSHOT 2026-11-07
📡 Crossref (1,250 artikel × 16 tahun)...
   📄 2011 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2012 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2013 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2014 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2015 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2016 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2017 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2018 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2019 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2020 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2021 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2022 | 13 request | 1,250 artikel | 0 DOI duplikat dilewati | ✅
   📄 2023 | 13 request | 1,250 artikel | 1 DOI duplikat dilewati | ✅
   📄 2024 | 13 request | 1,250 a

## 1D. Membaca Raw Snapshot

Raw dibaca kembali dari file snapshot (bukan dari variabel di memori), sehingga tahap selanjutnya selalu bisa diulang dari snapshot yang sama.

Hasil *flatten* Crossref disimpan di `staging/`, **bukan raw**, karena nested object sudah diubah (mis. hanya license & link pertama). Informasi lengkapnya tetap bisa ditelusuri lewat DOI ke file `.jsonl.gz`.

In [11]:
def flatten_crossref_item(item, sample_year):
    """Flatten satu item Crossref menjadi dictionary datar.
    Nested objects diubah ke string/nilai sederhana."""
    row = {"sample_year": sample_year}                   # tahun strata saat ekstraksi

    # Scalar / simple fields
    for key in ["DOI", "URL", "type", "publisher", "source", "prefix",
                "member", "volume", "issue", "page", "article-number",
                "reference-count", "references-count", "is-referenced-by-count",
                "score", "language"]:
        row[key] = item.get(key, None)

    # List[str] → join dengan semicolon
    row["title"] = "; ".join(item.get("title", []))
    row["subtitle"] = "; ".join(item.get("subtitle", []))
    row["container-title"] = "; ".join(item.get("container-title", []))
    row["short-container-title"] = "; ".join(item.get("short-container-title", []))
    row["subject"] = "; ".join(item.get("subject", []))
    row["alternative-id"] = "; ".join(item.get("alternative-id", []))

    # ISSN (bisa ada multiple)
    issn_list = item.get("ISSN", [])
    row["ISSN"] = "; ".join(issn_list) if issn_list else None
    row["ISSN_first"] = issn_list[0] if issn_list else None

    # ISSN type → ISSN_print, ISSN_electronic
    for it in item.get("issn-type", []):
        row[f"ISSN_{it.get('type', 'unknown')}"] = it.get("value", None)

    # Author → flatten
    authors = item.get("author", [])
    author_names = [f"{a.get('given', '')} {a.get('family', '')}".strip() for a in authors]
    author_names = [name for name in author_names if name]
    row["author"] = "; ".join(author_names) if author_names else None
    row["author_count"] = len(authors)

    # Date fields
    for date_key in ["issued", "published-print", "published-online",
                     "created", "deposited", "published"]:
        date_obj = item.get(date_key, {})
        parts = date_obj.get("date-parts", [[None]])
        dp = parts[0] if parts and parts[0] else []
        row[f"{date_key}_year"] = dp[0] if len(dp) > 0 else None
        row[f"{date_key}_month"] = dp[1] if len(dp) > 1 else None
        row[f"{date_key}_day"] = dp[2] if len(dp) > 2 else None
        if "date-time" in date_obj:
            row[f"{date_key}_datetime"] = date_obj["date-time"]

    # License (pertama; daftar lengkap tetap ada di raw)
    licenses = item.get("license", [])
    row["license_url"] = licenses[0].get("URL") if licenses else None
    row["license_delay_days"] = licenses[0].get("delay-in-days") if licenses else None
    row["license_content_version"] = licenses[0].get("content-version") if licenses else None

    # Funder
    funders = item.get("funder", [])
    funder_names = [f.get("name", "") for f in funders]
    row["funder"] = "; ".join(funder_names) if funder_names else None
    row["funder_count"] = len(funders)

    # Content-domain
    cd = item.get("content-domain", {})
    row["content_domain"] = "; ".join(cd.get("domain", []))
    row["crossmark_restriction"] = cd.get("crossmark-restriction", None)

    # Link (pertama)
    links = item.get("link", [])
    row["link_url"] = links[0].get("URL") if links else None
    row["link_content_type"] = links[0].get("content-type") if links else None
    return row


def load_raw(snapshot_date):
    """Baca snapshot raw yang LENGKAP (wajib punya manifest) menjadi DataFrame."""
    if not manifest_path(snapshot_date).exists():
        raise IncompleteSnapshotError(f"Snapshot {snapshot_date} belum lengkap (manifest tidak ada).")

    # Crossref: .jsonl.gz → flatten
    rows = []
    crossref_dir = raw_snapshot_dir("crossref", snapshot_date)
    for year in snapshot_config(snapshot_date)["years"]:
        with gzip.open(crossref_dir / f"crossref_{year}.jsonl.gz", "rt", encoding="utf-8") as f:
            rows.extend(flatten_crossref_item(json.loads(line), year) for line in f)
    crossref_raw = pd.DataFrame(rows)
    for col in ["ISSN_print", "ISSN_electronic"]:        # kolom dinamis: pastikan selalu ada
        if col not in crossref_raw.columns:
            crossref_raw[col] = None
    crossref_raw.to_csv(STAGING_DIR / f"crossref_flat_{snapshot_date}.csv", index=False, encoding="utf-8")

    # DOAJ: CSV apa adanya
    doaj_raw = pd.read_csv(raw_snapshot_dir("doaj", snapshot_date) / "doaj_journals.csv",
                           encoding="utf-8", encoding_errors="replace", low_memory=False)

    # Kurs: JSON → tabel
    fx_raw = json.loads((raw_snapshot_dir("fx_rates", snapshot_date) / "fx_rates_usd.json").read_text(encoding="utf-8"))
    fx_date = pd.to_datetime(fx_raw["time_last_update_utc"], utc=True).strftime("%Y-%m-%d")
    fx_table = pd.DataFrame({"currency": list(fx_raw["rates"].keys()),
                             "rate_per_usd": list(fx_raw["rates"].values())})   # 1 USD = rate unit
    fx_table["fx_date"] = fx_date
    fx_table["base_currency"] = fx_raw["base_code"]
    fx_table["source"] = "ExchangeRate-API (open.er-api.com)"
    return crossref_raw, doaj_raw, fx_table, fx_date


df_crossref_raw, df_doaj_raw, df_fx, fx_date = load_raw(SNAPSHOT_DATE)
print(f"✅ Raw snapshot {SNAPSHOT_DATE} dibaca:")
print(f"   Crossref (flatten) : {df_crossref_raw.shape[0]:,} baris × {df_crossref_raw.shape[1]} kolom")
print(f"   DOAJ               : {df_doaj_raw.shape[0]:,} baris × {df_doaj_raw.shape[1]} kolom")
print(f"   Kurs               : {len(df_fx)} mata uang (tanggal {fx_date})")
print(f"   Contoh kurs (1 USD =): " + ", ".join(
    f"{c} {r:,.2f}" for c, r in zip(df_fx["currency"], df_fx["rate_per_usd"])
    if c in ["EUR", "GBP", "IDR", "IRR", "INR", "BRL"]))

✅ Raw snapshot 2026-11-07 dibaca:
   Crossref (flatten) : 20,000 baris × 58 kolom
   DOAJ               : 23,578 baris × 52 kolom
   Kurs               : 166 mata uang (tanggal 2026-11-06)
   Contoh kurs (1 USD =): BRL 5.47, EUR 0.89, GBP 0.74, IDR 18,079.99, INR 94.54, IRR 1,798,420.93


## 1E. Raw Validation

Task terpisah setelah ingestion: raw yang **dibaca kembali dari file** dicek dulu sebelum masuk transformasi. Jika ada cek yang gagal, pipeline berhenti di sini (tidak ada gunanya mentransformasi raw yang rusak/tidak lengkap).

| Sumber | Cek |
|--------|-----|
| Crossref | jumlah artikel tiap tahun = kuota; total = manifest; DOI tidak kosong; field `issn-type` tersedia (kolom ISSN hasil flatten ada) |
| DOAJ | jumlah baris = manifest & ≥ batas minimum; kolom kunci (judul, Print ISSN, E-ISSN, license, APC) ada; judul jurnal tidak kosong |
| Kurs | basis USD; jumlah mata uang ≥ batas minimum; semua kurs > 0; ada kurs USD = 1 |

In [12]:
def validate_raw(snapshot_date, crossref_raw, doaj_raw, fx_table, verbose=True):
    """Task raw validation. Return DataFrame cek; raise IncompleteSnapshotError jika ada yang gagal."""
    manifest = json.loads(manifest_path(snapshot_date).read_text(encoding="utf-8"))
    checks = []

    def check(source, name, actual, expected):
        checks.append({"sumber": source, "cek": name, "actual": actual,
                       "expected": expected, "lolos": actual == expected})

    per_year = crossref_raw["sample_year"].value_counts()
    config = snapshot_config(snapshot_date)
    years_off_quota = sorted(y for y in config["years"] if per_year.get(y, 0) != config["quota_per_year"])
    check("crossref", "tahun dengan jumlah artikel ≠ kuota", len(years_off_quota), 0)
    check("crossref", "total artikel = manifest", len(crossref_raw), manifest["sources"]["crossref"]["records"])
    check("crossref", "DOI kosong", int(crossref_raw["DOI"].isna().sum()), 0)
    check("crossref", "kolom ISSN_print & ISSN_electronic ada",
          {"ISSN_print", "ISSN_electronic"} <= set(crossref_raw.columns), True)

    required_cols = {"Journal title", DOAJ_PRINT_ISSN_COL, DOAJ_ONLINE_ISSN_COL, "Journal license", "APC", "APC amount"}
    check("doaj", "jumlah baris = manifest", len(doaj_raw), manifest["sources"]["doaj"]["records"])
    check("doaj", f"jumlah baris ≥ {DOAJ_MIN_ROWS:,}", len(doaj_raw) >= DOAJ_MIN_ROWS, True)
    check("doaj", "kolom kunci hilang", sorted(required_cols - set(doaj_raw.columns)), [])
    check("doaj", "judul jurnal kosong", int(doaj_raw["Journal title"].isna().sum()), 0)

    check("fx_rates", "basis mata uang", fx_table["base_currency"].iloc[0], "USD")
    check("fx_rates", f"jumlah mata uang ≥ {FX_MIN_CURRENCIES}", len(fx_table) >= FX_MIN_CURRENCIES, True)
    check("fx_rates", "kurs ≤ 0", int((fx_table["rate_per_usd"] <= 0).sum()), 0)
    check("fx_rates", "kurs USD = 1", dict(zip(fx_table["currency"], fx_table["rate_per_usd"])).get("USD"), 1)

    report = pd.DataFrame(checks)
    if verbose:
        display(report)
    failed = report[~report["lolos"]]
    if len(failed) > 0:
        if not verbose:
            display(failed)
        raise IncompleteSnapshotError(f"{len(failed)} cek raw validation gagal → transformasi tidak dijalankan.")
    return report


print("=" * 60)
print(f"🧪 RAW VALIDATION snapshot {SNAPSHOT_DATE}")
print("=" * 60)
raw_report = validate_raw(SNAPSHOT_DATE, df_crossref_raw, df_doaj_raw, df_fx)
print(f"✅ {len(raw_report)}/{len(raw_report)} cek raw lolos")

🧪 RAW VALIDATION snapshot 2026-11-07


,sumber,cek,actual,expected,lolos
0,crossref,tahun dengan jumlah artikel ≠ kuota,0,0,True
1,crossref,total artikel = manifest,20000,20000,True
2,crossref,DOI kosong,0,0,True
3,crossref,kolom ISSN_print & ISSN_electronic ada,True,True,True
4,doaj,jumlah baris = manifest,23578,23578,True
5,doaj,"jumlah baris ≥ 20,000",True,True,True
6,doaj,kolom kunci hilang,[],[],True
7,doaj,judul jurnal kosong,0,0,True
8,fx_rates,basis mata uang,USD,USD,True
9,fx_rates,jumlah mata uang ≥ 100,True,True,True


✅ 12/12 cek raw lolos


---

# 🔍 BAGIAN 2 — DATA PROFILING

Analisis menyeluruh terhadap **data RAW mentah** (semua kolom, sebelum dipotong).

## 2A. Profiling Crossref (RAW)

In [13]:
print("=" * 60)
print("🔍 PROFILING: Crossref RAW (Source 1)")
print("=" * 60)

df = df_crossref_raw.copy()

# (a) Shape
print(f"\n📊 (a) Shape: {df.shape[0]:,} baris × {df.shape[1]} kolom")

🔍 PROFILING: Crossref RAW (Source 1)

📊 (a) Shape: 20,000 baris × 58 kolom


In [14]:
# (b) Nama kolom
print(f"📋 (b) Nama kolom ({len(df.columns)} kolom):")
for i, col in enumerate(df.columns):
    print(f"     {i:2d}. {col}")

📋 (b) Nama kolom (58 kolom):
      0. sample_year
      1. DOI
      2. URL
      3. type
      4. publisher
      5. source
      6. prefix
      7. member
      8. volume
      9. issue
     10. page
     11. article-number
     12. reference-count
     13. references-count
     14. is-referenced-by-count
     15. score
     16. language
     17. title
     18. subtitle
     19. container-title
     20. short-container-title
     21. subject
     22. alternative-id
     23. ISSN
     24. ISSN_first
     25. ISSN_print
     26. author
     27. author_count
     28. issued_year
     29. issued_month
     30. issued_day
     31. published-print_year
     32. published-print_month
     33. published-print_day
     34. published-online_year
     35. published-online_month
     36. published-online_day
     37. created_year
     38. created_month
     39. created_day
     40. created_datetime
     41. deposited_year
     42. deposited_month
     43. deposited_day
     44. deposited_datetim

In [15]:
# (c) Tipe data per kolom
print(f"🔠 (c) Tipe data per kolom:")
dtype_summary = df.dtypes.value_counts()
print(f"     Ringkasan: {dtype_summary.to_dict()}")
print(f"\n     Detail:")
for col in df.columns:
    print(f"       {col:40s} → {df[col].dtype}")

🔠 (c) Tipe data per kolom:
     Ringkasan: {dtype('O'): 31, dtype('int64'): 14, dtype('float64'): 12, dtype('bool'): 1}

     Detail:
       sample_year                              → int64
       DOI                                      → object
       URL                                      → object
       type                                     → object
       publisher                                → object
       source                                   → object
       prefix                                   → object
       member                                   → object
       volume                                   → object
       issue                                    → object
       page                                     → object
       article-number                           → object
       reference-count                          → int64
       references-count                         → int64
       is-referenced-by-count                   → int64
       score   

In [16]:
# (d) Missing value per kolom
print(f"❓ (d) Missing values per kolom:")
missing_cr = pd.DataFrame({
    "kolom": df.columns,
    "null_count": [df[c].isna().sum() for c in df.columns],
    "null_pct": [df[c].isna().sum() / len(df) * 100 for c in df.columns]
}).sort_values("null_count", ascending=False)

# Tampilkan kolom yang punya missing values
missing_cr_show = missing_cr[missing_cr["null_count"] > 0]
print(f"     Kolom dengan missing values: {len(missing_cr_show)} dari {len(df.columns)}")
display(missing_cr_show)

❓ (d) Missing values per kolom:
     Kolom dengan missing values: 28 dari 58


,kolom,null_count,null_pct
11,article-number,17325,86.625
51,funder,16304,81.520
33,published-print_day,15633,78.165
36,published-online_day,8784,43.920
30,issued_day,8494,42.470
47,published_day,8494,42.470
50,license_content_version,8348,41.740
48,license_url,8348,41.740
49,license_delay_days,8348,41.740
35,published-online_month,8347,41.735


In [17]:
# (e) Duplicate rows
dup_rows = df.duplicated().sum()
dup_doi = df["DOI"].duplicated().sum()
print(f"♻️  (e) Duplikat:")
print(f"     Duplicate rows (seluruh kolom): {dup_rows}")
print(f"     Duplicate DOI: {dup_doi}")

♻️  (e) Duplikat:
     Duplicate rows (seluruh kolom): 0
     Duplicate DOI: 0


In [18]:
# (f) Uniqueness integration key (ISSN)
print(f"🔑 (f) Uniqueness integration key:")

# ISSN_first = ISSN pertama dari list
issn_vals = df["ISSN_first"].dropna()
print(f"     ISSN_first (ISSN pertama per artikel):")
print(f"       Total non-null : {len(issn_vals):,}")
print(f"       Missing        : {df['ISSN_first'].isna().sum():,} ({df['ISSN_first'].isna().sum()/len(df)*100:.1f}%)")
print(f"       Unik           : {issn_vals.nunique():,}")
print(f"       (Catatan: beberapa artikel berasal dari jurnal yang sama, ")
print(f"        sehingga ISSN wajar berulang)")

# Cek juga ISSN_print dan ISSN_electronic jika ada
for col in ["ISSN_print", "ISSN_electronic"]:
    if col in df.columns:
        vals = df[col].dropna()
        print(f"\n     {col}:")
        print(f"       Total non-null : {len(vals):,}")
        print(f"       Missing        : {df[col].isna().sum():,}")
        print(f"       Unik           : {vals.nunique():,}")

🔑 (f) Uniqueness integration key:
     ISSN_first (ISSN pertama per artikel):
       Total non-null : 19,821
       Missing        : 179 (0.9%)
       Unik           : 12,015
       (Catatan: beberapa artikel berasal dari jurnal yang sama, 
        sehingga ISSN wajar berulang)

     ISSN_print:
       Total non-null : 15,535
       Missing        : 4,465
       Unik           : 9,473

     ISSN_electronic:
       Total non-null : 14,161
       Missing        : 5,839
       Unik           : 8,894


In [19]:
# (g) Min/Max date
print(f"📅 (g) Rentang tanggal:")
for date_prefix in ["issued", "published-print", "published-online", "created"]:
    year_col = f"{date_prefix}_year"
    if year_col in df.columns:
        yr = df[year_col].dropna()
        if len(yr) > 0:
            print(f"     {date_prefix}:")
            print(f"       Min year: {int(yr.min())} | Max year: {int(yr.max())} | "
                  f"Non-null: {len(yr):,} | Missing: {df[year_col].isna().sum():,}")

📅 (g) Rentang tanggal:
     issued:
       Min year: 2011 | Max year: 2026 | Non-null: 20,000 | Missing: 0
     published-print:
       Min year: 2011 | Max year: 2027 | Non-null: 14,308 | Missing: 5,692
     published-online:
       Min year: 2011 | Max year: 2026 | Non-null: 12,441 | Missing: 7,559
     created:
       Min year: 2004 | Max year: 2026 | Non-null: 20,000 | Missing: 0


In [20]:
# (h) Contoh nilai kategori penting
print(f"📎 (h) Contoh nilai kategori penting:")

print(f"\n     type (jenis dokumen):")
print(f"       {df['type'].value_counts().head(5).to_dict()}")

print(f"\n     publisher (top 10):")
for k, v in df['publisher'].value_counts().head(10).items():
    print(f"       {k}: {v}")

print(f"\n     source:")
print(f"       {df['source'].value_counts().to_dict()}")

print(f"\n     Distribusi tahun (sample_year = strata ekstraksi, issued_year = tahun terbit di metadata):")
year_dist = pd.DataFrame({
    "sample_year": df["sample_year"].value_counts(),
    "issued_year": df["issued_year"].value_counts(),
}).fillna(0).astype(int).sort_index()
year_dist.index = year_dist.index.astype(int)
display(year_dist)

# Konsistensi: filter Crossref memakai tanggal 'published', sedangkan issued_year dari field 'issued'
issued = pd.to_numeric(df["issued_year"], errors="coerce")
beda_tahun = issued.notna() & (issued != df["sample_year"])
print(f"     Artikel dengan issued_year ≠ sample_year: {beda_tahun.sum():,} ({beda_tahun.mean()*100:.1f}%)")

📎 (h) Contoh nilai kategori penting:

     type (jenis dokumen):
       {'journal-article': 20000}

     publisher (top 10):
       Elsevier BV: 3484
       Springer Science and Business Media LLC: 1723
       Wiley: 1261
       Informa UK Limited: 697
       Oxford University Press (OUP): 510
       MDPI AG: 495
       SAGE Publications: 482
       Ovid Technologies (Wolters Kluwer Health): 450
       Institute of Electrical and Electronics Engineers (IEEE): 286
       American Chemical Society (ACS): 269

     source:
       {'Crossref': 20000}

     Distribusi tahun (sample_year = strata ekstraksi, issued_year = tahun terbit di metadata):


,sample_year,issued_year
2011,1250,1250
2012,1250,1250
2013,1250,1250
2014,1250,1250
2015,1250,1250
2016,1250,1250
2017,1250,1250
2018,1250,1250
2019,1250,1250
2020,1250,1250


     Artikel dengan issued_year ≠ sample_year: 0 (0.0%)


In [21]:
# (i) Nilai kategori tidak konsisten
print(f"⚠️  (i) Cek konsistensi kategori:")

# Format ISSN
issn_check = df["ISSN_first"].dropna()
issn_with_dash = issn_check[issn_check.str.contains("-", na=False)]
issn_without_dash = issn_check[~issn_check.str.contains("-", na=False)]
print(f"     Format ISSN: {len(issn_with_dash):,} pakai strip, {len(issn_without_dash):,} tanpa strip")
if len(issn_without_dash) > 0:
    print(f"     ⚠️ Contoh ISSN tanpa strip: {issn_without_dash.head(5).tolist()}")
else:
    print(f"     ✅ Semua ISSN konsisten (format XXXX-XXXX)")

# Cek publisher case sensitivity
pub_vals = df["publisher"].dropna()
pub_lower = pub_vals.str.lower()
pub_unique = pub_vals.nunique()
pub_lower_unique = pub_lower.nunique()
if pub_unique != pub_lower_unique:
    print(f"     ⚠️ Publisher: {pub_unique} unik (case-sensitive) vs {pub_lower_unique} (case-insensitive)")
    print(f"       Ada perbedaan kapitalisasi pada {pub_unique - pub_lower_unique} publisher")
else:
    print(f"     ✅ Publisher: konsisten ({pub_unique} unik)")

⚠️  (i) Cek konsistensi kategori:
     Format ISSN: 19,821 pakai strip, 0 tanpa strip
     ✅ Semua ISSN konsisten (format XXXX-XXXX)
     ⚠️ Publisher: 3575 unik (case-sensitive) vs 3574 (case-insensitive)
       Ada perbedaan kapitalisasi pada 1 publisher


In [22]:
# (j) Kemungkinan outlier
print(f"📈 (j) Kemungkinan outlier:")

yr = pd.to_numeric(df["issued_year"], errors="coerce").dropna()
if yr.min() < MIN_YEAR:
    print(f"     ⚠️ Artikel dengan tahun < {MIN_YEAR}: {(yr < MIN_YEAR).sum():,}")
else:
    print(f"     ✅ Semua artikel dalam rentang {MIN_YEAR}+")
if yr.max() > CURRENT_YEAR:
    print(f"     ⚠️ Artikel tahun > {CURRENT_YEAR} (masa depan): {(yr > CURRENT_YEAR).sum():,}")
else:
    print(f"     ✅ Tidak ada artikel bertahun > {CURRENT_YEAR}")

cite = df["is-referenced-by-count"].dropna()
q99 = cite.quantile(0.99)
print(f"\n     Sitasi (is-referenced-by-count):")
print(f"       Mean: {cite.mean():.1f} | Median: {cite.median():.0f} | Max: {cite.max():,.0f}")
print(f"       P99: {q99:,.0f} | Artikel di atas P99: {(cite > q99).sum():,}")

📈 (j) Kemungkinan outlier:
     ✅ Semua artikel dalam rentang 2011+
     ✅ Tidak ada artikel bertahun > 2026

     Sitasi (is-referenced-by-count):
       Mean: 15.3 | Median: 1 | Max: 5,982
       P99: 165 | Artikel di atas P99: 200


In [23]:
print("=" * 60)
print("🔍 AUDIT KUALITAS DATA: CROSSREF (RAW)")
print("=" * 60)

cr = df_crossref_raw

# 1. Deteksi anomali tahun terbit
yr = pd.to_numeric(cr["issued_year"], errors="coerce")
mask_anomali_tahun = yr.isna() | (yr < MIN_YEAR) | (yr > CURRENT_YEAR)
anomali_tahun = cr[mask_anomali_tahun]
print(f"⚠️  Artikel dengan tahun terbit kosong/di luar {MIN_YEAR}–{CURRENT_YEAR}: {len(anomali_tahun):,}")
if len(anomali_tahun) > 0:
    display(anomali_tahun[["DOI", "title", "container-title", "issued_year",
                           "published-print_year", "published-online_year"]].head(10))

# 2. Kelengkapan & format kedua integration key (Print ISSN & Online ISSN)
print()
for col in ["ISSN_print", "ISSN_electronic"]:
    vals = cr[col].dropna().astype(str).str.strip().str.upper()
    bad = vals[~vals.str.match(ISSN_REGEX)]
    print(f"🔑 {col:16s}: terisi {len(vals):,} | kosong {cr[col].isna().sum():,} | format tidak standar {len(bad)}")
    if len(bad) > 0:
        print(f"     Contoh: {bad.head(5).tolist()}")

tanpa_kedua = cr["ISSN_print"].isna() & cr["ISSN_electronic"].isna()
print(f"\n⚠️  Artikel tanpa Print ISSN maupun Online ISSN (pasti gagal join): {tanpa_kedua.sum():,}")
issn_tanpa_tipe = tanpa_kedua & cr["ISSN"].notna()
print(f"   └ di antaranya punya ISSN tetapi tanpa label tipe (print/electronic): {issn_tanpa_tipe.sum():,}")

🔍 AUDIT KUALITAS DATA: CROSSREF (RAW)
⚠️  Artikel dengan tahun terbit kosong/di luar 2011–2026: 0

🔑 ISSN_print      : terisi 15,535 | kosong 4,465 | format tidak standar 0
🔑 ISSN_electronic : terisi 14,161 | kosong 5,839 | format tidak standar 0

⚠️  Artikel tanpa Print ISSN maupun Online ISSN (pasti gagal join): 179
   └ di antaranya punya ISSN tetapi tanpa label tipe (print/electronic): 0


## 2B. Profiling DOAJ (RAW)

In [24]:
print("=" * 60)
print("🔍 PROFILING: DOAJ RAW (Source 2)")
print("=" * 60)

df = df_doaj_raw.copy()
col_pissn = "Journal ISSN (print version)"
col_eissn = "Journal EISSN (online version)"

# (a) Shape
print(f"\n📊 (a) Shape: {df.shape[0]:,} baris × {df.shape[1]} kolom")

🔍 PROFILING: DOAJ RAW (Source 2)

📊 (a) Shape: 23,578 baris × 52 kolom


In [25]:
# (b) Nama kolom
print(f"📋 (b) Nama kolom ({len(df.columns)} kolom):")
for i, col in enumerate(df.columns):
    print(f"     {i:2d}. {col}")

📋 (b) Nama kolom (52 kolom):
      0. Journal title
      1. Journal URL
      2. URL in DOAJ
      3. When did the journal start to publish all content using an open license?
      4. Alternative title
      5. Journal ISSN (print version)
      6. Journal EISSN (online version)
      7. Keywords
      8. Languages in which the journal accepts manuscripts
      9. Publisher
     10. Country of publisher
     11. Other organisation
     12. Country of other organisation
     13. Journal license
     14. License attributes
     15. URL for license terms
     16. Machine-readable CC licensing information embedded or displayed in articles
     17. Author holds copyright without restrictions
     18. Copyright information URL
     19. Review process
     20. Review process information URL
     21. Journal plagiarism screening policy
     22. URL for journal's aims & scope
     23. URL for the Editorial Board page
     24. URL for journal's instructions for authors
     25. Average number o

In [26]:
# (c) Tipe data per kolom (SEMUA kolom)
print(f"🔠 (c) Tipe data per kolom:")
dtype_summary = df.dtypes.value_counts()
print(f"     Ringkasan: {dtype_summary.to_dict()}")
print(f"\n     Detail:")
for col in df.columns:
    print(f"       {col:65s} → {df[col].dtype}")

🔠 (c) Tipe data per kolom:
     Ringkasan: {dtype('O'): 49, dtype('int64'): 3}

     Detail:
       Journal title                                                     → object
       Journal URL                                                       → object
       URL in DOAJ                                                       → object
       When did the journal start to publish all content using an open license? → int64
       Alternative title                                                 → object
       Journal ISSN (print version)                                      → object
       Journal EISSN (online version)                                    → object
       Keywords                                                          → object
       Languages in which the journal accepts manuscripts                → object
       Publisher                                                         → object
       Country of publisher                                              → object

In [27]:
# (d) Missing value per kolom (SEMUA kolom)
print(f"❓ (d) Missing values per kolom:")
missing_dj = pd.DataFrame({
    "kolom": df.columns,
    "null_count": [df[c].isna().sum() for c in df.columns],
    "null_pct": [round(df[c].isna().sum() / len(df) * 100, 1) for c in df.columns]
}).sort_values("null_count", ascending=False)

display(missing_dj)

❓ (d) Missing values per kolom:


,kolom,null_count,null_pct
14,License attributes,23430,99.4
41,Continued By,23291,98.8
40,Continues,23149,98.2
34,Preservation Service: national library,20673,87.7
32,Other fees information URL,20530,87.1
30,Waiver policy information URL,17912,76.0
12,Country of other organisation,17498,74.2
37,URL for deposit policy,15635,66.3
28,APC amount,14731,62.5
36,Deposit policy directory,14574,61.8


In [28]:
# (e) Duplicate rows
dup_rows = df.duplicated().sum()
print(f"♻️  (e) Duplicate rows: {dup_rows}")

♻️  (e) Duplicate rows: 0


In [29]:
# (f) Uniqueness integration key (Print ISSN & E-ISSN)
print(f"🔑 (f) Uniqueness integration key:")

pissn = df[col_pissn].dropna()
eissn = df[col_eissn].dropna()

print(f"     Print ISSN:")
print(f"       Total non-null : {len(pissn):,}")
print(f"       Missing        : {df[col_pissn].isna().sum():,} ({df[col_pissn].isna().sum()/len(df)*100:.1f}%)")
print(f"       Unik           : {pissn.nunique():,}")
print(f"       Duplikat       : {pissn.duplicated().sum()}")
if pissn.duplicated().sum() > 0:
    dup_p = pissn[pissn.duplicated(keep=False)]
    for issn_val in dup_p.unique():
        journals = df[df[col_pissn] == issn_val]["Journal title"].tolist()
        print(f"       ⚠️ {issn_val}: {journals}")

print(f"\n     E-ISSN:")
print(f"       Total non-null : {len(eissn):,}")
print(f"       Missing        : {df[col_eissn].isna().sum():,} ({df[col_eissn].isna().sum()/len(df)*100:.1f}%)")
print(f"       Unik           : {eissn.nunique():,}")
print(f"       Duplikat       : {eissn.duplicated().sum()}")
if eissn.duplicated().sum() > 0:
    dup_e = eissn[eissn.duplicated(keep=False)]
    for issn_val in dup_e.unique():
        journals = df[df[col_eissn] == issn_val]["Journal title"].tolist()
        print(f"       ⚠️ {issn_val}: {journals}")

both_null = (df[col_pissn].isna() & df[col_eissn].isna()).sum()
print(f"\n     Jurnal tanpa Print ISSN maupun E-ISSN: {both_null}")

🔑 (f) Uniqueness integration key:
     Print ISSN:
       Total non-null : 13,493
       Missing        : 10,085 (42.8%)
       Unik           : 13,492
       Duplikat       : 1
       ⚠️ 2184-2574: ['Ethics, Politics & Society', 'Ethics, Politics & Society']

     E-ISSN:
       Total non-null : 22,603
       Missing        : 975 (4.1%)
       Unik           : 22,601
       Duplikat       : 2
       ⚠️ 2184-2582: ['Ethics, Politics & Society', 'Ethics, Politics & Society']
       ⚠️ 2500-154X: ['Известия вузов: Прикладная химия и биотехнология', 'Известия вузов: Инвестиции. Строительство. Недвижимость']

     Jurnal tanpa Print ISSN maupun E-ISSN: 0


In [30]:
# (g) Min/Max date
print(f"📅 (g) Rentang tanggal:")
date_cols = ["Added on Date", "Last updated Date", "Last Full Review Date", "Most Recent Article Added"]
for dc in date_cols:
    if dc in df.columns:
        vals = df[dc].dropna()
        if len(vals) > 0:
            parsed = pd.to_datetime(vals, format="mixed", dayfirst=True, errors="coerce").dropna()
            if len(parsed) > 0:
                print(f"     {dc}:")
                print(f"       Min: {parsed.min()} | Max: {parsed.max()} | Missing: {df[dc].isna().sum():,}")

# Open license start year
ol_col = "When did the journal start to publish all content using an open license?"
if ol_col in df.columns:
    ol = df[ol_col].dropna()
    print(f"\n     Open license start year:")
    print(f"       Min: {int(ol.min())} | Max: {int(ol.max())}")

📅 (g) Rentang tanggal:
     Added on Date:
       Min: 2002-05-01 20:12:48+00:00 | Max: 2026-10-05 17:28:16+00:00 | Missing: 0
     Last updated Date:
       Min: 2018-01-10 17:11:48+00:00 | Max: 2026-10-06 23:13:57+00:00 | Missing: 0
     Last Full Review Date:
       Min: 2014-05-09 00:00:00 | Max: 2026-10-06 00:00:00 | Missing: 2,895
     Most Recent Article Added:
       Min: 2004-05-31 00:00:00+00:00 | Max: 2026-10-06 23:30:59+00:00 | Missing: 3,868

     Open license start year:
       Min: 1900 | Max: 2026


In [31]:
# (h) Contoh nilai kategori penting
print(f"📎 (h) Contoh nilai kategori penting:")

print(f"\n     APC: {df['APC'].unique().tolist()}")

print(f"\n     Country of publisher (top 10):")
for k, v in df['Country of publisher'].value_counts().head(10).items():
    print(f"       {k}: {v:,}")

print(f"\n     Journal license (top 10):")
for k, v in df['Journal license'].value_counts().head(10).items():
    print(f"       {k}: {v:,}")

print(f"\n     Languages (top 5):")
lang_col = "Languages in which the journal accepts manuscripts"
if lang_col in df.columns:
    for k, v in df[lang_col].value_counts().head(5).items():
        print(f"       {k}: {v:,}")

print(f"\n     Review process:")
if "Review process" in df.columns:
    for k, v in df["Review process"].value_counts().head(5).items():
        print(f"       {k}: {v:,}")

📎 (h) Contoh nilai kategori penting:

     APC: ['Yes', 'No']

     Country of publisher (top 10):
       Indonesia: 2,730
       United Kingdom: 2,331
       Brazil: 1,395
       United States: 1,340
       Iran, Islamic Republic of: 1,096
       Spain: 1,031
       Poland: 1,013
       Switzerland: 821
       Türkiye: 716
       Russian Federation: 654

     Journal license (top 10):
       CC BY: 9,480
       CC BY-NC: 3,664
       CC BY-NC-ND: 3,254
       CC BY-NC-SA: 2,049
       CC BY-SA: 1,947
       CC BY, CC BY-NC-ND: 1,118
       CC BY, CC BY-NC: 674
       CC BY, CC BY-NC, CC BY-NC-ND: 644
       CC BY-ND: 235
       Publisher's own license: 147

     Languages (top 5):
       English: 11,280
       Spanish: 976
       English, Indonesian: 628
       Persian: 562
       Indonesian: 533

     Review process:
       Double anonymous peer review: 14,773
       Single anonymous peer review: 6,146
       Unspecified peer review: 1,530
       Editorial review, Double anonymous pe

In [32]:
# (i) Nilai kategori tidak konsisten
print(f"⚠️  (i) Cek konsistensi kategori:")

# License: single vs multiple
license_vals = df["Journal license"].dropna()
multi_license = license_vals[license_vals.str.contains(",", na=False)]
print(f"     Journal license:")
print(f"       Total varian unik: {license_vals.nunique()}")
print(f"       Multiple licenses (gabungan): {len(multi_license):,}")

# Country encoding
replacement_char = "\ufffd"
countries_bad = df["Country of publisher"].str.contains(replacement_char, na=False)
if countries_bad.sum() > 0:
    print(f"\n     ⚠️ Country dengan encoding rusak: {countries_bad.sum()}")
    print(f"       Contoh: {df[countries_bad]['Country of publisher'].unique()[:5].tolist()}")

# Date format inconsistency
for dc in ["Added on Date", "Last Full Review Date"]:
    if dc in df.columns:
        vals = df[dc].dropna()
        has_time = vals[vals.str.contains(" ", na=False)]
        no_time = vals[~vals.str.contains(" ", na=False)]
        if len(has_time) > 0 and len(no_time) > 0:
            print(f"\n     ⚠️ Format tanggal tidak konsisten di '{dc}':")
            print(f"       Dengan waktu  : {len(has_time):,} (contoh: {has_time.iloc[0]})")
            print(f"       Tanpa waktu   : {len(no_time):,} (contoh: {no_time.iloc[0]})")

⚠️  (i) Cek konsistensi kategori:
     Journal license:
       Total varian unik: 49
       Multiple licenses (gabungan): 2,798


In [33]:
# (j) Kemungkinan outlier
print(f"📈 (j) Kemungkinan outlier:")

weeks_col = "Average number of weeks between article submission and publication"
if weeks_col in df.columns:
    avg_weeks = df[weeks_col].dropna()
    q99 = avg_weeks.quantile(0.99)
    print(f"     {weeks_col}:")
    print(f"       Mean: {avg_weeks.mean():.1f} | Median: {avg_weeks.median():.0f} | Max: {avg_weeks.max()}")
    print(f"       P99: {q99:.0f} | Jurnal > P99: {(avg_weeks > q99).sum():,}")

art_col = "Number of Article Records"
if art_col in df.columns:
    art_count = df[art_col].dropna()
    q99 = art_count.quantile(0.99)
    print(f"\n     {art_col}:")
    print(f"       Mean: {art_count.mean():.0f} | Median: {art_count.median():.0f} | Max: {art_count.max():,}")
    print(f"       P99: {q99:,.0f} | Jurnal > P99: {(art_count > q99).sum():,}")

📈 (j) Kemungkinan outlier:
     Average number of weeks between article submission and publication:
       Mean: 17.4 | Median: 14 | Max: 100
       P99: 53 | Jurnal > P99: 209

     Number of Article Records:
       Mean: 584 | Median: 191 | Max: 371,799
       P99: 6,218 | Jurnal > P99: 236


In [34]:
print("=" * 60)
print("🔍 AUDIT KUALITAS DATA: MASTER DOAJ (RAW)")
print("=" * 60)

dj = df_doaj_raw

for col, label in [(col_pissn, "Print ISSN"), (col_eissn, "Online ISSN (E-ISSN)")]:
    vals = dj[col].dropna().astype(str).str.strip().str.upper()
    bad = vals[~vals.str.match(ISSN_REGEX)]
    dup = vals[vals.duplicated(keep=False)]
    print(f"🔑 {label}:")
    print(f"     Terisi: {len(vals):,} | Format tidak standar: {len(bad)} | Nilai duplikat: {vals.duplicated().sum()}")
    if len(dup) > 0:
        display(dj.loc[dup.index, ["Journal title", col_pissn, col_eissn]])

tanpa_kedua = dj[col_pissn].isna() & dj[col_eissn].isna()
print(f"\n⚠️  Jurnal DOAJ tanpa Print ISSN maupun E-ISSN: {tanpa_kedua.sum()}")

🔍 AUDIT KUALITAS DATA: MASTER DOAJ (RAW)
🔑 Print ISSN:
     Terisi: 13,493 | Format tidak standar: 0 | Nilai duplikat: 1


,Journal title,Journal ISSN (print version),Journal EISSN (online version)
13097,"Ethics, Politics & Society",2184-2574,2184-2582
14808,"Ethics, Politics & Society",2184-2574,2184-2582


🔑 Online ISSN (E-ISSN):
     Terisi: 22,603 | Format tidak standar: 0 | Nilai duplikat: 2


,Journal title,Journal ISSN (print version),Journal EISSN (online version)
13097,"Ethics, Politics & Society",2184-2574,2184-2582
14808,"Ethics, Politics & Society",2184-2574,2184-2582
17460,Известия вузов: Прикладная химия и биотехнология,2227-2925,2500-154X
23084,Известия вузов: Инвестиции. Строительство. Нед...,2227-2917,2500-154X



⚠️  Jurnal DOAJ tanpa Print ISSN maupun E-ISSN: 0


---

# ⚠️ BAGIAN 3 — DATA ISSUES

Identifikasi masalah-masalah pada data berdasarkan hasil profiling.

In [35]:
print("=" * 60)
print("⚠️  DATA ISSUES YANG DITEMUKAN")
print("=" * 60)
issue_count = 0

⚠️  DATA ISSUES YANG DITEMUKAN


In [36]:
# === DATA ISSUE #1: Missing Integration Key ===
issue_count += 1
print(f"\n{'='*60}")
print(f"🔴 Issue #{issue_count}: Missing Integration Key")
print(f"{'='*60}")

# Crossref
cr_no_issn = df_crossref_raw["ISSN_first"].isna().sum()
print(f"\n  📌 Crossref:")
print(f"     Artikel tanpa ISSN: {cr_no_issn:,} dari {len(df_crossref_raw):,} ({cr_no_issn/len(df_crossref_raw)*100:.1f}%)")
if cr_no_issn > 0 and cr_no_issn <= 20:
    no_issn_rows = df_crossref_raw[df_crossref_raw["ISSN_first"].isna()]
    print(f"     Contoh:")
    for _, row in no_issn_rows.head(5).iterrows():
        title_short = str(row['title'])[:60]
        print(f"       • {row['DOI']} — {title_short}...")

# DOAJ
doaj_no_pissn = df_doaj_raw[col_pissn].isna().sum()
doaj_no_eissn = df_doaj_raw[col_eissn].isna().sum()
print(f"\n  📌 DOAJ:")
print(f"     Tanpa Print ISSN : {doaj_no_pissn:,} ({doaj_no_pissn/len(df_doaj_raw)*100:.1f}%)")
print(f"     Tanpa E-ISSN     : {doaj_no_eissn:,} ({doaj_no_eissn/len(df_doaj_raw)*100:.1f}%)")
print(f"     Tanpa keduanya   : {(df_doaj_raw[col_pissn].isna() & df_doaj_raw[col_eissn].isna()).sum()}")
print(f"\n  💡 Dampak: artikel tanpa ISSN tidak bisa di-join ke DOAJ.")
print(f"  🛠️ Penanganan: TIDAK dihapus, ditandai match_status = 'no_issn'.")


🔴 Issue #1: Missing Integration Key

  📌 Crossref:
     Artikel tanpa ISSN: 179 dari 20,000 (0.9%)

  📌 DOAJ:
     Tanpa Print ISSN : 10,085 (42.8%)
     Tanpa E-ISSN     : 975 (4.1%)
     Tanpa keduanya   : 0

  💡 Dampak: artikel tanpa ISSN tidak bisa di-join ke DOAJ.
  🛠️ Penanganan: TIDAK dihapus, ditandai match_status = 'no_issn'.


In [37]:
# === DATA ISSUE #2: Duplicate Integration Key ===
issue_count += 1
print(f"\n{'='*60}")
print(f"🔴 Issue #{issue_count}: Duplicate Integration Key di DOAJ")
print(f"{'='*60}")

pissn_vals = df_doaj_raw[col_pissn].dropna()
eissn_vals = df_doaj_raw[col_eissn].dropna()
dup_p = pissn_vals.duplicated().sum()
dup_e = eissn_vals.duplicated().sum()

print(f"\n  📌 Print ISSN duplikat: {dup_p}")
if dup_p > 0:
    for issn_val in pissn_vals[pissn_vals.duplicated(keep=False)].unique():
        journals = df_doaj_raw[df_doaj_raw[col_pissn] == issn_val]["Journal title"].tolist()
        print(f"     ⚠️ {issn_val}: {journals}")

print(f"\n  📌 E-ISSN duplikat: {dup_e}")
if dup_e > 0:
    for issn_val in eissn_vals[eissn_vals.duplicated(keep=False)].unique():
        journals = df_doaj_raw[df_doaj_raw[col_eissn] == issn_val]["Journal title"].tolist()
        print(f"     ⚠️ {issn_val}: {journals}")

print(f"\n  💡 Dampak: join bisa menghasilkan duplikasi baris atau salah jurnal.")
print(f"  🛠️ Penanganan: entri identik → simpan satu (dicatat di log);")
print(f"     ISSN dipakai jurnal berbeda → tidak dipakai sebagai key, ditandai *_is_ambiguous.")


🔴 Issue #2: Duplicate Integration Key di DOAJ

  📌 Print ISSN duplikat: 1
     ⚠️ 2184-2574: ['Ethics, Politics & Society', 'Ethics, Politics & Society']

  📌 E-ISSN duplikat: 2
     ⚠️ 2184-2582: ['Ethics, Politics & Society', 'Ethics, Politics & Society']
     ⚠️ 2500-154X: ['Известия вузов: Прикладная химия и биотехнология', 'Известия вузов: Инвестиции. Строительство. Недвижимость']

  💡 Dampak: join bisa menghasilkan duplikasi baris atau salah jurnal.
  🛠️ Penanganan: entri identik → simpan satu (dicatat di log);
     ISSN dipakai jurnal berbeda → tidak dipakai sebagai key, ditandai *_is_ambiguous.


In [38]:
# === DATA ISSUE #3: Missing Values di Kolom Penting ===
issue_count += 1
print(f"\n{'='*60}")
print(f"🔴 Issue #{issue_count}: Missing Values di Kolom Penting")
print(f"{'='*60}")

print(f"\n  📌 Crossref (kolom dengan missing > 0):")
for col in df_crossref_raw.columns:
    null_ct = df_crossref_raw[col].isna().sum()
    if null_ct > 0:
        pct = null_ct / len(df_crossref_raw) * 100
        print(f"     {col:40s}: {null_ct:,} ({pct:.1f}%)")

print(f"\n  📌 DOAJ (kolom dengan missing > 50%):")
for col in df_doaj_raw.columns:
    null_ct = df_doaj_raw[col].isna().sum()
    pct = null_ct / len(df_doaj_raw) * 100
    if pct > 50:
        print(f"     {col:65s}: {null_ct:,} ({pct:.1f}%)")


🔴 Issue #3: Missing Values di Kolom Penting

  📌 Crossref (kolom dengan missing > 0):
     publisher                               : 10 (0.1%)
     member                                  : 10 (0.1%)
     volume                                  : 1,398 (7.0%)
     issue                                   : 3,813 (19.1%)
     page                                    : 3,300 (16.5%)
     article-number                          : 17,325 (86.6%)
     language                                : 6,310 (31.6%)
     ISSN                                    : 179 (0.9%)
     ISSN_first                              : 179 (0.9%)
     ISSN_print                              : 4,465 (22.3%)
     author                                  : 1,122 (5.6%)
     issued_month                            : 2,400 (12.0%)
     issued_day                              : 8,494 (42.5%)
     published-print_year                    : 5,692 (28.5%)
     published-print_month                   : 7,519 (37.6%)
     publishe

In [39]:
# === DATA ISSUE #4: Tahun Terbit Tidak Valid (Outlier Tanggal) ===
issue_count += 1
print(f"\n{'='*60}")
print(f"🔴 Issue #{issue_count}: Tahun Terbit Tidak Valid (Outlier Tanggal)")
print(f"{'='*60}")

n_anomali = len(anomali_tahun)
print(f"\n  📌 Crossref: {n_anomali:,} artikel dengan issued_year kosong/di luar {MIN_YEAR}–{CURRENT_YEAR}")
if n_anomali > 0:
    print(f"     Nilai tahun yang muncul : {sorted(anomali_tahun['issued_year'].dropna().astype(int).unique().tolist())}")
    print(f"     Jurnal terdampak        : {anomali_tahun['container-title'].value_counts().head(5).to_dict()}")
print(f"\n  💡 Dampak: tahun di luar rentang merusak analisis tren per tahun.")
print(f"  🛠️ Penanganan: baris TIDAK dihapus. Nilai asli di issued_year_raw,")
print(f"     issued_year dikosongkan, dan ditandai is_year_valid = False.")


🔴 Issue #4: Tahun Terbit Tidak Valid (Outlier Tanggal)

  📌 Crossref: 0 artikel dengan issued_year kosong/di luar 2011–2026

  💡 Dampak: tahun di luar rentang merusak analisis tren per tahun.
  🛠️ Penanganan: baris TIDAK dihapus. Nilai asli di issued_year_raw,
     issued_year dikosongkan, dan ditandai is_year_valid = False.


In [40]:
# === DATA ISSUE #5: Inconsistent Category Values (License Multi-Value) ===
issue_count += 1
print(f"\n{'='*60}")
print(f"🔴 Issue #{issue_count}: Inconsistent Category Values (License)")
print(f"{'='*60}")

license_vals = df_doaj_raw["Journal license"].dropna()
multi_license = license_vals[license_vals.str.contains(",", na=False)]
print(f"\n  📌 DOAJ Journal license:")
print(f"     Total varian unik     : {license_vals.nunique()}")
print(f"     Multi-license (koma)  : {len(multi_license):,}")
print(f"     Contoh multi-license  : {multi_license.drop_duplicates().head(3).tolist()}")
print(f"\n  💡 Dampak: satu sel berisi beberapa lisensi (1 jurnal : N lisensi).")
print(f"  🛠️ Penanganan: string asli disimpan utuh + dipecah ke tabel jembatan journal_license.")


🔴 Issue #5: Inconsistent Category Values (License)

  📌 DOAJ Journal license:
     Total varian unik     : 49
     Multi-license (koma)  : 2,798
     Contoh multi-license  : ['CC BY, CC BY-NC-ND', 'CC BY-NC-SA, CC BY-NC-ND', 'CC BY, CC BY-NC-ND, Public domain']

  💡 Dampak: satu sel berisi beberapa lisensi (1 jurnal : N lisensi).
  🛠️ Penanganan: string asli disimpan utuh + dipecah ke tabel jembatan journal_license.


In [41]:
# === DATA ISSUE #6: APC dalam Berbagai Mata Uang ===
issue_count += 1
print(f"\n{'='*60}")
print(f"🔴 Issue #{issue_count}: APC Amount dalam Berbagai Mata Uang")
print(f"{'='*60}")

apc_text = df_doaj_raw["APC amount"].dropna().astype(str)
apc_currencies = apc_text.str.findall(r"[0-9]\s*([A-Za-z]{3})")
currency_counts = apc_currencies.explode().dropna().str.upper().value_counts()
n_multi_currency = (apc_currencies.str.len() > 1).sum()

print(f"\n  📌 DOAJ APC amount:")
print(f"     Jurnal dengan nominal APC : {len(apc_text):,}")
print(f"     Jumlah mata uang berbeda  : {len(currency_counts)}")
print(f"     Lebih dari 1 mata uang    : {n_multi_currency:,} jurnal")
print(f"     Top 10 mata uang          : {currency_counts.head(10).to_dict()}")
print(f"\n  💡 Dampak: APC antar jurnal tidak bisa dibandingkan sebelum disamakan.")
print(f"  🛠️ Penanganan: konversi ke USD dengan kurs dari API (sumber & tanggal tercatat).")


🔴 Issue #6: APC Amount dalam Berbagai Mata Uang

  📌 DOAJ APC amount:
     Jurnal dengan nominal APC : 8,847
     Jumlah mata uang berbeda  : 53
     Lebih dari 1 mata uang    : 2,408 jurnal
     Top 10 mata uang          : {'USD': 4620, 'EUR': 2196, 'GBP': 1625, 'IDR': 1392, 'CHF': 648, 'IRR': 450, 'CNY': 254, 'EGP': 244, 'AUD': 170, 'UAH': 139}

  💡 Dampak: APC antar jurnal tidak bisa dibandingkan sebelum disamakan.
  🛠️ Penanganan: konversi ke USD dengan kurs dari API (sumber & tanggal tercatat).


In [42]:
# === Ringkasan ===
print(f"\n{'='*60}")
print(f"📋 RINGKASAN: {issue_count} Data Issues Ditemukan")
print(f"{'='*60}")
print(f"  1. Missing Integration Key — Crossref: {cr_no_issn:,} artikel tanpa ISSN")
print(f"  2. Duplicate Integration Key — DOAJ: {dup_p} Print ISSN, {dup_e} E-ISSN duplikat")
print(f"  3. Missing Values — banyak kolom dengan null signifikan")
print(f"  4. Tahun Terbit Tidak Valid — {n_anomali:,} artikel Crossref di luar {MIN_YEAR}–{CURRENT_YEAR}")
print(f"  5. Inconsistent Category — {license_vals.nunique()} varian license, {len(multi_license):,} multi-license")
print(f"  6. Multi Currency — APC dalam {len(currency_counts)} mata uang berbeda")


📋 RINGKASAN: 6 Data Issues Ditemukan
  1. Missing Integration Key — Crossref: 179 artikel tanpa ISSN
  2. Duplicate Integration Key — DOAJ: 1 Print ISSN, 2 E-ISSN duplikat
  3. Missing Values — banyak kolom dengan null signifikan
  4. Tahun Terbit Tidak Valid — 0 artikel Crossref di luar 2011–2026
  5. Inconsistent Category — 49 varian license, 2,798 multi-license
  6. Multi Currency — APC dalam 53 mata uang berbeda


---

# 🧹 BAGIAN 4 — CLEANING & STANDARDIZATION

Prinsip: **tandai, jangan buang**. Baris hanya dihapus bila benar-benar duplikat (dan dicatat); data bermasalah diberi *flag* agar tetap traceable ke raw.

Setiap tahap ditulis sebagai **fungsi** agar bisa dipakai ulang oleh `run_pipeline()`. Di walkthrough ini fungsi dipanggil dengan `verbose=True` supaya prosesnya terlihat.

In [43]:
def standardize_issn(series):
    """Trim + uppercase; nilai yang tidak sesuai format XXXX-XXXX dijadikan NA."""
    series = series.astype("string").str.strip().str.upper()
    return series.where(series.str.match(ISSN_REGEX, na=False), pd.NA)


def clean_crossref(crossref_raw, verbose=True):
    """Task clean Crossref → (df_cr, stats). Tidak ada baris yang dihapus kecuali duplikat DOI."""
    say = print if verbose else (lambda *args, **kwargs: None)
    stats = {"rows_before": len(crossref_raw), "cols_before": crossref_raw.shape[1]}
    say(f"  Sebelum: {stats['rows_before']:,} baris × {stats['cols_before']} kolom")

    # [1] Seleksi kolom yang relevan
    keep_cols = [
        "DOI", "title", "author", "author_count",
        "ISSN_print", "ISSN_electronic",
        "container-title", "publisher", "type",
        "sample_year", "issued_year", "issued_month", "issued_day",
        "subject", "is-referenced-by-count",
        "volume", "issue", "page", "license_url",
    ]
    cleaned = crossref_raw[keep_cols].copy()
    say(f"  [1] Kolom dipertahankan: {len(keep_cols)} dari {stats['cols_before']}")

    # [2] Standarisasi kedua integration key
    cleaned["clean_pissn"] = standardize_issn(cleaned["ISSN_print"])
    cleaned["clean_eissn"] = standardize_issn(cleaned["ISSN_electronic"])
    say(f"  [2] Print ISSN & Online ISSN distandarkan (trim, uppercase, validasi format)")

    # [3] Tandai (bukan hapus) artikel tanpa kedua key
    cleaned["has_issn_key"] = cleaned["clean_pissn"].notna() | cleaned["clean_eissn"].notna()
    stats["no_issn_key"] = int((~cleaned["has_issn_key"]).sum())
    say(f"  [3] Ditandai has_issn_key = False: {stats['no_issn_key']:,} baris (tetap disimpan)")

    # [4] Tandai (bukan hapus) tahun terbit tidak valid
    year = pd.to_numeric(cleaned["issued_year"], errors="coerce")
    max_valid_year = int(cleaned["sample_year"].max())
    bad_year = year.isna() | (year < MIN_YEAR) | (year > max_valid_year)
    cleaned["issued_year_raw"] = year.astype("Int64")                  # nilai asli untuk audit
    cleaned["issued_year"] = year.where(~bad_year).astype("Int64")
    cleaned["is_year_valid"] = ~bad_year
    stats["invalid_year"] = int(bad_year.sum())
    say(f"  [4] Ditandai is_year_valid = False: {stats['invalid_year']:,} baris (issued_year dikosongkan)")

    # [5] Konversi tipe data
    cleaned["sample_year"] = cleaned["sample_year"].astype("Int64")
    cleaned["is-referenced-by-count"] = pd.to_numeric(
        cleaned["is-referenced-by-count"], errors="coerce").astype("Int64")

    # [6] Author kosong → 'Unknown' (author_count = 0 tetap menjadi jejak bahwa aslinya kosong)
    empty_author = cleaned["author"].isna() | (cleaned["author"].astype(str).str.strip() == "")
    cleaned.loc[empty_author, "author"] = "Unknown"
    say(f"  [6] Author kosong → 'Unknown': {int(empty_author.sum()):,} baris")

    # [7] DOI lowercase (DOI case-insensitive) & hapus duplikat → grain 1 baris = 1 artikel
    cleaned["DOI"] = cleaned["DOI"].astype(str).str.strip().str.lower()
    stats["duplicate_doi"] = int(cleaned["DOI"].duplicated().sum())
    cleaned = cleaned.drop_duplicates(subset=["DOI"], keep="first").reset_index(drop=True)
    say(f"  [7] Duplikat DOI dihapus: {stats['duplicate_doi']:,}")

    stats["rows_after"], stats["cols_after"] = cleaned.shape
    say(f"\n  ✅ Crossref: {stats['rows_before']:,} → {stats['rows_after']:,} baris "
        f"({stats['cols_before']} → {stats['cols_after']} kolom)")
    return cleaned, stats


print("=" * 60)
print("🧹 CLEANING & STANDARDIZATION: Crossref")
print("=" * 60)
df_cr, cr_stats = clean_crossref(df_crossref_raw)

🧹 CLEANING & STANDARDIZATION: Crossref
  Sebelum: 20,000 baris × 58 kolom
  [1] Kolom dipertahankan: 19 dari 58
  [2] Print ISSN & Online ISSN distandarkan (trim, uppercase, validasi format)
  [3] Ditandai has_issn_key = False: 179 baris (tetap disimpan)
  [4] Ditandai is_year_valid = False: 0 baris (issued_year dikosongkan)
  [6] Author kosong → 'Unknown': 1,122 baris
  [7] Duplikat DOI dihapus: 0

  ✅ Crossref: 20,000 → 20,000 baris (58 → 24 kolom)


### Cleaning DOAJ: ISSN duplikat, license, dan APC

**ISSN duplikat punya dua kasus berbeda:**
- **[3a] Entri jurnal identik** (Print & E-ISSN sama, mis. *Ethics, Politics & Society*): disimpan satu, entri yang dihapus **dicatat** di `logs/doaj_dropped_duplicates_*.csv`.
- **[3b] Satu ISSN dipakai jurnal berbeda** (mis. E-ISSN `2500-154X` untuk dua jurnal *Известия вузов*): ISSN itu tidak bisa dipakai sebagai key karena akan menunjuk ke dua jurnal. Jurnalnya **tetap disimpan**, ISSN aslinya tetap ada (`*_standardized`), ditandai `eissn_is_ambiguous = True`, dan jurnal tetap bisa dicocokkan lewat Print ISSN-nya yang berbeda.

In [44]:
def parse_apc(value):
    """Ambil (nominal, mata uang) dari teks APC. Utamakan nominal USD bila tersedia."""
    if pd.isna(value):
        return np.nan, None
    text = re.sub(r"(?<=\d),(?=\d{3}\b)", "", str(value))          # "1,500" → "1500"
    matches = re.findall(r"([0-9]+(?:\.[0-9]+)?)\s*([A-Za-z]{3})", text)
    if not matches:
        return np.nan, None
    for amount, currency in matches:
        if currency.upper() == "USD":
            return float(amount), "USD"
    amount, currency = matches[0]
    return float(amount), currency.upper()


def clean_doaj(doaj_raw, fx_table, fx_date, snapshot_date, verbose=True):
    """Task clean DOAJ → (df_dj, df_journal_license, stats)."""
    say = print if verbose else (lambda *args, **kwargs: None)
    stats = {"rows_before": len(doaj_raw), "cols_before": doaj_raw.shape[1]}
    say(f"  Sebelum: {stats['rows_before']:,} baris × {stats['cols_before']} kolom")

    # [1] Seleksi kolom yang relevan
    keep_cols = ["Journal title", DOAJ_PRINT_ISSN_COL, DOAJ_ONLINE_ISSN_COL, "Publisher",
                 "Country of publisher", "Journal license", "APC", "APC amount", "Subjects"]
    journals = doaj_raw[keep_cols].copy()
    say(f"  [1] Kolom dipertahankan: {len(keep_cols)} dari {stats['cols_before']}")

    # [2] Standarisasi kedua integration key (nilai standar disimpan untuk jejak)
    journals["pissn_standardized"] = standardize_issn(journals[DOAJ_PRINT_ISSN_COL])
    journals["eissn_standardized"] = standardize_issn(journals[DOAJ_ONLINE_ISSN_COL])
    say(f"  [2] Print ISSN & E-ISSN distandarkan")

    # [3a] Entri jurnal identik → simpan satu, catat yang dihapus
    duplicate_entry = journals.duplicated(subset=["pissn_standardized", "eissn_standardized"], keep="first")
    dropped = journals.loc[duplicate_entry, ["Journal title", "pissn_standardized", "eissn_standardized"]]
    if len(dropped) > 0:
        dropped.assign(snapshot_date=snapshot_date, reason="identical_issn_pair").to_csv(
            LOG_DIR / f"doaj_dropped_duplicates_{snapshot_date}.csv", index=False)
    journals = journals[~duplicate_entry].reset_index(drop=True)
    stats["dropped_duplicate_entries"] = len(dropped)
    say(f"  [3a] Entri jurnal identik dihapus: {len(dropped)} (dicatat di logs/)")

    # [3b] ISSN dipakai jurnal berbeda → tidak dipakai sebagai key, diberi flag
    journals["clean_pissn"] = journals["pissn_standardized"].copy()
    journals["clean_eissn"] = journals["eissn_standardized"].copy()
    for key, flag in [("clean_pissn", "pissn_is_ambiguous"), ("clean_eissn", "eissn_is_ambiguous")]:
        ambiguous = journals[key].notna() & journals[key].duplicated(keep=False)
        journals[flag] = ambiguous
        if verbose and ambiguous.any():
            display(journals.loc[ambiguous, ["Journal title", "pissn_standardized", "eissn_standardized"]])
        journals.loc[ambiguous, key] = pd.NA
        stats[flag] = int(ambiguous.sum())
    say(f"  [3b] ISSN ambigu (tidak dipakai sebagai key): Print {stats['pissn_is_ambiguous']}, "
        f"E-ISSN {stats['eissn_is_ambiguous']}")

    journals["doaj_id"] = journals.index                                # primary key jurnal

    # [4] Rename kolom → snake_case
    journals = journals.rename(columns={
        "Journal title": "journal_title",
        DOAJ_PRINT_ISSN_COL: "print_issn_raw",
        DOAJ_ONLINE_ISSN_COL: "online_issn_raw",
        "Publisher": "doaj_publisher",
        "Country of publisher": "country",
        "Journal license": "license_raw",
        "APC": "apc_status",
        "APC amount": "apc_amount_raw",
        "Subjects": "doaj_subjects",
    })

    # [5] License: string asli utuh + tabel jembatan (1 jurnal : N lisensi)
    license_list = (journals["license_raw"].fillna("").astype(str).str.split(",")
                    .apply(lambda parts: [p.strip() for p in parts if p.strip()]))
    journals["license_count"] = license_list.str.len().astype("Int64")
    journal_license = (pd.DataFrame({"doaj_id": journals["doaj_id"], "license": license_list})
                       .explode("license").dropna(subset=["license"])
                       .drop_duplicates().reset_index(drop=True))
    say(f"  [5] License: {(journals['license_count'] > 1).sum():,} jurnal multi-license → "
        f"{len(journal_license):,} baris journal_license "
        f"({journal_license['license'].nunique()} lisensi unik)")

    # [6] APC → USD dengan kurs yang dapat ditelusuri
    fx_rates = dict(zip(fx_table["currency"], fx_table["rate_per_usd"]))
    parsed = pd.DataFrame(journals["apc_amount_raw"].apply(parse_apc).tolist(),
                          columns=["apc_amount_original", "apc_currency"], index=journals.index)
    journals["apc_amount_original"] = parsed["apc_amount_original"]
    journals["apc_currency"] = parsed["apc_currency"]
    journals["fx_rate"] = journals["apc_currency"].map(fx_rates)
    journals["apc_amount_usd"] = (journals["apc_amount_original"] / journals["fx_rate"]).round(2)

    no_apc = journals["apc_status"] == "No"
    journals["apc_conversion_status"] = np.select(
        [no_apc, journals["apc_amount_original"].isna(),
         journals["apc_currency"] == "USD", journals["fx_rate"].notna()],
        ["no_apc", "amount_not_parsed", "usd_native", "converted"],
        default="currency_not_found")
    journals.loc[no_apc, ["apc_amount_original", "fx_rate"]] = np.nan
    journals.loc[no_apc, "apc_currency"] = None
    journals.loc[no_apc, "apc_amount_usd"] = 0.0                       # APC=No → gratis
    journals["fx_date"] = np.where(journals["apc_conversion_status"].isin(["converted", "usd_native"]),
                                   fx_date, None)
    stats["apc_conversion_status"] = journals["apc_conversion_status"].value_counts().to_dict()
    say(f"  [6] APC → USD (kurs {fx_date}): {stats['apc_conversion_status']}")
    not_found = journals.loc[journals["apc_conversion_status"] == "currency_not_found", "apc_currency"]
    if len(not_found) > 0:
        say(f"      ⚠️ Mata uang tidak ada di sumber kurs: {not_found.value_counts().to_dict()}")

    stats["rows_after"], stats["cols_after"] = journals.shape
    say(f"\n  ✅ DOAJ: {stats['rows_before']:,} → {stats['rows_after']:,} baris "
        f"({stats['cols_before']} → {stats['cols_after']} kolom)")
    return journals, journal_license, stats


print("=" * 60)
print("🧹 CLEANING & STANDARDIZATION: DOAJ")
print("=" * 60)
df_dj, df_journal_license, dj_stats = clean_doaj(df_doaj_raw, df_fx, fx_date, SNAPSHOT_DATE)

print(f"\n  Contoh jejak konversi (apc_amount_original / fx_rate = apc_amount_usd):")
display(df_dj.loc[df_dj["apc_conversion_status"] == "converted",
                  ["journal_title", "apc_amount_raw", "apc_amount_original", "apc_currency",
                   "fx_rate", "fx_date", "apc_amount_usd"]].head(5))

🧹 CLEANING & STANDARDIZATION: DOAJ
  Sebelum: 23,578 baris × 52 kolom
  [1] Kolom dipertahankan: 9 dari 52
  [2] Print ISSN & E-ISSN distandarkan
  [3a] Entri jurnal identik dihapus: 1 (dicatat di logs/)


,Journal title,pissn_standardized,eissn_standardized
17459,Известия вузов: Прикладная химия и биотехнология,2227-2925,2500-154X
23083,Известия вузов: Инвестиции. Строительство. Нед...,2227-2917,2500-154X


  [3b] ISSN ambigu (tidak dipakai sebagai key): Print 0, E-ISSN 2
  [5] License: 2,798 jurnal multi-license → 27,566 baris journal_license (9 lisensi unik)
  [6] APC → USD (kurs 2026-11-06): {'no_apc': 14730, 'usd_native': 4620, 'converted': 4226, 'currency_not_found': 1}
      ⚠️ Mata uang tidak ada di sumber kurs: {'KPW': 1}

  ✅ DOAJ: 23,578 → 23,577 baris (52 → 23 kolom)

  Contoh jejak konversi (apc_amount_original / fx_rate = apc_amount_usd):


,journal_title,apc_amount_raw,apc_amount_original,apc_currency,fx_rate,fx_date,apc_amount_usd
1,مجله دامپزشکی ایران.,4000000 IRR,4000000.0,IRR,1.798421e+06,2026-11-06,2.22
3,Akroterion,1000 ZAR,1000.0,ZAR,1.645286e+01,2026-11-06,60.78
36,Shuichan Xuebao,8000 CNY,8000.0,CNY,6.682525e+00,2026-11-06,1197.15
43,Tiedao biaozhun sheji,4000 CNY,4000.0,CNY,6.682525e+00,2026-11-06,598.58
51,Al-Manhaj,500000 IDR,500000.0,IDR,1.807999e+04,2026-11-06,27.65


In [45]:
# Preview setelah cleaning
print("🔍 Preview Crossref setelah cleaning:")
display(df_cr.head(3))

print("\n🔍 Preview DOAJ setelah cleaning:")
display(df_dj.head(3))

🔍 Preview Crossref setelah cleaning:


,DOI,title,author,author_count,ISSN_print,ISSN_electronic,container-title,publisher,type,sample_year,...,is-referenced-by-count,volume,issue,page,license_url,clean_pissn,clean_eissn,has_issn_key,issued_year_raw,is_year_valid
0,10.1016/j.cclet.2010.10.004,Synthesis of novel calixcrown derivatives with...,Lu Zhang; Juan Du; Li Hua Yuan; Dong Zhang; Gu...,7,1001-8417,NaN,Chinese Chemical Letters,Elsevier BV,journal-article,2011,...,7,22,3,284-287,https://www.elsevier.com/tdm/userlicense/1.0/,1001-8417,<NA>,True,2011,True
1,10.3892/mmr.2011.666,Geranylgeranylacetone protects against myocard...,Neng Wang,1,1791-2997,1791-3004,Molecular Medicine Reports,Spandidos Publications,journal-article,2011,...,5,None,None,None,None,1791-2997,1791-3004,True,2011,True
2,10.1016/j.surfcoat.2011.04.043,Novel low temperature atmospheric pressure pla...,Wen-Jen Liu; Ruei-Chen Wang,2,0257-8972,NaN,Surface and Coatings Technology,Elsevier BV,journal-article,2011,...,4,206,5,925-928,https://www.elsevier.com/tdm/userlicense/1.0/,0257-8972,<NA>,True,2011,True



🔍 Preview DOAJ setelah cleaning:


,journal_title,print_issn_raw,online_issn_raw,doaj_publisher,country,license_raw,apc_status,apc_amount_raw,doaj_subjects,pissn_standardized,...,pissn_is_ambiguous,eissn_is_ambiguous,doaj_id,license_count,apc_amount_original,apc_currency,fx_rate,apc_amount_usd,apc_conversion_status,fx_date
0,International Journal of Applied Science and E...,1727-2394,1727-7841,Chaoyang University of Technology,"Taiwan, Province of China",CC BY,Yes,200 USD,Technology: Engineering (General). Civil engin...,1727-2394,...,False,False,0,1,200.0,USD,1.000000e+00,200.00,usd_native,2026-11-06
1,مجله دامپزشکی ایران.,1735-6873,2676-704X,Shahid Chamran University of Ahvaz,"Iran, Islamic Republic of",CC BY-NC,Yes,4000000 IRR,Agriculture: Animal culture: Veterinary medicine,1735-6873,...,False,False,1,1,4000000.0,IRR,1.798421e+06,2.22,converted,2026-11-06
2,Harčova Nauka ì Tehnologìâ,2073-8684,2409-7004,Odesa National University of Technology,Ukraine,CC BY,No,NaN,Agriculture | Technology,2073-8684,...,False,False,2,1,NaN,None,NaN,0.00,no_apc,None


In [46]:
print("=" * 60)
print("🔍 SIMULASI OVERLAP KEY: CROSSREF ↔ DOAJ")
print("=" * 60)

cr_p = set(df_cr["clean_pissn"].dropna()); cr_e = set(df_cr["clean_eissn"].dropna())
dj_p = set(df_dj["clean_pissn"].dropna()); dj_e = set(df_dj["clean_eissn"].dropna())

print(f"Print ISSN  : {len(cr_p):,} unik di Crossref | {len(cr_p & dj_p):,} ditemukan di DOAJ")
print(f"Online ISSN : {len(cr_e):,} unik di Crossref | {len(cr_e & dj_e):,} ditemukan di DOAJ")

🔍 SIMULASI OVERLAP KEY: CROSSREF ↔ DOAJ
Print ISSN  : 9,473 unik di Crossref | 1,253 ditemukan di DOAJ
Online ISSN : 8,894 unik di Crossref | 2,069 ditemukan di DOAJ


---

# 🔗 BAGIAN 5 — JOIN & INTEGRATION

**LEFT JOIN** agar seluruh artikel Crossref tetap utuh. Integration key berurutan:
1. `Crossref.clean_pissn = DOAJ.clean_pissn` (Print ISSN)
2. Bila belum cocok → `Crossref.clean_eissn = DOAJ.clean_eissn` (Online ISSN)

**`match_status`** (1 kolom, semua nilai adalah status):
| Nilai | Arti |
|-------|------|
| `matched_print_issn` | cocok lewat Print ISSN |
| `matched_online_issn` | cocok lewat Online ISSN |
| `ambiguous_issn` | ISSN artikel menunjuk ke >1 jurnal DOAJ dan tidak ada ISSN lain untuk memastikan → tidak dipaksakan |
| `no_issn` | artikel tidak punya ISSN sama sekali |
| `not_matched_to_doaj` | punya ISSN, tetapi tidak ditemukan di snapshot DOAJ |

Status `not_matched_to_doaj` **tidak** berarti jurnal pasti tidak terindeks DOAJ: bisa saja ISSN di Crossref berbeda/salah. Karena itu kolomnya `is_matched_to_doaj`, bukan `is_doaj_indexed`, dan atribut DOAJ untuk artikel yang tidak cocok dibiarkan **NULL** (tidak diisi label).

**Cardinality:** 1 jurnal DOAJ : N artikel Crossref; 1 jurnal DOAJ : N lisensi.

In [47]:
JOURNAL_ATTRIBUTE_COLS = ["journal_title", "doaj_publisher", "country", "license_raw", "license_count",
                          "apc_status", "apc_amount_original", "apc_currency", "fx_rate", "fx_date",
                          "apc_amount_usd", "apc_conversion_status", "doaj_subjects"]

ARTICLE_OUTPUT_COLS = [
    "DOI", "title", "author", "author_count",
    "sample_year", "issued_year", "issued_year_raw", "is_year_valid",
    "print_issn", "online_issn", "has_issn_key", "container-title", "publisher",
    "match_status", "is_matched_to_doaj", "doaj_id",
    *JOURNAL_ATTRIBUTE_COLS,
    "subject", "is-referenced-by-count", "volume", "issue", "page",
]
MATCH_STATUSES = ["matched_print_issn", "matched_online_issn", "ambiguous_issn", "no_issn", "not_matched_to_doaj"]


def integrate(cleaned_crossref, cleaned_doaj, verbose=True):
    """Task integrate: LEFT JOIN Crossref ← DOAJ → (df_final, stats). Grain: 1 baris = 1 DOI."""
    say = print if verbose else (lambda *args, **kwargs: None)
    articles = cleaned_crossref.copy()

    lookup_p = cleaned_doaj.dropna(subset=["clean_pissn"]).set_index("clean_pissn")["doaj_id"]
    lookup_e = cleaned_doaj.dropna(subset=["clean_eissn"]).set_index("clean_eissn")["doaj_id"]
    assert lookup_p.index.is_unique and lookup_e.index.is_unique

    doaj_id_p = articles["clean_pissn"].map(lookup_p)
    doaj_id_e = articles["clean_eissn"].map(lookup_e)
    articles["doaj_id"] = doaj_id_p.fillna(doaj_id_e).astype("Int64")

    # ISSN artikel yang di DOAJ tercatat untuk >1 jurnal
    ambiguous_p = set(cleaned_doaj.loc[cleaned_doaj["pissn_is_ambiguous"], "pissn_standardized"])
    ambiguous_e = set(cleaned_doaj.loc[cleaned_doaj["eissn_is_ambiguous"], "eissn_standardized"])
    hits_ambiguous = articles["clean_pissn"].isin(ambiguous_p) | articles["clean_eissn"].isin(ambiguous_e)

    articles["match_status"] = np.select(
        [doaj_id_p.notna(), doaj_id_e.notna(), hits_ambiguous, ~articles["has_issn_key"]],
        ["matched_print_issn", "matched_online_issn", "ambiguous_issn", "no_issn"],
        default="not_matched_to_doaj")
    conflict = int((doaj_id_p.notna() & doaj_id_e.notna() & (doaj_id_p != doaj_id_e)).sum())

    joined = articles.merge(
        cleaned_doaj[["doaj_id", *JOURNAL_ATTRIBUTE_COLS]].astype({"doaj_id": "Int64"}),
        on="doaj_id", how="left", validate="many_to_one")                 # N artikel : 1 jurnal
    joined["is_matched_to_doaj"] = joined["doaj_id"].notna()
    final = (joined.rename(columns={"clean_pissn": "print_issn", "clean_eissn": "online_issn"})
             [ARTICLE_OUTPUT_COLS].copy())

    stats = {"rows_before": len(cleaned_crossref), "rows_after": len(final), "key_conflicts": conflict}
    stats.update({status: int((final["match_status"] == status).sum()) for status in MATCH_STATUSES})
    stats["matched"] = stats["matched_print_issn"] + stats["matched_online_issn"]
    stats["not_matched"] = len(final) - stats["matched"]

    say(f"  Artikel Crossref (clean) : {stats['rows_before']:,}")
    say(f"  Baris hasil join         : {stats['rows_after']:,}")
    for status in MATCH_STATUSES:
        say(f"  {status:24s} : {stats[status]:,}")
    say(f"  Konflik key (print & online → jurnal berbeda): {conflict}")
    return final, stats


print("=" * 60)
print("🔗 JOIN: LEFT JOIN Crossref ← DOAJ (Print ISSN → Online ISSN)")
print("=" * 60)
df_final, join_stats = integrate(df_cr, df_dj)

🔗 JOIN: LEFT JOIN Crossref ← DOAJ (Print ISSN → Online ISSN)
  Artikel Crossref (clean) : 20,000
  Baris hasil join         : 20,000
  matched_print_issn       : 1,650
  matched_online_issn      : 2,148
  ambiguous_issn           : 0
  no_issn                  : 179
  not_matched_to_doaj      : 16,023
  Konflik key (print & online → jurnal berbeda): 0


In [48]:
print("=" * 60)
print("🧩 VALIDASI GRAIN & CARDINALITY")
print("=" * 60)

print(f"   DOI unik di hasil join     : {df_final['DOI'].is_unique}")
print(f"   Baris sebelum vs sesudah   : {len(df_cr):,} → {len(df_final):,} "
      f"({'tidak ada duplikasi' if len(df_cr) == len(df_final) else 'TERJADI DUPLIKASI'})")

articles_per_journal = df_final[df_final["is_matched_to_doaj"]].groupby("doaj_id").size()
id2title = df_dj.set_index("doaj_id")["journal_title"]
print(f"\n   [1 jurnal : N artikel]")
print(f"   Jurnal DOAJ yang ter-match : {len(articles_per_journal):,}")
if len(articles_per_journal) > 0:
    print(f"   Artikel per jurnal         : min {articles_per_journal.min()} | "
          f"median {articles_per_journal.median():.0f} | max {articles_per_journal.max()}")
    print(f"   Top 5 jurnal (artikel terbanyak):")
    for k, v in articles_per_journal.sort_values(ascending=False).head(5).items():
        print(f"     • {id2title[k]}: {v} artikel")

licenses_per_journal = df_journal_license.groupby("doaj_id").size()
print(f"\n   [1 jurnal : N lisensi] (tabel journal_license)")
print(f"   Lisensi per jurnal         : min {licenses_per_journal.min()} | max {licenses_per_journal.max()}")
print(f"   Distribusi                 : {licenses_per_journal.value_counts().sort_index().to_dict()}")

🧩 VALIDASI GRAIN & CARDINALITY
   DOI unik di hasil join     : True
   Baris sebelum vs sesudah   : 20,000 → 20,000 (tidak ada duplikasi)

   [1 jurnal : N artikel]
   Jurnal DOAJ yang ter-match : 2,350
   Artikel per jurnal         : min 1 | median 1 | max 104
   Top 5 jurnal (artikel terbanyak):
     • PLoS ONE: 104 artikel
     • Scientific Reports: 71 artikel
     • IEEE Access: 36 artikel
     • Sensors: 26 artikel
     • RSC Advances: 24 artikel

   [1 jurnal : N lisensi] (tabel journal_license)
   Lisensi per jurnal         : min 1 | max 7
   Distribusi                 : {1: 20779, 2: 1926, 3: 733, 4: 48, 5: 3, 6: 87, 7: 1}


---

# 📊 BAGIAN 6 — MATCHED & UNMATCHED COUNT

In [49]:
print("=" * 60)
print("📊 MATCHED vs NOT MATCHED")
print("=" * 60)

total = len(df_final)
labels = {
    "matched_print_issn": "✅ Matched via Print ISSN",
    "matched_online_issn": "✅ Matched via Online ISSN",
    "not_matched_to_doaj": "❌ Not matched (punya ISSN, tidak ditemukan di DOAJ)",
    "ambiguous_issn": "⚠️ Not matched (ISSN ambigu di DOAJ)",
    "no_issn": "⚠️ Not matched (tanpa ISSN)",
}
summary_df = pd.DataFrame({
    "Status": [labels[s] for s in MATCH_STATUSES[:2]] + ["✅ Total Matched"]
              + [labels[s] for s in MATCH_STATUSES[2:]] + ["❌ Total Not Matched", "📋 Total"],
    "Jumlah": [join_stats["matched_print_issn"], join_stats["matched_online_issn"], join_stats["matched"],
               join_stats["ambiguous_issn"], join_stats["no_issn"], join_stats["not_matched_to_doaj"],
               join_stats["not_matched"], total],
})
summary_df["Persentase"] = (summary_df["Jumlah"] / total * 100).map("{:.1f}%".format)
display(summary_df)

print("\n📅 Persentase artikel yang ter-match ke DOAJ per tahun:")
per_year = (df_final.groupby("issued_year", dropna=False)["is_matched_to_doaj"]
            .agg(total_artikel="size", matched_to_doaj="sum"))
per_year["pct_matched"] = (per_year["matched_to_doaj"] / per_year["total_artikel"] * 100).round(1)
display(per_year)

📊 MATCHED vs NOT MATCHED


,Status,Jumlah,Persentase
0,✅ Matched via Print ISSN,1650,8.2%
1,✅ Matched via Online ISSN,2148,10.7%
2,✅ Total Matched,3798,19.0%
3,⚠️ Not matched (ISSN ambigu di DOAJ),0,0.0%
4,⚠️ Not matched (tanpa ISSN),179,0.9%
5,"❌ Not matched (punya ISSN, tidak ditemukan di ...",16023,80.1%
6,❌ Total Not Matched,16202,81.0%
7,📋 Total,20000,100.0%



📅 Persentase artikel yang ter-match ke DOAJ per tahun:


,total_artikel,matched_to_doaj,pct_matched
issued_year,,,
2011,1250,154,12.3
2012,1250,156,12.5
2013,1250,185,14.8
2014,1250,196,15.7
2015,1250,204,16.3
2016,1250,194,15.5
2017,1250,246,19.7
2018,1250,246,19.7
2019,1250,265,21.2


---

# 📈 BAGIAN 7 — JUMLAH BARIS SEBELUM & SESUDAH TRANSFORM

In [50]:
print("=" * 60)
print("📈 JUMLAH BARIS SEBELUM vs SESUDAH TRANSFORM")
print("=" * 60)

transform_summary = pd.DataFrame({
    "Dataset": ["Crossref (Raw)", "Crossref (Clean)", "DOAJ (Raw)", "DOAJ (Clean)",
                "DOAJ journal_license", "Final (hasil LEFT JOIN)"],
    "Jumlah Baris": [cr_stats["rows_before"], cr_stats["rows_after"], dj_stats["rows_before"],
                     dj_stats["rows_after"], len(df_journal_license), len(df_final)],
    "Jumlah Kolom": [cr_stats["cols_before"], cr_stats["cols_after"], dj_stats["cols_before"],
                     dj_stats["cols_after"], df_journal_license.shape[1], df_final.shape[1]],
})
transform_summary["Jumlah Baris"] = transform_summary["Jumlah Baris"].map("{:,}".format)
display(transform_summary)

print(f"\nRincian baris Crossref:")
print(f"   Dihapus  — duplikat DOI            : {cr_stats['duplicate_doi']:,}")
print(f"   Ditandai — tanpa Print/Online ISSN  : {cr_stats['no_issn_key']:,} (match_status = 'no_issn')")
print(f"   Ditandai — tahun terbit tidak valid : {cr_stats['invalid_year']:,} (is_year_valid = False)")
print(f"\nRincian baris DOAJ:")
print(f"   Dihapus  — entri jurnal identik     : {dj_stats['dropped_duplicate_entries']:,} (dicatat di logs/)")
print(f"   Ditandai — Print ISSN ambigu        : {dj_stats['pissn_is_ambiguous']:,}")
print(f"   Ditandai — E-ISSN ambigu            : {dj_stats['eissn_is_ambiguous']:,}")

📈 JUMLAH BARIS SEBELUM vs SESUDAH TRANSFORM


,Dataset,Jumlah Baris,Jumlah Kolom
0,Crossref (Raw),"20,000",58
1,Crossref (Clean),"20,000",24
2,DOAJ (Raw),"23,578",52
3,DOAJ (Clean),"23,577",23
4,DOAJ journal_license,"27,566",2
5,Final (hasil LEFT JOIN),"20,000",34



Rincian baris Crossref:
   Dihapus  — duplikat DOI            : 0
   Ditandai — tanpa Print/Online ISSN  : 179 (match_status = 'no_issn')
   Ditandai — tahun terbit tidak valid : 0 (is_year_valid = False)

Rincian baris DOAJ:
   Dihapus  — entri jurnal identik     : 1 (dicatat di logs/)
   Ditandai — Print ISSN ambigu        : 0
   Ditandai — E-ISSN ambigu            : 2


---

# 👀 BAGIAN 8 — PREVIEW 5 BARIS TERATAS DATA FINAL

In [51]:
print("=" * 60)
print("👀 PREVIEW: 5 Baris Teratas Dataset Final")
print("=" * 60)
print(f"\n📊 Shape: {df_final.shape[0]:,} baris × {df_final.shape[1]} kolom")
print(f"📋 Kolom: {list(df_final.columns)}")

display(df_final.head(5))

print("\n🔎 Contoh 5 artikel yang ter-match ke DOAJ:")
display(df_final[df_final["is_matched_to_doaj"]].head(5))

👀 PREVIEW: 5 Baris Teratas Dataset Final

📊 Shape: 20,000 baris × 34 kolom
📋 Kolom: ['DOI', 'title', 'author', 'author_count', 'sample_year', 'issued_year', 'issued_year_raw', 'is_year_valid', 'print_issn', 'online_issn', 'has_issn_key', 'container-title', 'publisher', 'match_status', 'is_matched_to_doaj', 'doaj_id', 'journal_title', 'doaj_publisher', 'country', 'license_raw', 'license_count', 'apc_status', 'apc_amount_original', 'apc_currency', 'fx_rate', 'fx_date', 'apc_amount_usd', 'apc_conversion_status', 'doaj_subjects', 'subject', 'is-referenced-by-count', 'volume', 'issue', 'page']


,DOI,title,author,author_count,sample_year,issued_year,issued_year_raw,is_year_valid,print_issn,online_issn,...,fx_rate,fx_date,apc_amount_usd,apc_conversion_status,doaj_subjects,subject,is-referenced-by-count,volume,issue,page
0,10.1016/j.cclet.2010.10.004,Synthesis of novel calixcrown derivatives with...,Lu Zhang; Juan Du; Li Hua Yuan; Dong Zhang; Gu...,7,2011,2011,2011,True,1001-8417,<NA>,...,NaN,NaN,NaN,NaN,NaN,,7,22,3,284-287
1,10.3892/mmr.2011.666,Geranylgeranylacetone protects against myocard...,Neng Wang,1,2011,2011,2011,True,1791-2997,1791-3004,...,NaN,NaN,NaN,NaN,NaN,,5,None,None,None
2,10.1016/j.surfcoat.2011.04.043,Novel low temperature atmospheric pressure pla...,Wen-Jen Liu; Ruei-Chen Wang,2,2011,2011,2011,True,0257-8972,<NA>,...,NaN,NaN,NaN,NaN,NaN,,4,206,5,925-928
3,10.1021/ac202520p,Monodisperse Au<sub>11</sub> Clusters Prepared...,Grant E. Johnson; Chongmin Wang; Thomas Priest...,4,2011,2011,2011,True,0003-2700,1520-6882,...,NaN,NaN,NaN,NaN,NaN,,50,83,21,8069-8072
4,10.1007/jhep04(2011)043,See-saw masses for quarks and leptons in SU(5),Thorsten Feldmann,1,2011,2011,2011,True,<NA>,1029-8479,...,NaN,None,0.0,no_apc,Science: Physics: Nuclear and particle physics...,,29,2011,4,None



🔎 Contoh 5 artikel yang ter-match ke DOAJ:


,DOI,title,author,author_count,sample_year,issued_year,issued_year_raw,is_year_valid,print_issn,online_issn,...,fx_rate,fx_date,apc_amount_usd,apc_conversion_status,doaj_subjects,subject,is-referenced-by-count,volume,issue,page
4,10.1007/jhep04(2011)043,See-saw masses for quarks and leptons in SU(5),Thorsten Feldmann,1,2011,2011,2011,True,<NA>,1029-8479,...,NaN,None,0.00,no_apc,Science: Physics: Nuclear and particle physics...,,29,2011,4,None
20,10.1093/molbev/msr173,The 1.688 Repetitive DNA of Drosophila: Concer...,Gustavo C. S. Kuhn; Heinrich Küttler; Orlando ...,4,2011,2011,2011,True,0737-4038,1537-1719,...,1.000000,2026-11-06,3255.00,usd_native,Science: Biology (General): Evolution,,105,29,1,7-11
25,10.5354/0717-5051.1999.11753,Manejo y gestión de áreas históricas: El Plan ...,Vico Muñoz Köhler,1,2011,2011,2011,True,<NA>,0717-5051,...,NaN,None,0.00,no_apc,Geography. Anthropology. Recreation: Geography...,,0,0,1,None
32,10.1007/s11578-011-0107-0,Der Gefühlskapitalismus der Banken: Vom Ende d...,Sighard Neckel,1,2011,2011,2011,True,0340-0425,1861-8588,...,NaN,None,0.00,no_apc,Social Sciences: Social sciences (General),,28,39,1,39-53
34,10.4102/koedoe.v53i1.1032,Alien species in South Africa’s national parks,Dian Spear; Melodie A. McGeoch; Llewellyn C. F...,4,2011,2011,2011,True,0075-6458,2071-0771,...,16.452865,2026-11-06,905.86,converted,Science: Natural history (General): General. I...,,37,53,1,None


In [52]:
print("\n📋 Info tipe data final:")
df_final.info()


📋 Info tipe data final:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   DOI                     20000 non-null  object 
 1   title                   20000 non-null  object 
 2   author                  20000 non-null  object 
 3   author_count            20000 non-null  int64  
 4   sample_year             20000 non-null  Int64  
 5   issued_year             20000 non-null  Int64  
 6   issued_year_raw         20000 non-null  Int64  
 7   is_year_valid           20000 non-null  bool   
 8   print_issn              15535 non-null  string 
 9   online_issn             14161 non-null  string 
 10  has_issn_key            20000 non-null  bool   
 11  container-title         20000 non-null  object 
 12  publisher               19990 non-null  object 
 13  match_status            20000 non-null  object 
 14  is_matched_to

In [53]:
print("\n📊 Missing values dataset final (dipisah berdasarkan status match):")
missing_report = pd.DataFrame({
    "matched_to_doaj": df_final[df_final["is_matched_to_doaj"]].isna().sum(),
    "not_matched": df_final[~df_final["is_matched_to_doaj"]].isna().sum(),
})
missing_report = missing_report[missing_report.sum(axis=1) > 0]
display(missing_report)
print("Catatan: atribut DOAJ yang NULL pada kelompok 'not_matched' memang diharapkan (tidak ada jurnal yang dicocokkan).")
print("Yang perlu diperhatikan adalah NULL pada kelompok 'matched_to_doaj'.")


📊 Missing values dataset final (dipisah berdasarkan status match):


,matched_to_doaj,not_matched
print_issn,1879,2586
online_issn,409,5430
publisher,3,7
doaj_id,0,16202
journal_title,0,16202
doaj_publisher,0,16202
country,0,16202
license_raw,0,16202
license_count,0,16202
apc_status,0,16202


Catatan: atribut DOAJ yang NULL pada kelompok 'not_matched' memang diharapkan (tidak ada jurnal yang dicocokkan).
Yang perlu diperhatikan adalah NULL pada kelompok 'matched_to_doaj'.


---

# 💾 BAGIAN 9 — LOAD (Processed & Serving) + READ-BACK VALIDATION

```
processed/
├── articles_enriched/                 ← Parquet, partisi issued_year (grain: 1 baris = 1 DOI)
│   ├── issued_year=2011/ ... issued_year=2026/
│   └── issued_year=__HIVE_DEFAULT_PARTITION__/    (tahun tidak valid)
├── journals_doaj.parquet  journal_license.parquet  fx_rates.parquet
└── publications.db                    ← SQLite (serving layer)
```

### Strategi load
| Target | Strategi | Alasan |
|--------|----------|--------|
| Raw snapshot | **Append per `ingestion_date`** (file lama tidak pernah ditimpa) | Menyimpan histori snapshot; sampel acak hanya bisa direproduksi dari snapshot yang disimpan |
| Processed Parquet & tabel data SQLite | **Full refresh (overwrite)** dari satu snapshot | Sumber berupa snapshot (DOAJ, kurs) dan sampel acak (Crossref); processed mencerminkan **satu snapshot utuh**, sehingga run ulang dari snapshot yang sama menghasilkan state yang sama, bukan data dobel |
| `pipeline_runs` (SQLite) | **Append** | Log audit setiap run yang berhasil |

### Alur load yang aman
1. Processed dibangun di folder kerja lokal `work/` (bukan langsung di Drive).
2. **Read-back validation** dijalankan pada hasil yang baru ditulis.
3. Hanya jika semua cek lolos, folder baru **menggantikan** processed lama (*swap*):
   - hasil disalin ke `processed_incoming/` dan dicek utuh (daftar file + ukuran sama dengan folder kerja);
   - `processed/` lama dipindah ke `processed_previous/`, lalu `processed_incoming/` dipindah menjadi `processed/`;
   - jika pemindahan kedua gagal → **rollback**: `processed_previous/` dikembalikan menjadi `processed/`;
   - `processed_previous/` baru dihapus setelah `processed/` baru terpasang.
4. **Recovery**: jika sesi Colab mati tepat di tengah *swap* (`processed/` tidak ada, `processed_previous/` ada), `recover_processed()` mengembalikannya pada awal run berikutnya.

In [54]:
SQLITE_SCHEMA = """
PRAGMA foreign_keys = ON;

DROP VIEW  IF EXISTS v_enriched_publications;
DROP TABLE IF EXISTS journal_license;
DROP TABLE IF EXISTS articles;
DROP TABLE IF EXISTS journals_doaj;
DROP TABLE IF EXISTS fx_rates;

CREATE TABLE journals_doaj (
    doaj_id               INTEGER PRIMARY KEY,
    journal_title         TEXT NOT NULL,
    print_issn_raw        TEXT,
    online_issn_raw       TEXT,
    print_issn            TEXT UNIQUE,      -- key yang dipakai join (NULL bila ambigu)
    online_issn           TEXT UNIQUE,
    pissn_is_ambiguous    INTEGER,
    eissn_is_ambiguous    INTEGER,
    doaj_publisher        TEXT,
    country               TEXT,
    license_raw           TEXT,
    license_count         INTEGER,
    apc_status            TEXT,
    apc_amount_raw        TEXT,
    apc_amount_original   REAL,
    apc_currency          TEXT,
    fx_rate               REAL,
    fx_date               TEXT,
    apc_amount_usd        REAL,
    apc_conversion_status TEXT,
    doaj_subjects         TEXT
);

CREATE TABLE journal_license (
    doaj_id  INTEGER NOT NULL REFERENCES journals_doaj(doaj_id),
    license  TEXT NOT NULL,
    PRIMARY KEY (doaj_id, license)
);

CREATE TABLE articles (
    doi              TEXT PRIMARY KEY,
    title            TEXT,
    author           TEXT,
    author_count     INTEGER,
    sample_year      INTEGER NOT NULL,
    issued_year      INTEGER,
    issued_year_raw  INTEGER,
    is_year_valid    INTEGER NOT NULL,
    print_issn       TEXT,
    online_issn      TEXT,
    has_issn_key     INTEGER NOT NULL,
    container_title  TEXT,
    publisher        TEXT,
    subject          TEXT,
    cited_by_count   INTEGER,
    volume           TEXT,
    issue            TEXT,
    page             TEXT,
    match_status     TEXT NOT NULL,
    doaj_id          INTEGER REFERENCES journals_doaj(doaj_id)
);

CREATE TABLE fx_rates (
    currency      TEXT NOT NULL,
    fx_date       TEXT NOT NULL,
    rate_per_usd  REAL NOT NULL,
    base_currency TEXT,
    source        TEXT,
    PRIMARY KEY (currency, fx_date)
);

CREATE TABLE IF NOT EXISTS pipeline_runs (
    run_id               TEXT PRIMARY KEY,
    run_type             TEXT,
    run_timestamp        TEXT,
    snapshot_date        TEXT,
    crossref_raw_rows    INTEGER,
    articles_rows        INTEGER,
    doaj_raw_rows        INTEGER,
    journals_rows        INTEGER,
    matched_print_issn   INTEGER,
    matched_online_issn  INTEGER,
    ambiguous_issn       INTEGER,
    no_issn              INTEGER,
    not_matched_to_doaj  INTEGER,
    invalid_year         INTEGER,
    fx_date              TEXT,
    duration_seconds     REAL
);

CREATE INDEX idx_articles_year ON articles(issued_year);
CREATE INDEX idx_articles_doaj ON articles(doaj_id);

CREATE VIEW v_enriched_publications AS
SELECT a.*,
       j.journal_title, j.doaj_publisher, j.country, j.license_raw, j.license_count,
       j.apc_status, j.apc_amount_original, j.apc_currency, j.fx_rate, j.fx_date,
       j.apc_amount_usd, j.apc_conversion_status, j.doaj_subjects
FROM articles a
LEFT JOIN journals_doaj j ON a.doaj_id = j.doaj_id;
"""

YEAR_PARTITIONING = ds.partitioning(pa.schema([("issued_year", pa.int64())]), flavor="hive")


def to_sql_tables(final, journals):
    """Siapkan tabel SQLite: nama kolom ramah SQL, boolean → 0/1, tanpa atribut jurnal yang terduplikasi."""
    sql_articles = final.rename(columns={
        "DOI": "doi", "container-title": "container_title", "is-referenced-by-count": "cited_by_count",
    })[["doi", "title", "author", "author_count", "sample_year", "issued_year", "issued_year_raw",
        "is_year_valid", "print_issn", "online_issn", "has_issn_key", "container_title", "publisher",
        "subject", "cited_by_count", "volume", "issue", "page", "match_status", "doaj_id"]].copy()
    sql_journals = journals.rename(columns={"clean_pissn": "print_issn", "clean_eissn": "online_issn"})[
        ["doaj_id", "journal_title", "print_issn_raw", "online_issn_raw", "print_issn", "online_issn",
         "pissn_is_ambiguous", "eissn_is_ambiguous", "doaj_publisher", "country", "license_raw",
         "license_count", "apc_status", "apc_amount_raw", "apc_amount_original", "apc_currency",
         "fx_rate", "fx_date", "apc_amount_usd", "apc_conversion_status", "doaj_subjects"]].copy()
    for table, cols in [(sql_articles, ["is_year_valid", "has_issn_key"]),
                        (sql_journals, ["pissn_is_ambiguous", "eissn_is_ambiguous"])]:
        for col in cols:
            table[col] = table[col].astype(int)
    return sql_articles, sql_journals


def build_processed(final, journals, journal_license, fx_table, run_record, work_dir, target_dir=PROCESSED_DIR):
    """Task load (tahap 1): tulis Parquet + SQLite ke folder kerja lokal (belum dipublikasikan)."""
    if work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True)

    # Parquet: fakta artikel dipartisi per issued_year, tabel kecil 1 file
    final.to_parquet(work_dir / "articles_enriched", partition_cols=["issued_year"],
                     index=False, engine="pyarrow")
    journals.to_parquet(work_dir / "journals_doaj.parquet", index=False)
    journal_license.to_parquet(work_dir / "journal_license.parquet", index=False)
    fx_table.to_parquet(work_dir / "fx_rates.parquet", index=False)

    # SQLite: mulai dari DB yang sedang aktif di folder tujuan agar riwayat pipeline_runs tetap tersambung
    db_path = work_dir / "publications.db"
    if (target_dir / "publications.db").exists():
        shutil.copy2(target_dir / "publications.db", db_path)
    sql_articles, sql_journals = to_sql_tables(final, journals)
    with closing(sqlite3.connect(db_path)) as conn:
        conn.executescript(SQLITE_SCHEMA)                                # tabel data: full refresh
        run_columns = {row[1] for row in conn.execute("PRAGMA table_info(pipeline_runs)")}
        if "run_type" not in run_columns:                                # DB lama (dibuat sebelum ada run_type)
            conn.execute("ALTER TABLE pipeline_runs ADD COLUMN run_type TEXT")
        sql_journals.to_sql("journals_doaj", conn, if_exists="append", index=False)   # induk dulu (FK)
        journal_license.to_sql("journal_license", conn, if_exists="append", index=False)
        sql_articles.to_sql("articles", conn, if_exists="append", index=False)
        fx_table[["currency", "fx_date", "rate_per_usd", "base_currency", "source"]].to_sql(
            "fx_rates", conn, if_exists="append", index=False)
        pd.DataFrame([run_record]).to_sql("pipeline_runs", conn, if_exists="append", index=False)  # append
        conn.commit()


def recover_processed(processed_dir=PROCESSED_DIR):
    """Recovery: jika publish sebelumnya terhenti di tengah swap (processed/ hilang,
    tetapi processed_previous/ masih ada), kembalikan processed_previous menjadi processed."""
    previous = processed_dir.with_name(processed_dir.name + "_previous")
    if not processed_dir.exists() and previous.exists():
        os.replace(previous, processed_dir)
        print(f"   ♻️ Recovery: {processed_dir.name}/ dipulihkan dari {previous.name}/")
        return True
    return False


def same_files(folder_a, folder_b):
    """True jika kedua folder berisi file yang sama (path relatif + ukuran byte)."""
    def listing(folder):
        return {(str(p.relative_to(folder)), p.stat().st_size) for p in folder.rglob("*") if p.is_file()}
    return listing(folder_a) == listing(folder_b)


def publish_processed(work_dir, processed_dir=PROCESSED_DIR):
    """Task publish: ganti processed lama dengan hasil baru yang sudah lolos validasi.
    Processed lama baru dihapus setelah processed baru terpasang; jika swap gagal → rollback."""
    incoming = processed_dir.with_name(processed_dir.name + "_incoming")
    previous = processed_dir.with_name(processed_dir.name + "_previous")

    # [1] Recovery dulu: processed_previous tidak boleh dihapus selama processed/ belum ada
    recover_processed(processed_dir)
    for folder in [incoming, previous]:
        if folder.exists():
            shutil.rmtree(folder)

    # [2] Salin hasil baru ke Drive, lalu pastikan salinannya utuh (processed lama belum disentuh)
    shutil.copytree(work_dir, incoming)
    if not same_files(work_dir, incoming):
        shutil.rmtree(incoming)
        raise RuntimeError("Salinan processed_incoming tidak utuh → publish dibatalkan, processed lama tetap.")

    # [3] Swap dengan rollback
    had_old = processed_dir.exists()
    if had_old:
        os.replace(processed_dir, previous)                              # processed lama → previous
    try:
        os.replace(incoming, processed_dir)                              # incoming → processed
    except Exception as error:
        print(f"   ❌ Swap gagal: {error}")
        if had_old and not processed_dir.exists():
            os.replace(previous, processed_dir)                          # ROLLBACK: previous → processed
            print(f"   ↩️ Rollback: {processed_dir.name}/ lama dikembalikan")
        raise

    # [4] Processed lama baru dihapus setelah processed baru terpasang
    if previous.exists():
        shutil.rmtree(previous)


print("✅ Fungsi load siap.")

✅ Fungsi load siap.


### Read-back validation

Setelah ditulis, data **dibaca kembali dari file/database** (bukan dari DataFrame di memori) lalu dicek:

| Kategori | Cek |
|----------|-----|
| Row count | Parquet & SQLite = jumlah baris hasil transform |
| Key uniqueness | DOI unik & tidak NULL (Parquet & SQLite); PK `journal_license` |
| Null penting | `sample_year`, `match_status` tidak NULL; artikel yang ter-match wajib punya `doaj_id` & `journal_title`; APC yang dikonversi wajib punya `fx_rate` & `fx_date` |
| Konsistensi | `is_matched_to_doaj` ⇔ `doaj_id` terisi; nilai `match_status` valid; foreign key tidak dilanggar |
| Schema | Kolom Parquet & SQLite sesuai yang diharapkan |
| Sample record | 3 baris dari view `v_enriched_publications` |

In [55]:
def validate_readback(work_dir, expected_rows, expected_journals, verbose=True):
    """Baca ulang hasil load dan jalankan cek kualitas. Return (DataFrame cek, sample record)."""
    checks = []

    def check(category, name, actual, expected):
        checks.append({"kategori": category, "cek": name, "actual": actual,
                       "expected": expected, "lolos": actual == expected})

    articles = pd.read_parquet(work_dir / "articles_enriched", partitioning=YEAR_PARTITIONING)
    matched = articles[articles["is_matched_to_doaj"]]
    check("row count", "Parquet articles = hasil transform", len(articles), expected_rows)
    check("key", "Parquet: DOI NULL", int(articles["DOI"].isna().sum()), 0)
    check("key", "Parquet: DOI duplikat", int(articles["DOI"].duplicated().sum()), 0)
    check("schema", "Parquet: kolom sesuai", sorted(articles.columns) == sorted(ARTICLE_OUTPUT_COLS), True)
    check("null", "sample_year NULL", int(articles["sample_year"].isna().sum()), 0)
    check("null", "matched tanpa journal_title", int(matched["journal_title"].isna().sum()), 0)
    check("konsistensi", "is_matched_to_doaj ⇔ doaj_id terisi",
          bool((articles["is_matched_to_doaj"] == articles["doaj_id"].notna()).all()), True)
    check("konsistensi", "match_status di luar daftar",
          int((~articles["match_status"].isin(MATCH_STATUSES)).sum()), 0)

    with closing(sqlite3.connect(work_dir / "publications.db")) as conn:
        query = lambda sql: conn.execute(sql).fetchone()[0]
        check("row count", "SQLite articles = hasil transform", query("SELECT COUNT(*) FROM articles"), expected_rows)
        check("row count", "SQLite journals_doaj = hasil clean", query("SELECT COUNT(*) FROM journals_doaj"),
              expected_journals)
        check("key", "SQLite: COUNT(DISTINCT doi) = COUNT(*)",
              query("SELECT COUNT(DISTINCT doi) FROM articles"), expected_rows)
        check("key", "SQLite: journal_license (doaj_id, license) duplikat",
              query("SELECT COUNT(*) - COUNT(DISTINCT doaj_id || '|' || license) FROM journal_license"), 0)
        check("null", "match_status NULL", query("SELECT COUNT(*) FROM articles WHERE match_status IS NULL"), 0)
        check("null", "APC converted tanpa fx_rate/fx_date",
              query("SELECT COUNT(*) FROM journals_doaj WHERE apc_conversion_status = 'converted' "
                    "AND (fx_rate IS NULL OR fx_date IS NULL)"), 0)
        check("konsistensi", "Pelanggaran foreign key", len(conn.execute("PRAGMA foreign_key_check").fetchall()), 0)
        sqlite_cols = [row[1] for row in conn.execute("PRAGMA table_info(articles)")]
        check("schema", "SQLite articles: jumlah kolom", len(sqlite_cols), 20)
        check("konsistensi", "DOI Parquet = DOI SQLite",
              set(pd.read_sql_query("SELECT doi FROM articles", conn)["doi"]) == set(articles["DOI"]), True)
        sample = pd.read_sql_query(
            "SELECT doi, title, issued_year, match_status, journal_title, license_raw, apc_amount_usd "
            "FROM v_enriched_publications ORDER BY doaj_id IS NULL, doi LIMIT 3", conn)

    report = pd.DataFrame(checks)
    if verbose:
        display(report)
        print("🔎 Sample record (dari view v_enriched_publications):")
        display(sample)
    return report, sample


def articles_fingerprint(db_path):
    """Sidik jari isi tabel articles (urut DOI) untuk membandingkan state antar run."""
    with closing(sqlite3.connect(db_path)) as conn:
        table = pd.read_sql_query("SELECT * FROM articles ORDER BY doi", conn)
    return int(pd.util.hash_pandas_object(table, index=False).sum())


print("✅ Fungsi read-back validation siap.")

✅ Fungsi read-back validation siap.


---

# ▶️ BAGIAN 10 — `run_pipeline()`: Satu Fungsi Utama End-to-End

```
run_pipeline(snapshot_date)
  1. ingestion            extract_snapshot        → raw (dilewati jika snapshot lengkap sudah ada)
  2. raw validation       load_raw + validate_raw → jika gagal: STOP sebelum transformasi
  3. transformation       clean_crossref + clean_doaj
  4. integration          integrate               → LEFT JOIN, grain 1 DOI
  5. load                 build_processed         → Parquet + SQLite di folder kerja
  6. processed validation validate_readback       → jika gagal: STOP, processed lama tidak diganti
  7. publish              publish_processed       → swap ke processed/
```
Setiap run (berhasil maupun gagal) dicatat di `logs/pipeline_run_history.csv`; run yang berhasil juga tercatat di tabel `pipeline_runs`.

### DAG pipeline

Urutan task **tidak ditulis manual**, tetapi diturunkan dari daftar dependency (`PIPELINE_DAG`). Setiap task hanya boleh berjalan setelah **semua** task upstream-nya berstatus `success`.

![DAG Pipeline Kelompok 5](https://raw.githubusercontent.com/IRay3477/DatEng/main/docs/Diagram_DAG_Kelompok_5_Sprint_3.png)

`topological_levels()` mengelompokkan task per **level**. Task dalam level yang sama tidak saling bergantung, sehingga **dapat berjalan paralel**:
- Level 1: `extract_crossref`, `extract_doaj`, `extract_fx` (tiga sumber berbeda)
- Level 4: `clean_crossref`, `clean_doaj` (membersihkan dua sumber yang terpisah)

Pada implementasi ini task tetap dijalankan **berurutan**, karena (1) Crossref meminta jeda antar request (*polite pool*), sehingga paralel tidak mempercepat bagian terlama; (2) DOAJ, kurs, dan cleaning hanya butuh hitungan detik. Urutan dari DAG menjamin dependency selalu terpenuhi, dan bila nanti dipindah ke orchestrator (misalnya Airflow), level yang sama dapat dijalankan paralel tanpa mengubah DAG.

Fungsi ini juga memastikan DAG **acyclic**: jika ada dependency melingkar, notebook langsung berhenti dengan error.

In [56]:
PIPELINE_DAG = {                       # task: [daftar task upstream]
    "extract_crossref":     [],
    "extract_doaj":         [],
    "extract_fx":           [],
    "write_manifest":       ["extract_crossref", "extract_doaj", "extract_fx"],
    "raw_validation":       ["write_manifest"],
    "clean_crossref":       ["raw_validation"],
    "clean_doaj":           ["raw_validation"],
    "integration":          ["clean_crossref", "clean_doaj"],
    "load":                 ["integration"],
    "processed_validation": ["load"],
    "publish":              ["processed_validation"],
}


def topological_levels(dag):
    """Kelompokkan task per level (algoritma Kahn). Task dalam satu level tidak saling bergantung.
    Raise ValueError jika ada dependency melingkar atau upstream yang tidak terdaftar."""
    unknown = {up for ups in dag.values() for up in ups} - set(dag)
    if unknown:
        raise ValueError(f"Upstream tidak terdaftar di DAG: {unknown}")
    remaining = {task: set(ups) for task, ups in dag.items()}
    levels = []
    while remaining:
        ready = sorted(task for task, ups in remaining.items() if not ups)   # semua upstream sudah selesai
        if not ready:
            raise ValueError(f"DAG mengandung siklus: {sorted(remaining)}")
        levels.append(ready)
        for task in ready:
            del remaining[task]
        for ups in remaining.values():
            ups.difference_update(ready)
    return levels


TASK_LEVELS = topological_levels(PIPELINE_DAG)
TASKS = [task for level in TASK_LEVELS for task in level]   # urutan eksekusi

print("✅ DAG valid (tidak ada siklus).")
for number, level in enumerate(TASK_LEVELS, start=1):
    note = "  ← dapat paralel" if len(level) > 1 else ""
    print(f"   Level {number}: {', '.join(level)}{note}")

✅ DAG valid (tidak ada siklus).
   Level 1: extract_crossref, extract_doaj, extract_fx  ← dapat paralel
   Level 2: write_manifest
   Level 3: raw_validation
   Level 4: clean_crossref, clean_doaj  ← dapat paralel
   Level 5: integration
   Level 6: load
   Level 7: processed_validation
   Level 8: publish


### Identitas run: run_id, snapshot_date, logical date, dan run_type

Setiap run punya identitas yang terpisah dari data yang diprosesnya:

| Istilah | Di pipeline ini | Contoh |
|---|---|---|
| **run id** (identitas satu eksekusi) | `run_id`, dibentuk dari waktu mulai run (UTC) | `run_20261106T132230123456` |
| **snapshot date** (versi raw) | `snapshot_date` = folder `ingestion_date=...` di raw | `2026-10-30` |
| **execution / logical date** (periode data yang diproses) | sama dengan `snapshot_date`, karena satu run memproses tepat satu snapshot | `2026-10-30` |
| **runtime** (kapan benar-benar dieksekusi) | `run_timestamp` (disimpan UTC, ditampilkan WITA) | `2026-11-06 21:22 WITA` |

Sumber kita (Crossref sampling, DOAJ, kurs) hanya menyediakan **kondisi saat ini**, sehingga periode data ditentukan oleh **kapan snapshot diambil**. Karena itu logical date = snapshot date. Yang bisa berbeda adalah **runtime**: snapshot 30 Oktober bisa diproses ulang pada 6 November, dan `run_timestamp` mencatat kapan pemrosesan itu terjadi tanpa mengubah logical date-nya.

`run_type` ditentukan otomatis oleh `classify_run()`:

| run_type | Kondisi | Contoh | Hasil ditulis ke |
|---|---|---|---|
| `normal` | snapshot terbaru, belum pernah di-run | run bulanan pertama untuk snapshot baru | `processed/` |
| `rerun` | snapshot terbaru yang **sudah pernah** di-run (berhasil atau gagal) | run ulang setelah publish gagal | `processed/` |
| `backfill` | snapshot **lebih lama** dari snapshot lengkap terbaru | memproses ulang snapshot Oktober saat snapshot November sudah ada | `processed_backfill/snapshot_date=.../` |

`processed/` adalah data yang dipakai pengguna, sehingga **hanya snapshot terbaru** yang boleh menggantinya. Backfill ditulis ke folder terpisah per tanggal snapshot agar data terbaru tidak tertimpa data lama. Proses publish di folder backfill tetap memakai mekanisme yang sama (incoming → validasi → swap + rollback).

Backfill hanya bisa untuk snapshot lama yang **raw-nya sudah lengkap** (manifest ada). Crossref, DOAJ, dan kurs hanya menyediakan kondisi saat ini, sehingga snapshot lama yang raw-nya tidak ada tidak dapat diambil ulang; run seperti itu langsung ditolak sebelum task pertama.

`run_type` dicatat di `logs/pipeline_run_history.csv` dan tabel `pipeline_runs` (SQLite). Run yang dijalankan sebelum kolom ini ada tetap tercatat, dengan `run_type` kosong.

In [57]:
HISTORY_COLUMNS = ["run_id", "run_type", "snapshot_date", "run_timestamp", "output_dir", "status",
                   "articles_rows", "distinct_doi", "articles_fingerprint", "duration_seconds", "error"]


def complete_snapshots():
    """Tanggal snapshot raw yang sudah lengkap (punya manifest), urut dari terlama."""
    return sorted(path.stem.removeprefix("manifest_")
                  for path in (RAW_DIR / "_manifests").glob("manifest_*.json"))


def previous_runs(snapshot_date):
    """Berapa kali snapshot ini sudah pernah di-run (berhasil maupun gagal)."""
    history_file = LOG_DIR / "pipeline_run_history.csv"
    if not history_file.exists():
        return 0
    history = pd.read_csv(history_file, dtype=str)
    return int((history["snapshot_date"] == snapshot_date).sum())


def classify_run(snapshot_date):
    """Tentukan run_type: backfill (snapshot lama), rerun (snapshot terbaru, pernah di-run), atau normal."""
    snapshots = complete_snapshots()
    if snapshots and snapshot_date < snapshots[-1]:
        return "backfill"
    if previous_runs(snapshot_date) > 0:
        return "rerun"
    return "normal"


def publish_target(run_type, snapshot_date):
    """Folder tujuan publish: processed/ untuk snapshot terbaru, folder terpisah untuk backfill."""
    if run_type == "backfill":
        return BACKFILL_DIR / f"snapshot_date={snapshot_date}"
    return PROCESSED_DIR


print(f"✅ Snapshot lengkap di raw : {complete_snapshots()}")
print(f"   run_type untuk {SNAPSHOT_DATE} : {classify_run(SNAPSHOT_DATE)}")
print(f"   hasil akan ditulis ke     : {publish_target(classify_run(SNAPSHOT_DATE), SNAPSHOT_DATE)}")

✅ Snapshot lengkap di raw : ['2026-10-30', '2026-11-07']
   run_type untuk 2026-11-07 : normal
   hasil akan ditulis ke     : /content/drive/MyDrive/kelompok5_publication_pipeline/processed


### Task state & retry state

`TaskTracker` mencatat status setiap task dalam satu run:

| State | Arti |
|---|---|
| `pending` | belum dijalankan |
| `running` | sedang berjalan |
| `retrying` | request gagal sementara (timeout / HTTP 429 / 5xx) dan sedang dicoba ulang |
| `success` | selesai dengan benar |
| `failed` | gagal; pipeline berhenti |
| `skipped` | tidak dijalankan karena salah satu *upstream*-nya (menurut DAG) gagal atau ikut di-skip |

Jika satu task gagal, hanya task turunannya yang di-skip; task lain yang tidak bergantung padanya tetap dijalankan (misalnya `extract_crossref` gagal, tetapi `extract_doaj` dan `extract_fx` tetap jalan, sehingga saat run ulang kedua file tersebut sudah ada dan tidak diunduh lagi).

Status akhir per task disimpan di `logs/task_runs.csv`, sedangkan setiap perubahan status (termasuk `retrying`) disimpan di `logs/task_events.csv`.

In [58]:
class TaskTracker:
    """Mencatat status setiap task (sesuai DAG) dalam satu run."""

    def __init__(self, run_id, snapshot_date):
        self.run_id = run_id
        self.snapshot_date = snapshot_date
        self.tasks = {name: {"state": "pending", "retries": 0, "started_at": None,
                             "finished_at": None, "duration_seconds": None, "message": None}
                      for name in TASKS}
        self.events = []
        self.current = None                                   # task yang sedang berjalan

    def _record(self, name, state, message=None):
        """Ubah status task dan catat kejadiannya."""
        self.tasks[name]["state"] = state
        if message is not None:
            self.tasks[name]["message"] = message
        self.events.append({"run_id": self.run_id, "task": name, "state": state,
                            "time": pd.Timestamp.now(tz=LOCAL_TZ).isoformat(), "message": message})

    def blocked_by(self, name):
        """Upstream (menurut DAG) yang gagal atau di-skip → task ini tidak boleh jalan."""
        return [up for up in PIPELINE_DAG[name] if self.tasks[up]["state"] in ("failed", "skipped")]

    def start(self, name):
        not_done = [up for up in PIPELINE_DAG[name] if self.tasks[up]["state"] != "success"]
        if not_done:                                          # pengaman: dependency wajib terpenuhi
            raise RuntimeError(f"{name} tidak boleh jalan: upstream belum success {not_done}")
        self.current = name
        self.tasks[name]["started_at"] = pd.Timestamp.now(tz=LOCAL_TZ)
        self._record(name, "running")
        print(f"   [{TASKS.index(name) + 1}/{len(TASKS)}] {name}")

    def retrying(self, message):
        if self.current is not None:
            self.tasks[self.current]["retries"] += 1
            self._record(self.current, "retrying", message)

    def success(self, message=None):
        self._finish("success", message)

    def fail(self, error):
        """Tandai task yang sedang berjalan sebagai failed."""
        if self.current is not None:
            self._finish("failed", f"{type(error).__name__}: {error}")

    def skip(self, name, upstream):
        self._record(name, "skipped", f"upstream tidak success: {', '.join(upstream)}")
        print(f"   [{TASKS.index(name) + 1}/{len(TASKS)}] {name} → skipped")

    def skip_all_pending(self, reason):
        """Dipakai jika run terhenti total (mis. interupsi): semua task yang belum jalan → skipped."""
        for name in TASKS:
            if self.tasks[name]["state"] == "pending":
                self._record(name, "skipped", reason)

    def _finish(self, state, message):
        task = self.tasks[self.current]
        task["finished_at"] = pd.Timestamp.now(tz=LOCAL_TZ)
        task["duration_seconds"] = round((task["finished_at"] - task["started_at"]).total_seconds(), 1)
        self._record(self.current, state, message)
        self.current = None

    def table(self):
        return pd.DataFrame([{"run_id": self.run_id, "snapshot_date": self.snapshot_date, "task": name,
                              "state": t["state"], "retries": t["retries"],
                              "duration_seconds": t["duration_seconds"], "message": t["message"]}
                             for name, t in self.tasks.items()])

    def save(self):
        """Tambahkan (append) status task & riwayat kejadian ke folder logs/."""
        for table, file_name in [(self.table(), "task_runs.csv"), (pd.DataFrame(self.events), "task_events.csv")]:
            path = LOG_DIR / file_name
            table.to_csv(path, mode="a", header=not path.exists(), index=False)


print("✅ TaskTracker siap.")

✅ TaskTracker siap.


In [59]:
# ------------------------------------------------------------------ fungsi setiap task
# Setiap task menerima ctx (tempat berbagi hasil antar-task) dan mengembalikan pesan singkat.

def task_extract_crossref(ctx):
    if ctx["already_complete"]:
        return "snapshot sudah lengkap, dilewati"
    extract_crossref(ctx["snapshot_date"])
    return "Crossref diambil"


def task_extract_doaj(ctx):
    if ctx["already_complete"]:
        return "snapshot sudah lengkap, dilewati"
    extract_doaj(ctx["snapshot_date"])
    return "DOAJ diambil"


def task_extract_fx(ctx):
    if ctx["already_complete"]:
        return "snapshot sudah lengkap, dilewati"
    extract_fx(ctx["snapshot_date"])
    return "kurs diambil"


def task_write_manifest(ctx):
    if ctx["already_complete"]:
        return "manifest sudah ada, dilewati"
    write_manifest(ctx["snapshot_date"])
    return "manifest ditulis"


def task_raw_validation(ctx):
    ctx["crossref_raw"], ctx["doaj_raw"], ctx["fx_table"], ctx["fx_date"] = load_raw(ctx["snapshot_date"])
    report = validate_raw(ctx["snapshot_date"], ctx["crossref_raw"], ctx["doaj_raw"], ctx["fx_table"],
                          verbose=False)
    return f"{len(report)}/{len(report)} cek raw lolos"


def task_clean_crossref(ctx):
    ctx["cleaned_crossref"], ctx["crossref_stats"] = clean_crossref(ctx["crossref_raw"], verbose=False)
    return f"{len(ctx['cleaned_crossref']):,} artikel"


def task_clean_doaj(ctx):
    ctx["cleaned_doaj"], ctx["journal_license"], ctx["doaj_stats"] = clean_doaj(
        ctx["doaj_raw"], ctx["fx_table"], ctx["fx_date"], ctx["snapshot_date"], verbose=False)
    return f"{len(ctx['cleaned_doaj']):,} jurnal"


def task_integration(ctx):
    ctx["final"], ctx["integration_stats"] = integrate(ctx["cleaned_crossref"], ctx["cleaned_doaj"],
                                                       verbose=False)
    return f"{ctx['integration_stats']['matched']:,} artikel ter-match"


def task_load(ctx):
    run_record = {
        "run_id": ctx["run_id"], "run_type": ctx["run_type"], "run_timestamp": ctx["run_timestamp"],
        "snapshot_date": ctx["snapshot_date"], "crossref_raw_rows": len(ctx["crossref_raw"]),
        "articles_rows": len(ctx["final"]), "doaj_raw_rows": len(ctx["doaj_raw"]),
        "journals_rows": len(ctx["cleaned_doaj"]),
        **{status: ctx["integration_stats"][status] for status in MATCH_STATUSES},
        "invalid_year": ctx["crossref_stats"]["invalid_year"], "fx_date": ctx["fx_date"],
        "duration_seconds": round(time.time() - ctx["started"], 1),
    }
    ctx["work_dir"] = LOCAL_WORK_DIR / ctx["run_id"]
    build_processed(ctx["final"], ctx["cleaned_doaj"], ctx["journal_license"], ctx["fx_table"],
                    run_record, ctx["work_dir"], ctx["target_dir"])
    return f"ditulis ke {ctx['work_dir']}"


def task_processed_validation(ctx):
    report, _ = validate_readback(ctx["work_dir"], len(ctx["final"]), len(ctx["cleaned_doaj"]), verbose=False)
    failed = report[~report["lolos"]]
    if len(failed) > 0:
        display(failed)
        raise ValueError(f"{len(failed)} cek read-back gagal → processed lama TIDAK diganti.")
    return f"{len(report)}/{len(report)} cek lolos"


def task_publish(ctx):
    publish_processed(ctx["work_dir"], ctx["target_dir"])
    shutil.rmtree(ctx["work_dir"])
    return f"{ctx['target_dir'].relative_to(BASE_DIR)}/ diganti"


TASK_FUNCTIONS = {
    "extract_crossref": task_extract_crossref, "extract_doaj": task_extract_doaj,
    "extract_fx": task_extract_fx, "write_manifest": task_write_manifest,
    "raw_validation": task_raw_validation, "clean_crossref": task_clean_crossref,
    "clean_doaj": task_clean_doaj, "integration": task_integration, "load": task_load,
    "processed_validation": task_processed_validation, "publish": task_publish,
}
assert set(TASK_FUNCTIONS) == set(PIPELINE_DAG), "Setiap task di DAG harus punya fungsi"


# ------------------------------------------------------------------ runner
def run_pipeline(snapshot_date=None):
    """Jalankan seluruh task DAG untuk satu snapshot (urutan topological). Return ringkasan run."""
    global ACTIVE_TRACKER
    snapshot_date = snapshot_date or pd.Timestamp.now(tz=LOCAL_TZ).strftime("%Y-%m-%d")
    run_start = pd.Timestamp.now(tz="UTC")                       # runtime: kapan run ini benar-benar dieksekusi
    run_id = "run_" + run_start.strftime("%Y%m%dT%H%M%S%f")
    run_type = classify_run(snapshot_date)                       # normal / rerun / backfill
    target_dir = publish_target(run_type, snapshot_date)         # processed/ atau processed_backfill/...
    started = time.time()
    summary = {"run_id": run_id, "run_type": run_type, "snapshot_date": snapshot_date,
               "run_timestamp": run_start.isoformat(), "output_dir": str(target_dir.relative_to(BASE_DIR)),
               "status": "failed", "error": None}
    print(f"▶️ {run_id} | snapshot {snapshot_date} | {run_type} | "
          f"dieksekusi {run_start.tz_convert(LOCAL_TZ):%Y-%m-%d %H:%M:%S} WITA")
    print(f"   output → {summary['output_dir']}/")
    tracker = TaskTracker(run_id, snapshot_date)
    ACTIVE_TRACKER = tracker
    ctx = {"snapshot_date": snapshot_date, "run_id": run_id, "run_type": run_type,
           "run_timestamp": run_start.isoformat(), "target_dir": target_dir, "started": started,
           "already_complete": manifest_path(snapshot_date).exists()}
    first_error = None

    try:
        recover_processed()                     # pulihkan processed jika run sebelumnya terhenti saat publish
        if run_type == "backfill" and not ctx["already_complete"]:
            raise ValueError(f"Backfill {snapshot_date} ditolak: raw snapshot belum lengkap (manifest tidak ada). "
                             "Sumber hanya menyediakan data saat ini, jadi snapshot lama tidak bisa diambil ulang.")
        for task_name in TASKS:                 # urutan dari DAG (topological order)
            blocked = tracker.blocked_by(task_name)
            if blocked:                         # upstream gagal → task ini dan turunannya di-skip
                tracker.skip(task_name, blocked)
                continue
            tracker.start(task_name)
            try:
                tracker.success(TASK_FUNCTIONS[task_name](ctx))
            except Exception as error:          # task gagal → cabang lain yang tidak bergantung tetap jalan
                tracker.fail(error)
                print(f"      ❌ {type(error).__name__}: {error}")
                first_error = first_error or error
        if first_error is not None:
            raise first_error

        summary.update({
            "status": "success", "articles_rows": len(ctx["final"]),
            "distinct_doi": int(ctx["final"]["DOI"].nunique()), "journals_rows": len(ctx["cleaned_doaj"]),
            "journal_license_rows": len(ctx["journal_license"]), "matched": ctx["integration_stats"]["matched"],
            "articles_fingerprint": articles_fingerprint(target_dir / "publications.db"),
        })
        print(f"   ✅ selesai dalam {time.time() - started:.1f} detik")
        return summary
    except BaseException as error:              # termasuk interupsi (mis. tombol Stop) agar tetap tercatat
        summary["error"] = f"{type(error).__name__}: {error}"
        tracker.fail(error)                     # run terhenti total di tengah task (bukan Exception biasa)
        tracker.skip_all_pending(f"run terhenti: {type(error).__name__}")
        failed_tasks = [name for name, task in tracker.tasks.items() if task["state"] == "failed"]
        where = f"di task {', '.join(failed_tasks)}" if failed_tasks else "sebelum task pertama"
        print(f"   ❌ run gagal {where}: {summary['error']}")
        raise
    finally:
        ACTIVE_TRACKER = None
        tracker.save()
        task_table = tracker.table()
        task_table.insert(1, "upstream", [", ".join(PIPELINE_DAG[t]) or "-" for t in task_table["task"]])
        display(task_table[["task", "upstream", "state", "retries", "duration_seconds", "message"]])
        summary["duration_seconds"] = round(time.time() - started, 1)
        history_file = LOG_DIR / "pipeline_run_history.csv"
        history = pd.DataFrame([{k: summary.get(k) for k in HISTORY_COLUMNS}])
        if history_file.exists():                # baca sebagai teks agar fingerprint tidak berubah jadi float
            history = pd.concat([pd.read_csv(history_file, dtype=str), history], ignore_index=True)
        history[HISTORY_COLUMNS].to_csv(history_file, index=False)   # run lama tanpa run_type → kosong


print("✅ run_pipeline() siap (digerakkan oleh PIPELINE_DAG).")

✅ run_pipeline() siap (digerakkan oleh PIPELINE_DAG).


## 🔁 Uji Idempotency: Jalankan Dua Kali

Pipeline dijalankan **dua kali berturut-turut dengan input yang sama** (snapshot `SNAPSHOT_DATE`). Run kedua tidak melakukan request ulang karena snapshot sudah lengkap.

Setelah tiap run, tabel `articles` **dibaca dari storage** lalu dibandingkan per DOI. Yang dilaporkan:
- jumlah row Run 1 dan Run 2,
- jumlah duplicate key (DOI) di masing-masing run,
- jumlah record **baru** (DOI ada di Run 2, tidak ada di Run 1),
- jumlah record yang **hilang** (DOI ada di Run 1, tidak ada di Run 2),
- jumlah record yang **di-update** (DOI sama, isi kolom berbeda).

In [60]:
def read_articles_state():
    """Baca isi tabel articles dari storage (SQLite), diurutkan per DOI."""
    with closing(sqlite3.connect(PROCESSED_DIR / "publications.db")) as conn:
        return pd.read_sql_query("SELECT * FROM articles ORDER BY doi", conn)


run_1 = run_pipeline(SNAPSHOT_DATE)
articles_after_run_1 = read_articles_state()
print()
run_2 = run_pipeline(SNAPSHOT_DATE)
articles_after_run_2 = read_articles_state()

▶️ run_20261106T172342018571 | snapshot 2026-11-07 | normal | dieksekusi 2026-11-07 01:23:42 WITA
   output → processed/
   [1/11] extract_crossref
   [2/11] extract_doaj
   [3/11] extract_fx
   [4/11] write_manifest
   [5/11] raw_validation
   [6/11] clean_crossref
   [7/11] clean_doaj
   [8/11] integration
   [9/11] load
   [10/11] processed_validation
   [11/11] publish
   ✅ selesai dalam 10.2 detik


,task,upstream,state,retries,duration_seconds,message
0,extract_crossref,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
1,extract_doaj,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
2,extract_fx,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
3,write_manifest,"extract_crossref, extract_doaj, extract_fx",success,0,0.0,"manifest sudah ada, dilewati"
4,raw_validation,write_manifest,success,0,7.4,12/12 cek raw lolos
5,clean_crossref,raw_validation,success,0,0.1,"20,000 artikel"
6,clean_doaj,raw_validation,success,0,0.3,"23,577 jurnal"
7,integration,"clean_crossref, clean_doaj",success,0,0.2,"3,798 artikel ter-match"
8,load,integration,success,0,1.1,ditulis ke work/run_20261106T172342018571
9,processed_validation,load,success,0,0.2,17/17 cek lolos



▶️ run_20261106T172352715244 | snapshot 2026-11-07 | rerun | dieksekusi 2026-11-07 01:23:52 WITA
   output → processed/
   [1/11] extract_crossref
   [2/11] extract_doaj
   [3/11] extract_fx
   [4/11] write_manifest
   [5/11] raw_validation
   [6/11] clean_crossref
   [7/11] clean_doaj
   [8/11] integration
   [9/11] load
   [10/11] processed_validation
   [11/11] publish
   ✅ selesai dalam 9.6 detik


,task,upstream,state,retries,duration_seconds,message
0,extract_crossref,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
1,extract_doaj,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
2,extract_fx,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
3,write_manifest,"extract_crossref, extract_doaj, extract_fx",success,0,0.0,"manifest sudah ada, dilewati"
4,raw_validation,write_manifest,success,0,5.8,12/12 cek raw lolos
5,clean_crossref,raw_validation,success,0,0.2,"20,000 artikel"
6,clean_doaj,raw_validation,success,0,0.4,"23,577 jurnal"
7,integration,"clean_crossref, clean_doaj",success,0,0.3,"3,798 artikel ter-match"
8,load,integration,success,0,1.6,ditulis ke work/run_20261106T172352715244
9,processed_validation,load,success,0,0.3,17/17 cek lolos


In [61]:
print("=" * 60)
print("🔁 PERBANDINGAN RUN #1 vs RUN #2")
print("=" * 60)

comparison = pd.DataFrame([run_1, run_2]).set_index("run_id")[
    ["run_type", "snapshot_date", "status", "articles_rows", "distinct_doi", "journals_rows",
     "journal_license_rows", "matched", "articles_fingerprint", "duration_seconds"]]
display(comparison)

# Bandingkan isi tabel articles per DOI (dibaca dari storage, bukan dari memori)
doi_run_1 = set(articles_after_run_1["doi"])
doi_run_2 = set(articles_after_run_2["doi"])
common_doi = sorted(doi_run_1 & doi_run_2)
row_hash_1 = pd.util.hash_pandas_object(articles_after_run_1.set_index("doi").loc[common_doi], index=True)
row_hash_2 = pd.util.hash_pandas_object(articles_after_run_2.set_index("doi").loc[common_doi], index=True)

idempotency_report = pd.DataFrame({
    "Metrik": ["Jumlah row Run 1", "Jumlah row Run 2",
               "Duplicate DOI Run 1", "Duplicate DOI Run 2",
               "Record baru (DOI hanya ada di Run 2)", "Record hilang (DOI hanya ada di Run 1)",
               "Record di-update (DOI sama, isi berbeda)"],
    "Nilai": [len(articles_after_run_1), len(articles_after_run_2),
              int(articles_after_run_1["doi"].duplicated().sum()),
              int(articles_after_run_2["doi"].duplicated().sum()),
              len(doi_run_2 - doi_run_1), len(doi_run_1 - doi_run_2),
              int((row_hash_1.values != row_hash_2.values).sum())],
    "Harapan": [len(articles_after_run_1), len(articles_after_run_1), 0, 0, 0, 0, 0],
})
idempotency_report["Sesuai"] = idempotency_report["Nilai"] == idempotency_report["Harapan"]
print("\n📋 Laporan idempotency (tabel articles):")
display(idempotency_report)

same_state = bool(idempotency_report["Sesuai"].all()) and all(
    run_1[k] == run_2[k] for k in ["journals_rows", "journal_license_rows", "matched", "articles_fingerprint"])
print(f"   State processed identik      : {same_state}")

with closing(sqlite3.connect(PROCESSED_DIR / "publications.db")) as conn:
    counts = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
              for t in ["articles", "journals_doaj", "journal_license", "fx_rates", "pipeline_runs"]}
    print(f"   Isi SQLite setelah 2 run     : {counts}")
    print("\n📝 Riwayat pipeline_runs (append, 1 baris per run):")
    display(pd.read_sql_query(
        "SELECT run_id, run_type, snapshot_date, articles_rows, journals_rows, matched_print_issn, "
        "matched_online_issn, duration_seconds FROM pipeline_runs ORDER BY run_timestamp", conn))
assert same_state, "State processed berbeda antar run!"

🔁 PERBANDINGAN RUN #1 vs RUN #2


,run_type,snapshot_date,status,articles_rows,distinct_doi,journals_rows,journal_license_rows,matched,articles_fingerprint,duration_seconds
run_id,,,,,,,,,,
run_20261106T172342018571,normal,2026-11-07,success,20000,20000,23577,27566,3798,5741008006864240858,10.3
run_20261106T172352715244,rerun,2026-11-07,success,20000,20000,23577,27566,3798,5741008006864240858,9.7



📋 Laporan idempotency (tabel articles):


,Metrik,Nilai,Harapan,Sesuai
0,Jumlah row Run 1,20000,20000,True
1,Jumlah row Run 2,20000,20000,True
2,Duplicate DOI Run 1,0,0,True
3,Duplicate DOI Run 2,0,0,True
4,Record baru (DOI hanya ada di Run 2),0,0,True
5,Record hilang (DOI hanya ada di Run 1),0,0,True
6,"Record di-update (DOI sama, isi berbeda)",0,0,True


   State processed identik      : True
   Isi SQLite setelah 2 run     : {'articles': 20000, 'journals_doaj': 23577, 'journal_license': 27566, 'fx_rates': 166, 'pipeline_runs': 27}

📝 Riwayat pipeline_runs (append, 1 baris per run):


,run_id,run_type,snapshot_date,articles_rows,journals_rows,matched_print_issn,matched_online_issn,duration_seconds
0,run_20261030T123211248313,None,2026-10-30,20000,23559,1587,2106,9.7
1,run_20261030T123224368032,None,2026-10-30,20000,23559,1587,2106,10.3
2,run_20261030T125559038235,None,2026-10-30,20000,23559,1587,2106,9.8
3,run_20261106T074009596646,None,2026-10-30,20000,23559,1587,2106,10.0
4,run_20261106T074024166464,None,2026-10-30,20000,23559,1587,2106,9.5
5,run_20261106T074102797788,None,2026-10-30,20000,23559,1587,2106,7.9
6,run_20261106T102744857648,None,2026-10-30,20000,23559,1587,2106,8.0
7,run_20261106T102758547878,None,2026-10-30,20000,23559,1587,2106,9.1
8,run_20261106T102837653179,None,2026-10-30,20000,23559,1587,2106,9.9
9,run_20261106T111340939422,None,2026-10-30,20000,23559,1587,2106,8.7


### Mengapa hasil ini benar?

- **Row Run 1 = Row Run 2, duplicate DOI = 0.** Tabel data dimuat dengan strategi **full refresh**: setiap run membangun ulang `articles` dari nol berdasarkan satu snapshot, lalu menggantikan versi lama. Tidak ada `append` ke tabel data, sehingga run ulang tidak mungkin menambah baris. DOI juga dijaga di tiga tempat: dedup DOI saat transformasi, `validate="many_to_one"` saat join (join tidak boleh menggandakan artikel), dan `PRIMARY KEY (doi)` di SQLite (insert DOI ganda akan ditolak).
- **Record baru = 0 dan record hilang = 0.** Kedua run membaca **snapshot raw yang sama**. Run kedua tidak mengambil sampel Crossref baru karena snapshot sudah lengkap (ada manifest).
- **Record di-update = 0.** Transformasi bersifat deterministik: input sama → output sama. Tidak ada nilai yang bergantung pada waktu eksekusi (misalnya kurs diambil dari file snapshot, bukan dari API saat run).
- **Yang memang bertambah hanya log.** `pipeline_runs` dan `logs/pipeline_run_history.csv` bertambah satu baris per run. Ini disengaja (*append*) untuk audit, bukan data artikel.

### Bagaimana jika pipeline dijalankan pada tanggal berbeda?

Run bulan berikutnya (snapshot baru) akan mengambil **sampel Crossref acak yang baru** dan DOAJ/kurs terbaru. Karena strategi load adalah full refresh, processed **diganti seluruhnya** oleh snapshot terbaru:
- sebagian besar DOI akan berbeda dari snapshot sebelumnya. Ini **bukan duplikat dan bukan kehilangan data**, karena snapshot lama tetap tersimpan utuh di `raw/.../ingestion_date=<tanggal lama>/` dan dapat diproses ulang kapan saja dengan `run_pipeline("<tanggal lama>")`;
- grain tetap 1 baris = 1 DOI dan DOI tetap unik di dalam processed.

Full refresh dipilih (bukan upsert/akumulasi) karena Crossref diambil sebagai **sampel stratified**: jika sampel tiap bulan diakumulasi, jumlah artikel per tahun tidak lagi seimbang dan dataset tidak lagi mewakili satu titik waktu DOAJ & kurs.

In [62]:
print("=" * 60)
print("🔎 READ-BACK VALIDATION pada processed yang aktif")
print("=" * 60)
readback_report, readback_sample = validate_readback(PROCESSED_DIR, run_2["articles_rows"], run_2["journals_rows"])
print(f"\n{'✅ Semua cek lolos' if readback_report['lolos'].all() else '❌ Ada cek yang gagal'} "
      f"({readback_report['lolos'].sum()}/{len(readback_report)})")

articles_check = pd.read_parquet(PROCESSED_DIR / "articles_enriched", partitioning=YEAR_PARTITIONING)
print(f"\n📋 Schema Parquet articles_enriched ({articles_check.shape[1]} kolom):")
print(articles_check.dtypes.to_string())
partitions = sorted(p.name for p in (PROCESSED_DIR / "articles_enriched").iterdir() if p.is_dir())
print(f"\n🗂️ Partisi: {partitions}")
example_year = int(articles_check["issued_year"].dropna().iloc[0])
one_year = pd.read_parquet(PROCESSED_DIR / "articles_enriched", partitioning=YEAR_PARTITIONING,
                           filters=[("issued_year", "=", example_year)])
print(f"⚡ Partition pruning: baca hanya issued_year={example_year} → {len(one_year):,} baris")

🔎 READ-BACK VALIDATION pada processed yang aktif


,kategori,cek,actual,expected,lolos
0,row count,Parquet articles = hasil transform,20000,20000,True
1,key,Parquet: DOI NULL,0,0,True
2,key,Parquet: DOI duplikat,0,0,True
3,schema,Parquet: kolom sesuai,True,True,True
4,null,sample_year NULL,0,0,True
5,null,matched tanpa journal_title,0,0,True
6,konsistensi,is_matched_to_doaj ⇔ doaj_id terisi,True,True,True
7,konsistensi,match_status di luar daftar,0,0,True
8,row count,SQLite articles = hasil transform,20000,20000,True
9,row count,SQLite journals_doaj = hasil clean,23577,23577,True


🔎 Sample record (dari view v_enriched_publications):


,doi,title,issued_year,match_status,journal_title,license_raw,apc_amount_usd
0,10.1001/jamahealthforum.2024.4487,Unwinding Medicaid Eligibility: Lessons for He...,2024,matched_online_issn,JAMA Health Forum,"CC BY, CC BY-NC-ND",4000.0
1,10.1001/jamanetworkopen.2025.22585,Emergency Department Use Prior to Cancer Diagn...,2025,matched_online_issn,JAMA Network Open,"CC BY, CC BY-NC-ND",3000.0
2,10.1001/jamanetworkopen.2026.7424,Retracted Randomized Clinical Trials From Supe...,2026,matched_online_issn,JAMA Network Open,"CC BY, CC BY-NC-ND",3000.0



✅ Semua cek lolos (17/17)

📋 Schema Parquet articles_enriched (34 kolom):
DOI                               object
title                             object
author                            object
author_count                       int64
sample_year                        Int64
issued_year_raw                    Int64
is_year_valid                       bool
print_issn                string[python]
online_issn               string[python]
has_issn_key                        bool
container-title                   object
publisher                         object
match_status                      object
is_matched_to_doaj                  bool
doaj_id                            Int64
journal_title                     object
doaj_publisher                    object
country                           object
license_raw                       object
license_count                      Int64
apc_status                        object
apc_amount_original              float64
apc_currency           

## 🛟 Uji Kegagalan Publish pada Run Sebenarnya

Kegagalan **disuntikkan ke `run_pipeline()` yang sebenarnya** dan terjadi pada processed asli di Google Drive, tepat di task *publish* (task terakhir DAG). Kode pipeline tidak diubah. Yang dimanipulasi hanya `os.replace` (fungsi pemindah folder) selama simulasi, lalu dikembalikan seperti semula.

| Skenario | Kejadian yang disimulasikan | Penanganan yang diharapkan |
|---|---|---|
| 1. Swap gagal | `processed_incoming/` gagal dipindah menjadi `processed/` | **Rollback**: processed lama dikembalikan |
| 2. Sesi terputus di tengah swap | Proses mati setelah `processed/` dipindah ke `processed_previous/`, sehingga rollback tidak sempat jalan | Run berikutnya melakukan **recovery** otomatis |

Uji dibagi menjadi 4 tahap agar kondisi Google Drive dapat diperiksa (dan di-*screenshot*) di antara setiap tahap. Bukti utamanya adalah **waktu terakhir diubah (*last modified*) `processed/publications.db`**. Selama processed lama dipertahankan, waktunya tidak berubah.

> Google Drive di browser bisa terlambat beberapa detik sampai 1 menit dalam menampilkan perubahan. Tunggu sebentar lalu *refresh* halaman Drive sebelum mengambil *screenshot*. Output `drive_state()` di notebook menampilkan kondisi yang sama secara langsung.

In [63]:
class SimulatedCrash(BaseException):
    """Meniru sesi Colab yang mati mendadak: bukan turunan Exception,
    sehingga blok rollback (except Exception) tidak ikut berjalan."""


real_replace = os.replace


def inject_failure(error):
    """Ganti os.replace sementara: gagal saat processed_incoming dipindah menjadi processed."""
    def failing_replace(source, destination):
        if Path(source).name == "processed_incoming":
            raise error
        return real_replace(source, destination)
    os.replace = failing_replace


def drive_state(title):
    """Tampilkan kondisi folder processed di Google Drive beserta waktu terakhir diubah (WITA)."""
    print(f"\n📁 {title}")
    for name in ["processed", "processed_incoming", "processed_previous"]:
        folder = BASE_DIR / name
        if folder.exists():
            db = folder / "publications.db"
            modified = (pd.Timestamp(db.stat().st_mtime, unit="s", tz="UTC").tz_convert("Asia/Makassar")
                        .strftime("%Y-%m-%d %H:%M:%S WITA") if db.exists() else "-")
            print(f"   ✅ {name + '/':22s} ada   | publications.db last modified: {modified}")
        else:
            print(f"   ⬜ {name + '/':22s} tidak ada")


def count_pipeline_runs():
    with closing(sqlite3.connect(PROCESSED_DIR / "publications.db")) as conn:
        return conn.execute("SELECT COUNT(*) FROM pipeline_runs").fetchone()[0]


fingerprint_before = articles_fingerprint(PROCESSED_DIR / "publications.db")
runs_before = count_pipeline_runs()
print("=" * 60)
print("🛟 [Tahap 0] KONDISI AWAL")
print("=" * 60)
print(f"Fingerprint articles : {fingerprint_before}")
print(f"Baris pipeline_runs  : {runs_before}")
drive_state("Kondisi Google Drive sebelum simulasi")

🛟 [Tahap 0] KONDISI AWAL
Fingerprint articles : 5741008006864240858
Baris pipeline_runs  : 27

📁 Kondisi Google Drive sebelum simulasi
   ✅ processed/             ada   | publications.db last modified: 2026-11-07 01:24:01 WITA
   ⬜ processed_incoming/    tidak ada
   ⬜ processed_previous/    tidak ada


📸 **Screenshot 1 (kondisi awal):** buka folder `MyDrive/kelompok5_publication_pipeline/` di Google Drive. Hanya ada folder `processed/` (tanpa `processed_incoming/` dan `processed_previous/`). Buka `processed/` dan perhatikan kolom **Last modified** pada `publications.db`.

In [64]:
print("=" * 60)
print("💥 [Tahap 1] SKENARIO 1: swap gagal → rollback")
print("=" * 60)
inject_failure(OSError("SIMULASI: processed_incoming rusak / gagal dipindahkan"))
try:
    run_pipeline(SNAPSHOT_DATE)
except OSError:
    pass
finally:
    os.replace = real_replace

fingerprint_1 = articles_fingerprint(PROCESSED_DIR / "publications.db")
drive_state("Kondisi Google Drive setelah skenario 1")
print(f"\nFingerprint sama dengan awal : {fingerprint_1 == fingerprint_before}")
print("Catatan: processed_incoming/ yang gagal sengaja dibiarkan untuk diperiksa; folder ini dibersihkan otomatis pada publish berikutnya.")

💥 [Tahap 1] SKENARIO 1: swap gagal → rollback
▶️ run_20261106T172404266785 | snapshot 2026-11-07 | rerun | dieksekusi 2026-11-07 01:24:04 WITA
   output → processed/
   [1/11] extract_crossref
   [2/11] extract_doaj
   [3/11] extract_fx
   [4/11] write_manifest
   [5/11] raw_validation
   [6/11] clean_crossref
   [7/11] clean_doaj
   [8/11] integration
   [9/11] load
   [10/11] processed_validation
   [11/11] publish
   ❌ Swap gagal: SIMULASI: processed_incoming rusak / gagal dipindahkan
   ↩️ Rollback: processed/ lama dikembalikan
      ❌ OSError: SIMULASI: processed_incoming rusak / gagal dipindahkan
   ❌ run gagal di task publish: OSError: SIMULASI: processed_incoming rusak / gagal dipindahkan


,task,upstream,state,retries,duration_seconds,message
0,extract_crossref,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
1,extract_doaj,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
2,extract_fx,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
3,write_manifest,"extract_crossref, extract_doaj, extract_fx",success,0,0.0,"manifest sudah ada, dilewati"
4,raw_validation,write_manifest,success,0,5.5,12/12 cek raw lolos
5,clean_crossref,raw_validation,success,0,0.1,"20,000 artikel"
6,clean_doaj,raw_validation,success,0,0.3,"23,577 jurnal"
7,integration,"clean_crossref, clean_doaj",success,0,0.2,"3,798 artikel ter-match"
8,load,integration,success,0,1.1,ditulis ke work/run_20261106T172404266785
9,processed_validation,load,success,0,0.2,17/17 cek lolos



📁 Kondisi Google Drive setelah skenario 1
   ✅ processed/             ada   | publications.db last modified: 2026-11-07 01:24:01 WITA
   ✅ processed_incoming/    ada   | publications.db last modified: 2026-11-07 01:24:11 WITA
   ⬜ processed_previous/    tidak ada

Fingerprint sama dengan awal : True
Catatan: processed_incoming/ yang gagal sengaja dibiarkan untuk diperiksa; folder ini dibersihkan otomatis pada publish berikutnya.


📸 **Screenshot 2 (setelah rollback):** `processed/` masih ada, dan **Last modified `publications.db` sama dengan Screenshot 1**. Artinya data lama tidak tergantikan. Folder `processed_incoming/` (hasil baru yang gagal dipasang) juga terlihat.

In [65]:
print("=" * 60)
print("💥 [Tahap 2] SKENARIO 2: sesi terputus di tengah swap")
print("=" * 60)
inject_failure(SimulatedCrash("SIMULASI: sesi Colab mati di tengah swap"))
try:
    run_pipeline(SNAPSHOT_DATE)
except SimulatedCrash as crash:
    print(f"   💥 {crash}")
finally:
    os.replace = real_replace

drive_state("Kondisi Google Drive setelah crash")
print("\n⚠️ processed/ hilang: data hanya tersisa di processed_previous/. Rollback tidak sempat berjalan.")

💥 [Tahap 2] SKENARIO 2: sesi terputus di tengah swap
▶️ run_20261106T172412721016 | snapshot 2026-11-07 | rerun | dieksekusi 2026-11-07 01:24:12 WITA
   output → processed/
   [1/11] extract_crossref
   [2/11] extract_doaj
   [3/11] extract_fx
   [4/11] write_manifest
   [5/11] raw_validation
   [6/11] clean_crossref
   [7/11] clean_doaj
   [8/11] integration
   [9/11] load
   [10/11] processed_validation
   [11/11] publish
   ❌ run gagal di task publish: SimulatedCrash: SIMULASI: sesi Colab mati di tengah swap


,task,upstream,state,retries,duration_seconds,message
0,extract_crossref,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
1,extract_doaj,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
2,extract_fx,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
3,write_manifest,"extract_crossref, extract_doaj, extract_fx",success,0,0.0,"manifest sudah ada, dilewati"
4,raw_validation,write_manifest,success,0,7.4,12/12 cek raw lolos
5,clean_crossref,raw_validation,success,0,0.1,"20,000 artikel"
6,clean_doaj,raw_validation,success,0,0.2,"23,577 jurnal"
7,integration,"clean_crossref, clean_doaj",success,0,0.2,"3,798 artikel ter-match"
8,load,integration,success,0,1.1,ditulis ke work/run_20261106T172412721016
9,processed_validation,load,success,0,0.2,17/17 cek lolos


   💥 SIMULASI: sesi Colab mati di tengah swap

📁 Kondisi Google Drive setelah crash
   ⬜ processed/             tidak ada
   ✅ processed_incoming/    ada   | publications.db last modified: 2026-11-07 01:24:21 WITA
   ✅ processed_previous/    ada   | publications.db last modified: 2026-11-07 01:24:01 WITA

⚠️ processed/ hilang: data hanya tersisa di processed_previous/. Rollback tidak sempat berjalan.


📸 **Screenshot 3 (setelah crash):** folder `processed/` **tidak ada**, sedangkan `processed_previous/` ada. Buka `processed_previous/`: **Last modified `publications.db` masih sama dengan Screenshot 1**, jadi data lama aman di folder ini.

In [66]:
print("=" * 60)
print("♻️ [Tahap 3] RUN BERIKUTNYA (normal) → recovery otomatis")
print("=" * 60)
run_after_crash = run_pipeline(SNAPSHOT_DATE)
fingerprint_2 = articles_fingerprint(PROCESSED_DIR / "publications.db")
drive_state("Kondisi Google Drive setelah recovery + run berhasil")

print("\n" + "=" * 60)
print("📋 RINGKASAN UJI KEGAGALAN")
print("=" * 60)
history = pd.read_csv(LOG_DIR / "pipeline_run_history.csv").tail(3)
display(history[["run_id", "run_type", "snapshot_date", "status", "articles_rows", "error"]])
recovery_report = pd.DataFrame({
    "Cek": ["processed/ tetap ada", "Tidak ada folder sisa (incoming/previous)",
            "Fingerprint setelah skenario 1 = awal", "Fingerprint setelah recovery = awal",
            "pipeline_runs bertambah hanya untuk run sukses"],
    "Hasil": [PROCESSED_DIR.exists(),
              not (BASE_DIR / "processed_incoming").exists() and not (BASE_DIR / "processed_previous").exists(),
              fingerprint_1 == fingerprint_before, fingerprint_2 == fingerprint_before,
              count_pipeline_runs() == runs_before + 1],
})
display(recovery_report)
assert recovery_report["Hasil"].all(), "Penanganan kegagalan publish tidak sesuai!"
print("✅ Rollback dan recovery berjalan sesuai rancangan pada run sebenarnya.")

♻️ [Tahap 3] RUN BERIKUTNYA (normal) → recovery otomatis
▶️ run_20261106T172422407275 | snapshot 2026-11-07 | rerun | dieksekusi 2026-11-07 01:24:22 WITA
   output → processed/
   ♻️ Recovery: processed/ dipulihkan dari processed_previous/
   [1/11] extract_crossref
   [2/11] extract_doaj
   [3/11] extract_fx
   [4/11] write_manifest
   [5/11] raw_validation
   [6/11] clean_crossref
   [7/11] clean_doaj
   [8/11] integration
   [9/11] load
   [10/11] processed_validation
   [11/11] publish
   ✅ selesai dalam 10.1 detik


,task,upstream,state,retries,duration_seconds,message
0,extract_crossref,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
1,extract_doaj,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
2,extract_fx,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
3,write_manifest,"extract_crossref, extract_doaj, extract_fx",success,0,0.0,"manifest sudah ada, dilewati"
4,raw_validation,write_manifest,success,0,6.7,12/12 cek raw lolos
5,clean_crossref,raw_validation,success,0,0.2,"20,000 artikel"
6,clean_doaj,raw_validation,success,0,0.4,"23,577 jurnal"
7,integration,"clean_crossref, clean_doaj",success,0,0.2,"3,798 artikel ter-match"
8,load,integration,success,0,1.4,ditulis ke work/run_20261106T172422407275
9,processed_validation,load,success,0,0.2,17/17 cek lolos



📁 Kondisi Google Drive setelah recovery + run berhasil
   ✅ processed/             ada   | publications.db last modified: 2026-11-07 01:24:31 WITA
   ⬜ processed_incoming/    tidak ada
   ⬜ processed_previous/    tidak ada

📋 RINGKASAN UJI KEGAGALAN


,run_id,run_type,snapshot_date,status,articles_rows,error
43,run_20261106T172404266785,rerun,2026-11-07,failed,NaN,OSError: SIMULASI: processed_incoming rusak / ...
44,run_20261106T172412721016,rerun,2026-11-07,failed,NaN,SimulatedCrash: SIMULASI: sesi Colab mati di t...
45,run_20261106T172422407275,rerun,2026-11-07,success,20000.0,NaN


,Cek,Hasil
0,processed/ tetap ada,True
1,Tidak ada folder sisa (incoming/previous),True
2,Fingerprint setelah skenario 1 = awal,True
3,Fingerprint setelah recovery = awal,True
4,pipeline_runs bertambah hanya untuk run sukses,True


✅ Rollback dan recovery berjalan sesuai rancangan pada run sebenarnya.


📸 **Screenshot 4 (setelah recovery):** hanya ada folder `processed/`; `processed_incoming/` dan `processed_previous/` sudah hilang. Last modified `publications.db` sekarang **lebih baru**, karena run normal berhasil mempublikasikan hasilnya. Isinya tetap identik dengan data awal, dan ini dibuktikan oleh *fingerprint* di tabel ringkasan.

In [67]:
# Contoh query SQL (bisa didemokan di rekaman)
with closing(sqlite3.connect(PROCESSED_DIR / "publications.db")) as conn:
    print("📊 Q1. Persentase artikel yang ter-match ke DOAJ per tahun")
    display(pd.read_sql_query("""
        SELECT issued_year,
               COUNT(*)                                              AS total_artikel,
               SUM(doaj_id IS NOT NULL)                              AS matched_to_doaj,
               ROUND(100.0 * SUM(doaj_id IS NOT NULL) / COUNT(*), 1) AS pct_matched
        FROM articles
        GROUP BY issued_year
        ORDER BY issued_year
    """, conn))

    print("\n📜 Q2. Jumlah artikel per lisensi (lewat tabel jembatan journal_license)")
    display(pd.read_sql_query("""
        SELECT l.license, COUNT(DISTINCT a.doi) AS jumlah_artikel
        FROM articles a
        JOIN journal_license l ON a.doaj_id = l.doaj_id
        GROUP BY l.license
        ORDER BY jumlah_artikel DESC
    """, conn))

    print("\n💱 Q3. Jejak konversi APC ke USD")
    display(pd.read_sql_query("""
        SELECT journal_title, apc_amount_raw, apc_amount_original, apc_currency, fx_rate, fx_date, apc_amount_usd
        FROM journals_doaj
        WHERE apc_conversion_status = 'converted'
        LIMIT 5
    """, conn))

    print("\n🔗 Q4. Status integrasi artikel")
    display(pd.read_sql_query("""
        SELECT match_status, COUNT(*) AS jumlah
        FROM articles
        GROUP BY match_status
        ORDER BY jumlah DESC
    """, conn))

    print("\n🧭 Q5. Jejak ISSN ambigu di DOAJ")
    display(pd.read_sql_query("""
        SELECT journal_title, print_issn_raw, online_issn_raw, print_issn, online_issn,
               pissn_is_ambiguous, eissn_is_ambiguous
        FROM journals_doaj
        WHERE pissn_is_ambiguous = 1 OR eissn_is_ambiguous = 1
    """, conn))

📊 Q1. Persentase artikel yang ter-match ke DOAJ per tahun


,issued_year,total_artikel,matched_to_doaj,pct_matched
0,2011,1250,154,12.3
1,2012,1250,156,12.5
2,2013,1250,185,14.8
3,2014,1250,196,15.7
4,2015,1250,204,16.3
5,2016,1250,194,15.5
6,2017,1250,246,19.7
7,2018,1250,246,19.7
8,2019,1250,265,21.2
9,2020,1250,287,23.0



📜 Q2. Jumlah artikel per lisensi (lewat tabel jembatan journal_license)


,license,jumlah_artikel
0,CC BY,2857
1,CC BY-NC-ND,1047
2,CC BY-NC,768
3,CC BY-NC-SA,220
4,CC BY-SA,131
5,CC BY-ND,75
6,Publisher's own license,37
7,CC0,31
8,Public domain,3



💱 Q3. Jejak konversi APC ke USD


,journal_title,apc_amount_raw,apc_amount_original,apc_currency,fx_rate,fx_date,apc_amount_usd
0,مجله دامپزشکی ایران.,4000000 IRR,4000000.0,IRR,1.798421e+06,2026-11-06,2.22
1,Akroterion,1000 ZAR,1000.0,ZAR,1.645286e+01,2026-11-06,60.78
2,Shuichan Xuebao,8000 CNY,8000.0,CNY,6.682525e+00,2026-11-06,1197.15
3,Tiedao biaozhun sheji,4000 CNY,4000.0,CNY,6.682525e+00,2026-11-06,598.58
4,Al-Manhaj,500000 IDR,500000.0,IDR,1.807999e+04,2026-11-06,27.65



🔗 Q4. Status integrasi artikel


,match_status,jumlah
0,not_matched_to_doaj,16023
1,matched_online_issn,2148
2,matched_print_issn,1650
3,no_issn,179



🧭 Q5. Jejak ISSN ambigu di DOAJ


,journal_title,print_issn_raw,online_issn_raw,print_issn,online_issn,pissn_is_ambiguous,eissn_is_ambiguous
0,Известия вузов: Прикладная химия и биотехнология,2227-2925,2500-154X,2227-2925,None,0,1
1,Известия вузов: Инвестиции. Строительство. Нед...,2227-2917,2500-154X,2227-2917,None,0,1


In [68]:
# Opsional: unduh processed layer sebagai zip (file juga sudah tersimpan permanen di Google Drive)
archive_path = shutil.make_archive(f"processed_{SNAPSHOT_DATE}", "zip", root_dir=BASE_DIR, base_dir="processed")
print(f"📦 Arsip: {archive_path} ({os.path.getsize(archive_path) / 1024:.1f} KB)")
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print(f"ℹ️  Bukan di Colab — processed tersimpan di {PROCESSED_DIR}")

📦 Arsip: /content/processed_2026-11-07.zip (12341.5 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

# 🧪 BAGIAN 11 — UJI SPRINT 3: RETRY STATE, SKIPPED, DAN BACKFILL

Bagian ini membuktikan tiga perilaku orkestrasi yang belum terlihat pada run normal:

| Uji | Yang dibuktikan |
|---|---|
| **A. API bermasalah di task awal** | state `retrying`, transient failure yang pulih setelah retry, task `failed` setelah retry habis, task turunan `skipped`, dan cabang DAG lain tetap berjalan |
| **B. Backfill snapshot lama** | snapshot lama diproses ke `processed_backfill/` tanpa menyentuh `processed/`, hasilnya identik dengan hasil Sprint 2, dan backfill tanpa raw ditolak |

Uji B memerlukan **dua snapshot**: `INGESTION_DATE` di sel konfigurasi adalah snapshot terbaru (diproses ke `processed/`), sedangkan snapshot `2026-10-30` menjadi snapshot lama yang di-backfill.

### Uji A — API bermasalah di task awal

Gangguan API **disimulasikan** dengan mengganti sementara `requests.get`, sama seperti uji kegagalan publish yang mengganti `os.replace`:

| Sumber | Simulasi | Hasil yang diharapkan |
|---|---|---|
| Crossref | request pertama HTTP 503, berikutnya normal | `retrying` → `success` (transient failure pulih) |
| DOAJ | normal (request sungguhan) | `success`: cabang ini tidak bergantung pada kurs |
| Kurs | selalu HTTP 503 (API down) | `retrying` 2× → `failed` |
| `write_manifest` dan semua task sesudahnya | — | `skipped`, karena upstream `extract_fx` gagal |

Uji ini memakai kode pipeline yang sama (`run_pipeline()`), tetapi dijalankan di folder terpisah `uji_sprint3/` dengan Crossref dibatasi 1 tahun × 100 artikel, agar cepat dan **tidak mengotori raw, processed, maupun log produksi**. Setelah uji, semua path dikembalikan ke folder produksi.

In [69]:
print("=" * 60)
print("🧪 [Uji A] API BERMASALAH DI TASK AWAL (simulasi)")
print("=" * 60)

TEST_DIR = BASE_DIR / "uji_sprint3"
production_settings = {"RAW_DIR": RAW_DIR, "PROCESSED_DIR": PROCESSED_DIR, "BACKFILL_DIR": BACKFILL_DIR,
                       "LOG_DIR": LOG_DIR, "MIN_YEAR": MIN_YEAR, "CROSSREF_ROWS_PER_YEAR": CROSSREF_ROWS_PER_YEAR}
production_fingerprint = articles_fingerprint(PROCESSED_DIR / "publications.db")
production_manifests = complete_snapshots()
real_get = requests.get
request_count = {"crossref": 0, "fx": 0}


def fake_response(status_code):
    response = requests.models.Response()
    response.status_code = status_code
    return response


def unstable_get(url, *args, **kwargs):
    """Crossref: request pertama 503 lalu normal. Kurs: selalu 503. DOAJ: normal."""
    if url == CROSSREF_URL:
        request_count["crossref"] += 1
        if request_count["crossref"] == 1:
            return fake_response(503)
    if url == FX_API_URL:
        request_count["fx"] += 1
        return fake_response(503)
    return real_get(url, *args, **kwargs)


# Arahkan pipeline ke folder uji (dibuat ulang setiap uji dijalankan)
if TEST_DIR.exists():
    shutil.rmtree(TEST_DIR)
RAW_DIR, PROCESSED_DIR = TEST_DIR / "raw", TEST_DIR / "processed"
BACKFILL_DIR, LOG_DIR = TEST_DIR / "processed_backfill", TEST_DIR / "logs"
LOG_DIR.mkdir(parents=True)
MIN_YEAR = int(SNAPSHOT_DATE[:4])                    # Crossref cukup 1 tahun
CROSSREF_ROWS_PER_YEAR = CROSSREF_SAMPLE_SIZE        # 100 artikel = 1 request

requests.get = unstable_get
try:
    run_pipeline(SNAPSHOT_DATE)
except Exception as error:
    print(f"\n✅ Run gagal sesuai skenario: {type(error).__name__}")
finally:
    requests.get = real_get                          # kembalikan semua ke kondisi produksi
    test_tasks = pd.read_csv(LOG_DIR / "task_runs.csv")
    test_events = pd.read_csv(LOG_DIR / "task_events.csv")
    shutil.rmtree(RAW_DIR, ignore_errors=True)       # raw uji tidak diperlukan lagi (log uji tetap disimpan)
    globals().update(production_settings)

print("\n🕒 Riwayat state task extract (logs/task_events.csv):")
display(test_events[test_events["task"].str.startswith("extract")][["task", "state", "time", "message"]])

state = test_tasks.set_index("task")["state"]
retries = test_tasks.set_index("task")["retries"]
downstream = [task for task in TASKS if not task.startswith("extract")]
uji_a = pd.DataFrame({
    "Cek": ["extract_crossref: retry lalu success (transient pulih)",
            "extract_doaj: success (cabang independen tetap jalan)",
            "extract_fx: failed setelah 2× retry",
            "write_manifest s.d. publish: semua skipped",
            "processed/ produksi tidak berubah",
            "raw produksi tidak bertambah snapshot"],
    "Hasil": [state["extract_crossref"] == "success" and retries["extract_crossref"] == 1,
              state["extract_doaj"] == "success",
              state["extract_fx"] == "failed" and retries["extract_fx"] == HTTP_MAX_RETRIES - 1,
              bool((state[downstream] == "skipped").all()),
              articles_fingerprint(PROCESSED_DIR / "publications.db") == production_fingerprint,
              complete_snapshots() == production_manifests],
})
display(uji_a)
assert uji_a["Hasil"].all(), "Uji A tidak sesuai!"
print(f"✅ Uji A sesuai. Path kembali ke produksi: {RAW_DIR}")

🧪 [Uji A] API BERMASALAH DI TASK AWAL (simulasi)
▶️ run_20261106T172435344010 | snapshot 2026-11-07 | normal | dieksekusi 2026-11-07 01:24:35 WITA
   output → uji_sprint3/processed/
   [1/11] extract_crossref
      ⚠️ 2026: status 503 (percobaan 1/3)
   📄 2026 |  1 request | 100 artikel | 0 DOI duplikat dilewati | ✅
   [2/11] extract_doaj
   💾 DOAJ: 23,578 jurnal × 52 kolom (24.7 MB)
   [3/11] extract_fx
      ⚠️ Kurs: status 503 (percobaan 1/3)
      ⚠️ Kurs: status 503 (percobaan 2/3)
      ⚠️ Kurs: status 503 (percobaan 3/3)
      ❌ IncompleteSnapshotError: Request kurs gagal setelah semua percobaan.
   [4/11] write_manifest → skipped
   [5/11] raw_validation → skipped
   [6/11] clean_crossref → skipped
   [7/11] clean_doaj → skipped
   [8/11] integration → skipped
   [9/11] load → skipped
   [10/11] processed_validation → skipped
   [11/11] publish → skipped
   ❌ run gagal di task extract_fx: IncompleteSnapshotError: Request kurs gagal setelah semua percobaan.


,task,upstream,state,retries,duration_seconds,message
0,extract_crossref,-,success,1,4.0,Crossref diambil
1,extract_doaj,-,success,0,3.3,DOAJ diambil
2,extract_fx,-,failed,2,6.0,IncompleteSnapshotError: Request kurs gagal se...
3,write_manifest,"extract_crossref, extract_doaj, extract_fx",skipped,0,NaN,upstream tidak success: extract_fx
4,raw_validation,write_manifest,skipped,0,NaN,upstream tidak success: write_manifest
5,clean_crossref,raw_validation,skipped,0,NaN,upstream tidak success: raw_validation
6,clean_doaj,raw_validation,skipped,0,NaN,upstream tidak success: raw_validation
7,integration,"clean_crossref, clean_doaj",skipped,0,NaN,"upstream tidak success: clean_crossref, clean_..."
8,load,integration,skipped,0,NaN,upstream tidak success: integration
9,processed_validation,load,skipped,0,NaN,upstream tidak success: load



✅ Run gagal sesuai skenario: IncompleteSnapshotError

🕒 Riwayat state task extract (logs/task_events.csv):


,task,state,time,message
0,extract_crossref,running,2026-11-07T01:24:35.346225+08:00,NaN
1,extract_crossref,retrying,2026-11-07T01:24:35.358738+08:00,"2026: status 503, percobaan ulang ke-1"
2,extract_crossref,success,2026-11-07T01:24:39.390311+08:00,Crossref diambil
3,extract_doaj,running,2026-11-07T01:24:39.390354+08:00,NaN
4,extract_doaj,success,2026-11-07T01:24:42.668567+08:00,DOAJ diambil
5,extract_fx,running,2026-11-07T01:24:42.668663+08:00,NaN
6,extract_fx,retrying,2026-11-07T01:24:42.683635+08:00,"Kurs: status 503, percobaan ulang ke-1"
7,extract_fx,retrying,2026-11-07T01:24:44.684009+08:00,"Kurs: status 503, percobaan ulang ke-2"
8,extract_fx,failed,2026-11-07T01:24:48.684593+08:00,IncompleteSnapshotError: Request kurs gagal se...


,Cek,Hasil
0,extract_crossref: retry lalu success (transien...,True
1,extract_doaj: success (cabang independen tetap...,True
2,extract_fx: failed setelah 2× retry,True
3,write_manifest s.d. publish: semua skipped,True
4,processed/ produksi tidak berubah,True
5,raw produksi tidak bertambah snapshot,True


✅ Uji A sesuai. Path kembali ke produksi: /content/drive/MyDrive/kelompok5_publication_pipeline/raw


📸 **Screenshot Uji A:** tabel status 11 task (Crossref `success` dengan `retries` 1, DOAJ `success`, kurs `failed` dengan `retries` 2, sisanya `skipped`) dan tabel riwayat state `running → retrying → retrying → failed` untuk `extract_fx`.

### Uji B — Backfill snapshot lama

Snapshot `2026-10-30` (snapshot Sprint 2) diproses ulang sebagai **backfill**, karena sudah ada snapshot yang lebih baru (`INGESTION_DATE`). Yang diharapkan:

1. `run_type` otomatis `backfill`, dan hasilnya ditulis ke `processed_backfill/snapshot_date=2026-10-30/`.
2. `processed/` **tidak berubah** dan tetap berisi snapshot terbaru.
3. Hasil backfill **identik dengan hasil Sprint 2** (fingerprint sama). Ini membuktikan raw lama yang disimpan cukup untuk membangun ulang hasilnya kapan saja.
4. Backfill untuk tanggal yang raw-nya tidak ada (`2026-10-01`) **ditolak** sebelum task pertama, tanpa request ke API.

In [70]:
print("=" * 60)
print("🧪 [Uji B] BACKFILL SNAPSHOT LAMA")
print("=" * 60)

BACKFILL_SNAPSHOT = "2026-10-30"
MISSING_SNAPSHOT = "2026-10-01"                      # tidak ada raw-nya
SPRINT2_FINGERPRINT = 5652964027706646736            # fingerprint articles snapshot 2026-10-30 (Sprint 2)
assert BACKFILL_SNAPSHOT < SNAPSHOT_DATE, "INGESTION_DATE harus lebih baru dari snapshot yang di-backfill"

processed_before = articles_fingerprint(PROCESSED_DIR / "publications.db")
backfill_run = run_pipeline(BACKFILL_SNAPSHOT)
backfill_dir = publish_target("backfill", BACKFILL_SNAPSHOT)

print(f"\n▶️ Backfill {MISSING_SNAPSHOT} (raw tidak ada) → harus ditolak")
try:
    run_pipeline(MISSING_SNAPSHOT)
    missing_rejected = False
except ValueError:
    missing_rejected = True

with closing(sqlite3.connect(PROCESSED_DIR / "publications.db")) as conn:
    processed_snapshot = conn.execute(
        "SELECT snapshot_date FROM pipeline_runs ORDER BY run_timestamp DESC LIMIT 1").fetchone()[0]

uji_b = pd.DataFrame({
    "Cek": ["run_type otomatis backfill",
            f"output ke {backfill_dir.relative_to(BASE_DIR)}/",
            "processed/ tidak berubah",
            f"processed/ tetap berisi snapshot terbaru ({SNAPSHOT_DATE})",
            "hasil backfill identik dengan hasil Sprint 2",
            f"backfill {MISSING_SNAPSHOT} (raw tidak ada) ditolak"],
    "Hasil": [backfill_run["run_type"] == "backfill",
              (backfill_dir / "publications.db").exists(),
              articles_fingerprint(PROCESSED_DIR / "publications.db") == processed_before,
              processed_snapshot == SNAPSHOT_DATE,
              backfill_run["articles_fingerprint"] == SPRINT2_FINGERPRINT,
              missing_rejected],
})
display(uji_b)
print("\n📝 Riwayat run terakhir (logs/pipeline_run_history.csv):")
display(pd.read_csv(LOG_DIR / "pipeline_run_history.csv", dtype=str).tail(4)[
    ["run_id", "run_type", "snapshot_date", "output_dir", "status", "articles_fingerprint"]])
assert uji_b["Hasil"].all(), "Uji B tidak sesuai!"
print("✅ Uji B sesuai: backfill tidak menyentuh processed/ dan hasilnya dapat direproduksi.")

🧪 [Uji B] BACKFILL SNAPSHOT LAMA
▶️ run_20261106T172449747389 | snapshot 2026-10-30 | backfill | dieksekusi 2026-11-07 01:24:49 WITA
   output → processed_backfill/snapshot_date=2026-10-30/
   [1/11] extract_crossref
   [2/11] extract_doaj
   [3/11] extract_fx
   [4/11] write_manifest
   [5/11] raw_validation
   [6/11] clean_crossref
   [7/11] clean_doaj
   [8/11] integration
   [9/11] load
   [10/11] processed_validation
   [11/11] publish
   ✅ selesai dalam 9.9 detik


,task,upstream,state,retries,duration_seconds,message
0,extract_crossref,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
1,extract_doaj,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
2,extract_fx,-,success,0,0.0,"snapshot sudah lengkap, dilewati"
3,write_manifest,"extract_crossref, extract_doaj, extract_fx",success,0,0.0,"manifest sudah ada, dilewati"
4,raw_validation,write_manifest,success,0,6.7,12/12 cek raw lolos
5,clean_crossref,raw_validation,success,0,0.2,"20,000 artikel"
6,clean_doaj,raw_validation,success,0,0.4,"23,559 jurnal"
7,integration,"clean_crossref, clean_doaj",success,0,0.2,"3,693 artikel ter-match"
8,load,integration,success,0,1.4,ditulis ke work/run_20261106T172449747389
9,processed_validation,load,success,0,0.2,17/17 cek lolos



▶️ Backfill 2026-10-01 (raw tidak ada) → harus ditolak
▶️ run_20261106T172459758334 | snapshot 2026-10-01 | backfill | dieksekusi 2026-11-07 01:24:59 WITA
   output → processed_backfill/snapshot_date=2026-10-01/
   ❌ run gagal sebelum task pertama: ValueError: Backfill 2026-10-01 ditolak: raw snapshot belum lengkap (manifest tidak ada). Sumber hanya menyediakan data saat ini, jadi snapshot lama tidak bisa diambil ulang.


,task,upstream,state,retries,duration_seconds,message
0,extract_crossref,-,skipped,0,None,run terhenti: ValueError
1,extract_doaj,-,skipped,0,None,run terhenti: ValueError
2,extract_fx,-,skipped,0,None,run terhenti: ValueError
3,write_manifest,"extract_crossref, extract_doaj, extract_fx",skipped,0,None,run terhenti: ValueError
4,raw_validation,write_manifest,skipped,0,None,run terhenti: ValueError
5,clean_crossref,raw_validation,skipped,0,None,run terhenti: ValueError
6,clean_doaj,raw_validation,skipped,0,None,run terhenti: ValueError
7,integration,"clean_crossref, clean_doaj",skipped,0,None,run terhenti: ValueError
8,load,integration,skipped,0,None,run terhenti: ValueError
9,processed_validation,load,skipped,0,None,run terhenti: ValueError


,Cek,Hasil
0,run_type otomatis backfill,True
1,output ke processed_backfill/snapshot_date=202...,True
2,processed/ tidak berubah,True
3,processed/ tetap berisi snapshot terbaru (2026...,True
4,hasil backfill identik dengan hasil Sprint 2,True
5,backfill 2026-10-01 (raw tidak ada) ditolak,True



📝 Riwayat run terakhir (logs/pipeline_run_history.csv):


,run_id,run_type,snapshot_date,output_dir,status,articles_fingerprint
44,run_20261106T172412721016,rerun,2026-11-07,processed,failed,NaN
45,run_20261106T172422407275,rerun,2026-11-07,processed,success,5741008006864240858
46,run_20261106T172449747389,backfill,2026-10-30,processed_backfill/snapshot_date=2026-10-30,success,5652964027706646736
47,run_20261106T172459758334,backfill,2026-10-01,processed_backfill/snapshot_date=2026-10-01,failed,NaN


✅ Uji B sesuai: backfill tidak menyentuh processed/ dan hasilnya dapat direproduksi.


📸 **Screenshot Uji B:** folder Google Drive `kelompok5_publication_pipeline/` yang memperlihatkan `processed/` (snapshot terbaru) dan `processed_backfill/snapshot_date=2026-10-30/` berdampingan, serta tabel hasil Uji B.

---

# 🗺️ BAGIAN 12 — DESAIN OPERASIONAL PIPELINE

### Diagram workflow

![Diagram pipeline Sprint 2](https://raw.githubusercontent.com/IRay3477/DatEng/main/docs/Diagram_Pipeline_Kelompok_5_Sprint_2.png)

### Daftar task, input/output, dependency, failure & retry

**Prinsip retry:** retry hanya untuk **transient failure** (timeout, koneksi putus, HTTP 429/5xx). Error permanen (HTTP 4xx lain, schema berubah, key tidak unik, tipe data rusak) adalah kesalahan logic/data → **tidak di-retry**, pipeline berhenti agar masalahnya terlihat.

| # | Task | Fungsi | Input | Output | Dependency | Boleh retry? | Failure policy |
|---|------|--------|-------|--------|------------|--------------|----------------|
| 1 | Ingestion | `extract_snapshot` (`extract_crossref`, `extract_doaj`, `extract_fx`) | Crossref API, `doaj.org/csv`, `open.er-api.com` | `raw/<sumber>/ingestion_date=<tgl>/...` + manifest | — | ✅ Ya: 3× per request, backoff 2s & 4s, hanya untuk timeout/koneksi/429/5xx | 4xx lain → `NonRetryableError`, berhenti. Sumber/tahun yang tidak lengkap tidak di-commit (`.tmp` → `.incomplete`), manifest tidak ditulis. Run ulang melanjutkan bagian yang belum lengkap |
| 2 | Raw validation | `load_raw` + `validate_raw` | snapshot raw + manifest | DataFrame raw, `staging/crossref_flat_<tgl>.csv`, laporan 12 cek | 1 | ❌ Tidak (raw yang sama akan gagal lagi) | Manifest tidak ada / cek gagal → `IncompleteSnapshotError`, berhenti sebelum transformasi |
| 3 | Transformation | `clean_crossref`, `clean_doaj` | raw Crossref, raw DOAJ, kurs | artikel bersih + flag; jurnal bersih; `journal_license`; log DOAJ ganda | 2 | ❌ Tidak (error logic) | Error → berhenti; processed lama tidak tersentuh. Mata uang tanpa kurs tidak error, ditandai `currency_not_found` |
| 4 | Integration | `integrate` | output task 3 | `articles_enriched` (1 baris = 1 DOI) | 3 | ❌ Tidak | Key DOAJ tidak unik → `assert` gagal; join menggandakan baris → `validate="many_to_one"` gagal; berhenti |
| 5 | Load | `build_processed` | output task 3 & 4 | Parquet (partisi `issued_year`) + SQLite di folder kerja `work/` | 4 | ❌ Tidak | Error tulis → folder kerja dibuang, processed lama tetap |
| 6 | Processed validation | `validate_readback` | `work/` (dibaca ulang dari file) | laporan 17 cek | 5 | ❌ Tidak | Ada cek gagal → berhenti, **tidak publish** |
| 7 | Publish | `publish_processed` | `work/` | `processed/` di Google Drive | 6 | ❌ Tidak (dijalankan ulang manual) | Disalin ke `processed_incoming/` dan dicek utuh dulu, baru di-swap. Swap gagal → rollback `processed_previous/` menjadi `processed/`; sesi terputus di tengah swap → `recover_processed()` di awal run berikutnya |

Setiap run, berhasil maupun gagal, tercatat di `logs/pipeline_run_history.csv` (status + pesan error). Run yang berhasil juga tercatat di tabel `pipeline_runs`.

### Jadwal eksekusi
| Sumber | Sifat | Pertimbangan |
|--------|-------|--------------|
| Crossref | Metadata terus bertambah; kita mengambil **sampel** per tahun | Sampel baru tiap hari tidak menambah nilai analisis |
| DOAJ | CSV diperbarui terus, tetapi status jurnal & APC jarang berubah | Perubahan bermakna terjadi dalam skala minggu–bulan |
| Kurs | Update harian | Hanya dipakai untuk APC; fluktuasi harian tidak material |

**Usulan: dijalankan bulanan**, misalnya setiap tanggal 1 pukul 02.00 WITA (di luar jam sibuk API). Jika run gagal karena transient failure, run ulang pada hari yang sama cukup memanggil `run_pipeline()` lagi: bagian snapshot yang sudah lengkap dilewati. Setiap run menghasilkan snapshot raw baru (`ingestion_date` baru), dan processed di-*refresh* ke snapshot terbaru. Untuk saat ini eksekusi masih dipicu manual dari notebook; penjadwalan otomatis (mis. cron/GitHub Actions/Airflow memanggil `run_pipeline()`) dapat dibahas di sprint berikutnya.

### Desain storage
| Pertanyaan | Jawaban |
|-----------|---------|
| Isi raw layer | Respons sumber **apa adanya** (Crossref `.jsonl.gz`, CSV DOAJ, JSON kurs) + manifest metadata run. Hasil flatten bukan raw → `staging/` |
| Isi processed layer | Data bersih & terintegrasi + flag kualitas (`match_status`, `has_issn_key`, `is_year_valid`, `*_is_ambiguous`, `apc_conversion_status`) |
| Format | Raw: format asli (dikompres). Processed: Parquet. Serving: SQLite |
| Perlu database? | Ya sebagai **serving layer**: query SQL, relasi PK/FK eksplisit, log `pipeline_runs`. Parquet tetap penyimpanan analitik utama |
| Parquet cocok? | Ya: tipe data terjaga, kolumnar, terkompresi |
| Partitioning? | Untuk ±2 MB tidak dibutuhkan demi performa, tetapi diterapkan karena alasan desain: sejalan dengan ekstraksi per tahun & mendukung *partition pruning* |
| Partition key | Processed: `issued_year` (tahun tidak valid → `__HIVE_DEFAULT_PARTITION__`). Raw: `source` + `ingestion_date`. DOI/ISSN tidak cocok (kardinalitas terlalu tinggi) |

---

## 🔄 Ringkasan Perubahan Sprint 1 → Sprint 2

| Aspek | Sprint 1 | Sprint 2 |
|-------|----------|----------|
| Bentuk pipeline | Sel notebook dijalankan berurutan (Run All) | 7 task sebagai fungsi Python, dijalankan lewat `run_pipeline(snapshot_date)` |
| Ekstraksi Crossref | Cursor dengan filter `from-pub-date:2011` → 88% artikel tahun 2011 | Stratified random sampling ±1.250 artikel per tahun 2011–2026 |
| Raw storage | `raw/crossref_raw.json` (ditimpa tiap run) + `crossref_raw.csv` (sebenarnya hasil flatten) | Per sumber & `ingestion_date`, tidak pernah ditimpa; flatten dipindah ke `staging/`; manifest per snapshot; disimpan di Google Drive |
| Kurs APC | Di-*hardcode* di kode, tanpa sumber & tanggal | Dari API, respons mentah disimpan, `fx_rate` & `fx_date` per jurnal |
| Data bermasalah | Baris tanpa ISSN & tahun tidak valid dihapus; license diambil 1 | Ditandai (`match_status`, `is_year_valid`, `*_is_ambiguous`); license utuh + tabel `journal_license` |
| Label hasil join | `is_doaj_indexed`, "Unindexed / Non-DOAJ" | `is_matched_to_doaj`, `not_matched_to_doaj`, atribut DOAJ NULL |
| Processed storage | 1 file Parquet | Parquet partisi `issued_year` + SQLite (PK/FK, view, `pipeline_runs`) |
| Strategi load | Overwrite file tanpa pengaman | Full refresh per snapshot; dibangun di folder kerja → divalidasi → baru menggantikan processed lama |
| Validasi | Cek shape setelah tulis | Raw validation (12 cek) + read-back validation (17 cek) |
| Kegagalan | Tidak ditangani (error di tengah bisa meninggalkan file setengah jadi) | Retry hanya untuk transient failure; file `.tmp` → rename; publish dengan rollback & recovery sehingga processed lama tetap utuh jika gagal |
| Idempotency | Belum diuji | Run 2× → row, DOI, record baru/hilang/di-update dilaporkan |
| Jejak run | Tidak ada | `run_id`, `pipeline_runs`, `pipeline_run_history.csv`, `crossref_extraction_log.csv` |
| Schedule | Belum ada | Bulanan, dengan alasan per sumber |

---

## ✅ Jawaban Checklist Sprint 2

| # | Pertanyaan | Jawaban |
|---|-----------|---------|
| 1 | Apa grain processed dataset? | **1 baris = 1 artikel ilmiah (DOI)** pada tabel `articles_enriched`/`articles`. Tabel pendukung: 1 baris = 1 jurnal DOAJ (`journals_doaj`), 1 baris = 1 pasangan jurnal–lisensi (`journal_license`) |
| 2 | Apa primary / integration key? | Primary key artikel: **DOI** (lowercase). Integration key Crossref ↔ DOAJ: **Print ISSN**, lalu **Online ISSN** (E-ISSN). Primary key jurnal: `doaj_id` |
| 3 | Di mana raw data disimpan? | Google Drive: `raw/crossref/`, `raw/doaj/`, `raw/fx_rates/` dipartisi `ingestion_date=<tgl>`, dengan `raw/_manifests/` |
| 4 | Di mana processed data disimpan? | Google Drive: `processed/articles_enriched/` (Parquet partisi `issued_year`), `processed/*.parquet`, dan `processed/publications.db` (SQLite) |
| 5 | Kenapa memilih storage tersebut? | Raw: format asli agar tidak kehilangan informasi. Parquet: kolumnar, schema & tipe data terjaga, untuk analisis. SQLite: ringan, satu file, cocok untuk project lokal/single user, mendukung SQL & PK/FK sebagai serving layer. PostgreSQL tidak diperlukan karena tidak ada akses bersamaan |
| 6 | Apa yang terjadi saat pipeline dijalankan ulang? | Snapshot sama → tidak ada request ulang, processed dibangun ulang dan hasilnya identik (row, DOI, isi sama). Snapshot baru → raw baru ditambahkan, processed di-refresh ke snapshot terbaru |
| 7 | Bagaimana mencegah duplicate? | Full refresh (bukan append) untuk tabel data; dedup DOI saat transformasi; dedup ISSN DOAJ + `validate="many_to_one"` saat join; `PRIMARY KEY (doi)` di SQLite; read-back mengecek DOI unik |
| 8 | Task mana yang boleh retry? | Hanya **ingestion** (request HTTP), dan hanya untuk timeout/koneksi/429/5xx. Task lain tidak di-retry karena error-nya berasal dari data/logic |
| 9 | Seberapa sering pipeline dijalankan? | **Bulanan**. Sampel Crossref tidak perlu diperbarui harian, status jurnal & APC di DOAJ berubah dalam skala minggu–bulan, dan fluktuasi kurs harian tidak material untuk APC |
| 10 | Bagaimana mengetahui run berhasil? | `run_pipeline()` mengembalikan `status = "success"` hanya jika raw validation dan read-back validation lolos semua; status setiap run (termasuk yang gagal beserta pesan error) tercatat di `logs/pipeline_run_history.csv` dan run sukses di tabel `pipeline_runs` |